# ARC-AGI-2 — self-contained submission notebook

No code dataset or API token needed: the entire solver is embedded below.

**Steps:** attach the competition data (auto on a competition notebook) + a model dataset, set `MODEL_DS`, **Internet = Off**, then **Run All**. Phase A: leave `MODEL_DS = None` to validate the plumbing with the DSL ensemble (no model).

In [ ]:
# Kaggle env prep (offline-safe). Some Kaggle images ship a `torchao` too old
# for the installed `peft`, which makes LoRA (TTT) adapter creation raise
# `ImportError: incompatible version of torchao` on EVERY task -> TTT silently
# degrades to the DSL-only ensemble. We don't use torchao, so drop the
# incompatible version and let peft fall back to the standard LoRA path.
import sys, subprocess
try:
    import torchao
    from packaging.version import parse as _p
    if _p(getattr(torchao, '__version__', '0')) < _p('0.16.0'):
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
                       check=False)
        for _m in [k for k in list(sys.modules) if k.startswith('torchao')]:
            del sys.modules[_m]
        print('removed incompatible torchao (<0.16) so PEFT/LoRA can run')
    else:
        print('torchao', torchao.__version__, 'is compatible')
except ImportError:
    print('torchao not installed -> nothing to do')


In [ ]:
# Self-contained bootstrap: write the embedded `arc` package to disk.
import base64, json, os, sys
FILES = json.loads(r'''{
"src/arc/__init__.py": "",
"src/arc/augment/__init__.py": "",
"src/arc/augment/color.py": "IiIiQ29sb3VyIChzeW1ib2wpIHBlcm11dGF0aW9ucyBvbiBncmlkcywgZWFjaCB3aXRoIGFuIGV4YWN0IGludmVyc2UuCgpBIHBlcm11dGF0aW9uIGlzIGEgbGVuZ3RoLTEwIHR1cGxlIGBwZXJtYCBtYXBwaW5nIHN5bWJvbCBzIC0+IHBlcm1bc10uIEFwcGx5aW5nCmBwZXJtYCB0aGVuIGl0cyBpbnZlcnNlIHJlY292ZXJzIHRoZSBvcmlnaW5hbCBncmlkLiBPcHRpb25hbGx5IHN5bWJvbCAwIGlzIGhlbGQKZml4ZWQgKHRyZWF0IDAgYXMgYmFja2dyb3VuZCk7IGJ5IGRlZmF1bHQgYWxsIDEwIHN5bWJvbHMgYXJlIHBlcm11dGVkLCBzaW5jZQpBUkMtQUdJLTIncyBoYXJkZXIgdGFza3Mgb2Z0ZW4gZG8gbm90IHVzZSAwIGFzIGJhY2tncm91bmQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJhbmRvbQoKZnJvbSAuLmlvLmdyaWQgaW1wb3J0IE5VTV9DT0xPUlMsIEdyaWQKClBlcm0gPSB0dXBsZVtpbnQsIC4uLl0KCklERU5USVRZX1BFUk06IFBlcm0gPSB0dXBsZShyYW5nZShOVU1fQ09MT1JTKSkKCgpkZWYgcmFuZG9tX3Blcm0oc2VlZDogaW50LCBrZWVwX3plcm86IGJvb2wgPSBGYWxzZSkgLT4gUGVybToKICAgICIiIkEgcmFuZG9tIHN5bWJvbCBwZXJtdXRhdGlvbiwgZGV0ZXJtaW5pc3RpYyBpbiBgc2VlZGAuCgogICAgSWYgYGtlZXBfemVyb2AsIHN5bWJvbCAwIG1hcHMgdG8gaXRzZWxmIGFuZCBvbmx5IDEtOSBhcmUgc2h1ZmZsZWQuCiAgICAiIiIKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIGlmIGtlZXBfemVybzoKICAgICAgICBtb3ZhYmxlID0gbGlzdChyYW5nZSgxLCBOVU1fQ09MT1JTKSkKICAgICAgICBzaHVmZmxlZCA9IG1vdmFibGVbOl0KICAgICAgICBybmcuc2h1ZmZsZShzaHVmZmxlZCkKICAgICAgICBwZXJtID0gWzBdICsgWzBdICogKE5VTV9DT0xPUlMgLSAxKQogICAgICAgIGZvciBzcmMsIGRzdCBpbiB6aXAobW92YWJsZSwgc2h1ZmZsZWQsIHN0cmljdD1GYWxzZSk6CiAgICAgICAgICAgIHBlcm1bc3JjXSA9IGRzdAogICAgICAgIHJldHVybiB0dXBsZShwZXJtKQogICAgc3ltYm9scyA9IGxpc3QocmFuZ2UoTlVNX0NPTE9SUykpCiAgICBzaHVmZmxlZCA9IHN5bWJvbHNbOl0KICAgIHJuZy5zaHVmZmxlKHNodWZmbGVkKQogICAgcmV0dXJuIHR1cGxlKHNodWZmbGVkKQoKCmRlZiBpbnZlcnRfcGVybShwZXJtOiBQZXJtKSAtPiBQZXJtOgogICAgIiIiVGhlIGludmVyc2UgcGVybXV0YXRpb24sIHN1Y2ggdGhhdCBpbnZlcnRfcGVybShwZXJtKVtwZXJtW3NdXSA9PSBzLiIiIgogICAgaW52ID0gWzBdICogTlVNX0NPTE9SUwogICAgZm9yIHNyYywgZHN0IGluIGVudW1lcmF0ZShwZXJtKToKICAgICAgICBpbnZbZHN0XSA9IHNyYwogICAgcmV0dXJuIHR1cGxlKGludikKCgpkZWYgYXBwbHkocGVybTogUGVybSwgZ3JpZDogR3JpZCkgLT4gR3JpZDoKICAgICIiIlJlY29sb3VyIGEgZ3JpZCBieSBtYXBwaW5nIGV2ZXJ5IHN5bWJvbCBzIC0+IHBlcm1bc10uIiIiCiAgICByZXR1cm4gdHVwbGUodHVwbGUocGVybVtjXSBmb3IgYyBpbiByb3cpIGZvciByb3cgaW4gZ3JpZCkKCgpkZWYgaW52ZXJ0KHBlcm06IFBlcm0sIGdyaWQ6IEdyaWQpIC0+IEdyaWQ6CiAgICAiIiJVbmRvIGEgcmVjb2xvdXJpbmcgKGFwcGx5IHRoZSBpbnZlcnNlIHBlcm11dGF0aW9uKS4iIiIKICAgIHJldHVybiBhcHBseShpbnZlcnRfcGVybShwZXJtKSwgZ3JpZCkK",
"src/arc/augment/symmetry.py": "IiIiRDQgZGloZWRyYWwgc3ltbWV0cmllcyBvbiBncmlkcywgZWFjaCB3aXRoIGFuIGV4YWN0IGludmVyc2UuCgpUaGUgOCBlbGVtZW50cyBvZiB0aGUgc3ltbWV0cnkgZ3JvdXAgb2YgdGhlIHNxdWFyZS4gQXVnbWVudGF0aW9uIHRyYW5zZm9ybXMgYQp0YXNrJ3MgaW5wdXRzOyBhdCBpbmZlcmVuY2Ugd2UgdHJhbnNmb3JtIHRoZSBpbnB1dCwgcHJlZGljdCwgdGhlbiBhcHBseSB0aGUKSU5WRVJTRSB0cmFuc2Zvcm0gdG8gYnJpbmcgdGhlIHByZWRpY3Rpb24gYmFjayB0byB0aGUgY2Fub25pY2FsIGZyYW1lIGJlZm9yZQp2b3RpbmcuIEV4YWN0IGludmVydGliaWxpdHkgaXMgcmVxdWlyZWQg4oCUIGl0IGlzIGVuZm9yY2VkIGJ5IHJvdW5kLXRyaXAgdGVzdHMuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC4uaW8uZ3JpZCBpbXBvcnQgR3JpZCwgZnJvbV9udW1weSwgdG9fbnVtcHkKCiMgRm9yd2FyZCB0cmFuc2Zvcm1zIG9uIG51bXB5IGFycmF5cy4gbnAucm90OTAgcm90YXRlcyBjb3VudGVyLWNsb2Nrd2lzZS4KX0ZPUldBUkQgPSB7CiAgICAiaWRlbnRpdHkiOiBsYW1iZGEgYTogYSwKICAgICJyb3Q5MCI6IGxhbWJkYSBhOiBucC5yb3Q5MChhLCAxKSwKICAgICJyb3QxODAiOiBsYW1iZGEgYTogbnAucm90OTAoYSwgMiksCiAgICAicm90MjcwIjogbGFtYmRhIGE6IG5wLnJvdDkwKGEsIDMpLAogICAgImZsaXBfaCI6IGxhbWJkYSBhOiBucC5mbGlwbHIoYSksCiAgICAiZmxpcF92IjogbGFtYmRhIGE6IG5wLmZsaXB1ZChhKSwKICAgICJ0cmFuc3Bvc2UiOiBsYW1iZGEgYTogYS5ULAogICAgImFudGlfdHJhbnNwb3NlIjogbGFtYmRhIGE6IG5wLnJvdDkwKG5wLmZsaXBscihhKSwgMSksCn0KCiMgSW52ZXJzZSBlbGVtZW50IG9mIGVhY2ggdHJhbnNmb3JtIHdpdGhpbiBENC4KX0lOVkVSU0VfTkFNRSA9IHsKICAgICJpZGVudGl0eSI6ICJpZGVudGl0eSIsCiAgICAicm90OTAiOiAicm90MjcwIiwKICAgICJyb3QxODAiOiAicm90MTgwIiwKICAgICJyb3QyNzAiOiAicm90OTAiLAogICAgImZsaXBfaCI6ICJmbGlwX2giLAogICAgImZsaXBfdiI6ICJmbGlwX3YiLAogICAgInRyYW5zcG9zZSI6ICJ0cmFuc3Bvc2UiLAogICAgImFudGlfdHJhbnNwb3NlIjogImFudGlfdHJhbnNwb3NlIiwKfQoKRDRfTkFNRVMgPSB0dXBsZShfRk9SV0FSRC5rZXlzKCkpCgoKZGVmIGFwcGx5KG5hbWU6IHN0ciwgZ3JpZDogR3JpZCkgLT4gR3JpZDoKICAgICIiIkFwcGx5IHRoZSBuYW1lZCBENCB0cmFuc2Zvcm0gdG8gYSBncmlkLiIiIgogICAgcmV0dXJuIGZyb21fbnVtcHkobnAuYXNjb250aWd1b3VzYXJyYXkoX0ZPUldBUkRbbmFtZV0odG9fbnVtcHkoZ3JpZCkpKSkKCgpkZWYgaW52ZXJ0KG5hbWU6IHN0ciwgZ3JpZDogR3JpZCkgLT4gR3JpZDoKICAgICIiIkFwcGx5IHRoZSBpbnZlcnNlIG9mIHRoZSBuYW1lZCBENCB0cmFuc2Zvcm0gdG8gYSBncmlkLiIiIgogICAgcmV0dXJuIGFwcGx5KF9JTlZFUlNFX05BTUVbbmFtZV0sIGdyaWQpCgoKZGVmIGludmVyc2VfbmFtZShuYW1lOiBzdHIpIC0+IHN0cjoKICAgIHJldHVybiBfSU5WRVJTRV9OQU1FW25hbWVdCg==",
"src/arc/augment/task_aug.py": "IiIiVGFzay1sZXZlbCBhdWdtZW50YXRpb246IGNvbXBvc2UgRDQgc3ltbWV0cnkgKyBjb2xvdXIgcGVybXV0YXRpb24sIGFuZApyZWZvcm11bGF0ZSBhIHNpbmdsZSB0YXNrIGludG8gbWFueSB0cmFpbmluZyB2aWV3cyAobGVhdmUtb25lLW91dCwgc2h1ZmZsZSkuCgpBIGBUYXNrQXVnYCBhcHBsaWVzIG9uZSBjb25zaXN0ZW50IHRyYW5zZm9ybSB0byBldmVyeSBncmlkIGluIGEgdGFzayBzbyB0aGUKdGFzaydzIHVuZGVybHlpbmcgcnVsZSBpcyBwcmVzZXJ2ZWQuIEF0IGluZmVyZW5jZSB3ZSBhdWdtZW50IHRoZSB0ZXN0IGlucHV0LApwcmVkaWN0IGluIHRoZSBhdWdtZW50ZWQgZnJhbWUsIHRoZW4gYGludmVydF9ncmlkYCB0byByZWNvdmVyIHRoZSBjYW5vbmljYWwKYW5zd2VyIGZvciB2b3RpbmcuIFN5bW1ldHJ5IGFuZCBjb2xvdXIgY29tbXV0ZSwgc28gaW52ZXJ0IG9yZGVyIGlzIGltbWF0ZXJpYWw7CndlIGtlZXAgYSBmaXhlZCBjb252ZW50aW9uIGZvciBjbGFyaXR5LgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCByYW5kb20KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgpmcm9tIC4uaW8uZ3JpZCBpbXBvcnQgR3JpZApmcm9tIC4uaW8ubG9hZGVyIGltcG9ydCBQYWlyLCBUYXNrCmZyb20gLiBpbXBvcnQgY29sb3IsIHN5bW1ldHJ5CmZyb20gLmNvbG9yIGltcG9ydCBQZXJtCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgVGFza0F1ZzoKICAgICIiIkEgY29tcG9zZWQsIGludmVydGlibGUgdGFzayBhdWdtZW50YXRpb24uIiIiCgogICAgc3ltX25hbWU6IHN0cgogICAgcGVybTogUGVybQoKICAgIGRlZiBhcHBseV9ncmlkKHNlbGYsIGdyaWQ6IEdyaWQpIC0+IEdyaWQ6CiAgICAgICAgcmV0dXJuIGNvbG9yLmFwcGx5KHNlbGYucGVybSwgc3ltbWV0cnkuYXBwbHkoc2VsZi5zeW1fbmFtZSwgZ3JpZCkpCgogICAgZGVmIGludmVydF9ncmlkKHNlbGYsIGdyaWQ6IEdyaWQpIC0+IEdyaWQ6CiAgICAgICAgcmV0dXJuIHN5bW1ldHJ5LmludmVydChzZWxmLnN5bV9uYW1lLCBjb2xvci5pbnZlcnQoc2VsZi5wZXJtLCBncmlkKSkKCiAgICBkZWYgYXBwbHlfcGFpcihzZWxmLCBwYWlyOiBQYWlyKSAtPiBQYWlyOgogICAgICAgIHJldHVybiBQYWlyKAogICAgICAgICAgICBpbnB1dD1zZWxmLmFwcGx5X2dyaWQocGFpci5pbnB1dCksCiAgICAgICAgICAgIG91dHB1dD1zZWxmLmFwcGx5X2dyaWQocGFpci5vdXRwdXQpIGlmIHBhaXIub3V0cHV0IGlzIG5vdCBOb25lIGVsc2UgTm9uZSwKICAgICAgICApCgogICAgZGVmIGFwcGx5X3Rhc2soc2VsZiwgdGFzazogVGFzaykgLT4gVGFzazoKICAgICAgICByZXR1cm4gVGFzaygKICAgICAgICAgICAgdGFza19pZD10YXNrLnRhc2tfaWQsCiAgICAgICAgICAgIHRyYWluPXR1cGxlKHNlbGYuYXBwbHlfcGFpcihwKSBmb3IgcCBpbiB0YXNrLnRyYWluKSwKICAgICAgICAgICAgdGVzdD10dXBsZShzZWxmLmFwcGx5X3BhaXIocCkgZm9yIHAgaW4gdGFzay50ZXN0KSwKICAgICAgICApCgoKSURFTlRJVFlfQVVHID0gVGFza0F1ZyhzeW1fbmFtZT0iaWRlbnRpdHkiLCBwZXJtPWNvbG9yLklERU5USVRZX1BFUk0pCgoKZGVmIHJhbmRvbV9hdWcoc2VlZDogaW50LCBrZWVwX3plcm86IGJvb2wgPSBGYWxzZSkgLT4gVGFza0F1ZzoKICAgICIiIkEgcmFuZG9tIGNvbXBvc2VkIGF1Z21lbnRhdGlvbiwgZGV0ZXJtaW5pc3RpYyBpbiBgc2VlZGAuIiIiCiAgICBybmcgPSByYW5kb20uUmFuZG9tKHNlZWQpCiAgICBzeW1fbmFtZSA9IHJuZy5jaG9pY2Uoc3ltbWV0cnkuRDRfTkFNRVMpCiAgICBwZXJtID0gY29sb3IucmFuZG9tX3Blcm0oc2VlZD1ybmcucmFuZHJhbmdlKDEgPDwgMzApLCBrZWVwX3plcm89a2VlcF96ZXJvKQogICAgcmV0dXJuIFRhc2tBdWcoc3ltX25hbWU9c3ltX25hbWUsIHBlcm09cGVybSkKCgpkZWYgZGlzdGluY3RfYXVncyhuOiBpbnQsIHNlZWQ6IGludCA9IDAsIGtlZXBfemVybzogYm9vbCA9IEZhbHNlKSAtPiBsaXN0W1Rhc2tBdWddOgogICAgIiIiVXAgdG8gYG5gIGRpc3RpbmN0IGF1Z21lbnRhdGlvbnMgKGFsd2F5cyBpbmNsdWRlcyB0aGUgaWRlbnRpdHkgZmlyc3QpLiIiIgogICAgc2Vlbjogc2V0W3R1cGxlW3N0ciwgUGVybV1dID0gc2V0KCkKICAgIG91dDogbGlzdFtUYXNrQXVnXSA9IFtdCiAgICBpZGVudF9rZXkgPSAoSURFTlRJVFlfQVVHLnN5bV9uYW1lLCBJREVOVElUWV9BVUcucGVybSkKICAgIHNlZW4uYWRkKGlkZW50X2tleSkKICAgIG91dC5hcHBlbmQoSURFTlRJVFlfQVVHKQogICAgaSA9IDAKICAgIHdoaWxlIGxlbihvdXQpIDwgbiBhbmQgaSA8IG4gKiA1MDoKICAgICAgICBhdWcgPSByYW5kb21fYXVnKHNlZWQ9c2VlZCAqIDFfMDAwXzAwMyArIGksIGtlZXBfemVybz1rZWVwX3plcm8pCiAgICAgICAga2V5ID0gKGF1Zy5zeW1fbmFtZSwgYXVnLnBlcm0pCiAgICAgICAgaWYga2V5IG5vdCBpbiBzZWVuOgogICAgICAgICAgICBzZWVuLmFkZChrZXkpCiAgICAgICAgICAgIG91dC5hcHBlbmQoYXVnKQogICAgICAgIGkgKz0gMQogICAgcmV0dXJuIG91dAoKCmRlZiBsZWF2ZV9vbmVfb3V0KHRhc2s6IFRhc2spIC0+IGxpc3RbdHVwbGVbdHVwbGVbUGFpciwgLi4uXSwgUGFpcl1dOgogICAgIiIiUmVmb3JtdWxhdGUgYSB0YXNrIGludG8gKHN1cHBvcnRfcGFpcnMsIHF1ZXJ5X3BhaXIpIHZpZXdzLgoKICAgIEVhY2ggZGVtb25zdHJhdGlvbiBwYWlyIGJlY29tZXMgdGhlIHF1ZXJ5IG9uY2UsIHdpdGggdGhlIHJlbWFpbmluZyBwYWlycyBhcwogICAgc3VwcG9ydC4gVGhpcyB0dXJucyBhIHNpbmdsZSB0YXNrJ3Mgc3VwZXJ2aXNpb24gaW50byBOIHNlbGYtc3VwZXJ2aXNlZAogICAgcHJlZGljdGlvbiBwcm9ibGVtcyDigJQgdGhlIGNvcmUgZGF0YSBzb3VyY2UgZm9yIHRlc3QtdGltZSB0cmFpbmluZy4KICAgICIiIgogICAgdmlld3MgPSBbXQogICAgdHJhaW4gPSB0YXNrLnRyYWluCiAgICBpZiBsZW4odHJhaW4pIDwgMjoKICAgICAgICByZXR1cm4gdmlld3MKICAgIGZvciBpLCBxdWVyeSBpbiBlbnVtZXJhdGUodHJhaW4pOgogICAgICAgIHN1cHBvcnQgPSB0cmFpbls6aV0gKyB0cmFpbltpICsgMSA6XQogICAgICAgIHZpZXdzLmFwcGVuZCgoc3VwcG9ydCwgcXVlcnkpKQogICAgcmV0dXJuIHZpZXdzCgoKZGVmIHNodWZmbGVfdHJhaW4odGFzazogVGFzaywgc2VlZDogaW50KSAtPiBUYXNrOgogICAgIiIiUmV0dXJuIGEgY29weSBvZiBgdGFza2Agd2l0aCBkZW1vbnN0cmF0aW9uIHBhaXJzIHJlb3JkZXJlZC4iIiIKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIG9yZGVyID0gbGlzdCh0YXNrLnRyYWluKQogICAgcm5nLnNodWZmbGUob3JkZXIpCiAgICByZXR1cm4gVGFzayh0YXNrX2lkPXRhc2sudGFza19pZCwgdHJhaW49dHVwbGUob3JkZXIpLCB0ZXN0PXRhc2sudGVzdCkK",
"src/arc/config.py": "IiIiRW52aXJvbm1lbnQgY29uZmlndXJhdGlvbiDigJQgdGhlIE9OTFkgZW52aXJvbm1lbnQtYXdhcmUgbW9kdWxlIGluIHRoZSBjb2RlYmFzZS4KClJlc29sdmVzIGRhdGEvb3V0cHV0IHBhdGhzIGFuZCB0aGUgcnVudGltZSBtb2RlIChTTU9LRSBvbiBhIGxvY2FsIENQVSBib3ggdnMKS0FHR0xFIG9uIHRoZSBMNHg0IEdQVSkuIEV2ZXJ5dGhpbmcgZWxzZSBpbiBgYXJjYCByZWNlaXZlcyBwYXRocy9vYmplY3RzIGFuZCBpcwplbnZpcm9ubWVudC1hZ25vc3RpYywgc28gdGhlIGlkZW50aWNhbCBjb2RlIHBhdGggcnVucyBpbiBib3RoIHBsYWNlcy4KCkRldGVjdGlvbiBvcmRlciAoZmlyc3QgbWF0Y2ggd2lucyk6CiAgMS4gRXhwbGljaXQgZW52IHZhcnMgKGBBUkNfREFUQV9ESVJgLCBgQVJDX09VVFBVVF9ESVJgLCBgQVJDX01PREVgKS4KICAyLiBLYWdnbGUsIGlmIGAva2FnZ2xlL2lucHV0YCBleGlzdHMuCiAgMy4gTG9jYWwgZmFsbGJhY2sgKHRoZSB1c2VyJ3MgRG93bmxvYWRzIGNvcHkgb2YgdGhlIGNvbXBldGl0aW9uIGRhdGEpLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgojIC0tLS0gUnVudGltZSBtb2RlcyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KTU9ERV9TTU9LRSA9ICJTTU9LRSIgICAjIGxvY2FsIENQVSwgdGlueS9tb2NrIG1vZGVsLCBmYXN0IHBsdW1iaW5nIGNoZWNrcwpNT0RFX0tBR0dMRSA9ICJLQUdHTEUiICAjIG9mZmxpbmUgTDR4NCBHUFUsIHJlYWwgbW9kZWwgKyBUVFQKCiMgS2FnZ2xlIG1vdW50cyBjb21wZXRpdGlvbiBkYXRhIHJlYWQtb25seSB1bmRlciB0aGlzIGRpcmVjdG9yeS4KX0tBR0dMRV9JTlBVVCA9IFBhdGgoIi9rYWdnbGUvaW5wdXQiKQpfS0FHR0xFX0RBVEEgPSBfS0FHR0xFX0lOUFVUIC8gImFyYy1wcml6ZS0yMDI2LWFyYy1hZ2ktMiIKX0tBR0dMRV9XT1JLSU5HID0gUGF0aCgiL2thZ2dsZS93b3JraW5nIikKCiMgTG9jYWwgZmFsbGJhY2s6IHRoZSB1c2VyJ3MgZG93bmxvYWRlZCBjb21wZXRpdGlvbiBidW5kbGUuCl9MT0NBTF9EQVRBID0gUGF0aCgKICAgICIvVXNlcnMvc2ViYXN0aWVuaGVucnkvRG93bmxvYWRzL2FyYy1wcml6ZS0yMDI2LWFyYy1hZ2ktMiIKKQpfUkVQT19ST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KX0xPQ0FMX09VVFBVVCA9IF9SRVBPX1JPT1QgLyAiYXJ0aWZhY3RzIgoKIyBDYW5vbmljYWwgZmlsZSBuYW1lcyB3aXRoaW4gYSBkYXRhIGRpcmVjdG9yeS4KRklMRV9OQU1FUyA9IHsKICAgICJ0cmFpbmluZ19jaGFsbGVuZ2VzIjogImFyYy1hZ2lfdHJhaW5pbmdfY2hhbGxlbmdlcy5qc29uIiwKICAgICJ0cmFpbmluZ19zb2x1dGlvbnMiOiAiYXJjLWFnaV90cmFpbmluZ19zb2x1dGlvbnMuanNvbiIsCiAgICAiZXZhbHVhdGlvbl9jaGFsbGVuZ2VzIjogImFyYy1hZ2lfZXZhbHVhdGlvbl9jaGFsbGVuZ2VzLmpzb24iLAogICAgImV2YWx1YXRpb25fc29sdXRpb25zIjogImFyYy1hZ2lfZXZhbHVhdGlvbl9zb2x1dGlvbnMuanNvbiIsCiAgICAidGVzdF9jaGFsbGVuZ2VzIjogImFyYy1hZ2lfdGVzdF9jaGFsbGVuZ2VzLmpzb24iLAogICAgInNhbXBsZV9zdWJtaXNzaW9uIjogInNhbXBsZV9zdWJtaXNzaW9uLmpzb24iLAp9CgoKZGVmIF9kZXRlY3RfbW9kZSgpIC0+IHN0cjoKICAgIGZvcmNlZCA9IG9zLmVudmlyb24uZ2V0KCJBUkNfTU9ERSIpCiAgICBpZiBmb3JjZWQ6CiAgICAgICAgcmV0dXJuIGZvcmNlZC51cHBlcigpCiAgICByZXR1cm4gTU9ERV9LQUdHTEUgaWYgX0tBR0dMRV9JTlBVVC5leGlzdHMoKSBlbHNlIE1PREVfU01PS0UKCgpkZWYgX2ZpbmRfa2FnZ2xlX2RhdGFfZGlyKCkgLT4gUGF0aCB8IE5vbmU6CiAgICAiIiJMb2NhdGUgdGhlIGNvbXBldGl0aW9uIGRhdGEgdW5kZXIgL2thZ2dsZS9pbnB1dC4KCiAgICBLYWdnbGUgbW91bnRzIHRoZSBjb21wZXRpdGlvbiB1bmRlciBhIGZvbGRlciBuYW1lZCBhZnRlciBpdHMgc2x1ZywgYnV0IHJhdGhlcgogICAgdGhhbiB0cnVzdCBhIHNpbmdsZSBoYXJkY29kZWQgbmFtZSB3ZSBwcmVmZXIgdGhlIGRvY3VtZW50ZWQgc2x1ZyBhbmQKICAgIG90aGVyd2lzZSBzY2FuIGZvciB3aGljaGV2ZXIgbW91bnRlZCBmb2xkZXIgYWN0dWFsbHkgY29udGFpbnMgdGhlCiAgICB0ZXN0LWNoYWxsZW5nZXMgZmlsZSAoYSBjb3VwbGUgb2YgbmVzdGluZyBsZXZlbHMgZGVlcCkuIFJldHVybnMgTm9uZSBpZgogICAgbm90aGluZyBtYXRjaGluZyBpcyBtb3VudGVkLgogICAgIiIiCiAgICBpZiBub3QgX0tBR0dMRV9JTlBVVC5leGlzdHMoKToKICAgICAgICByZXR1cm4gTm9uZQogICAgaWYgX0tBR0dMRV9EQVRBLmV4aXN0cygpOgogICAgICAgIHJldHVybiBfS0FHR0xFX0RBVEEKICAgIGZuYW1lID0gRklMRV9OQU1FU1sidGVzdF9jaGFsbGVuZ2VzIl0KICAgIGZvciBwYXR0ZXJuIGluIChmIiove2ZuYW1lfSIsIGYiKi8qL3tmbmFtZX0iKToKICAgICAgICBmb3IgbWF0Y2ggaW4gc29ydGVkKF9LQUdHTEVfSU5QVVQuZ2xvYihwYXR0ZXJuKSk6CiAgICAgICAgICAgIHJldHVybiBtYXRjaC5wYXJlbnQKICAgIHJldHVybiBOb25lCgoKZGVmIF9kZXRlY3RfZGF0YV9kaXIoKSAtPiBQYXRoOgogICAgb3ZlcnJpZGUgPSBvcy5lbnZpcm9uLmdldCgiQVJDX0RBVEFfRElSIikKICAgIGlmIG92ZXJyaWRlOgogICAgICAgIHJldHVybiBQYXRoKG92ZXJyaWRlKQogICAgaWYgX0tBR0dMRV9JTlBVVC5leGlzdHMoKToKICAgICAgICAjIE9uIEthZ2dsZTogdXNlIHRoZSBtb3VudGVkIGNvbXBldGl0aW9uIGRhdGEgd2hlcmV2ZXIgaXQgaXM7IG5ldmVyIGZhbGwKICAgICAgICAjIGJhY2sgdG8gYSBkZXZlbG9wZXItbWFjaGluZSBwYXRoICh0aGF0IHByb2R1Y2VzIGEgYmFmZmxpbmcKICAgICAgICAjIEZpbGVOb3RGb3VuZEVycm9yIHBvaW50aW5nIGF0IGEgYm94IHRoYXQgaXNuJ3QgZXZlbiBydW5uaW5nKS4gSWYgdGhlCiAgICAgICAgIyBkYXRhIGlzbid0IGF0dGFjaGVkIHlldCwgcmV0dXJuIHRoZSBkb2N1bWVudGVkIHNsdWcgc28gdGhlIGV2ZW50dWFsCiAgICAgICAgIyBlcnJvciByZWZlcmVuY2VzIGEgcmVhbCAva2FnZ2xlIHBhdGgg4oCUIHRoZSBlbnRyeXBvaW50IHR1cm5zIHRoYXQgaW50bwogICAgICAgICMgYW4gYWN0aW9uYWJsZSAiYXR0YWNoIHRoZSBjb21wZXRpdGlvbiBkYXRhIiBtZXNzYWdlLgogICAgICAgIGZvdW5kID0gX2ZpbmRfa2FnZ2xlX2RhdGFfZGlyKCkKICAgICAgICByZXR1cm4gZm91bmQgaWYgZm91bmQgaXMgbm90IE5vbmUgZWxzZSBfS0FHR0xFX0RBVEEKICAgIHJldHVybiBfTE9DQUxfREFUQQoKCmRlZiBfZGV0ZWN0X291dHB1dF9kaXIoKSAtPiBQYXRoOgogICAgb3ZlcnJpZGUgPSBvcy5lbnZpcm9uLmdldCgiQVJDX09VVFBVVF9ESVIiKQogICAgaWYgb3ZlcnJpZGU6CiAgICAgICAgcmV0dXJuIFBhdGgob3ZlcnJpZGUpCiAgICBpZiBfS0FHR0xFX1dPUktJTkcuZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIF9LQUdHTEVfV09SS0lORwogICAgcmV0dXJuIF9MT0NBTF9PVVRQVVQKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBDb25maWc6CiAgICAiIiJSZXNvbHZlZCwgaW1tdXRhYmxlIHZpZXcgb2YgdGhlIHJ1bnRpbWUgZW52aXJvbm1lbnQuIiIiCgogICAgbW9kZTogc3RyCiAgICBkYXRhX2RpcjogUGF0aAogICAgb3V0cHV0X2RpcjogUGF0aAoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGlzX2thZ2dsZShzZWxmKSAtPiBib29sOgogICAgICAgIHJldHVybiBzZWxmLm1vZGUgPT0gTU9ERV9LQUdHTEUKCiAgICBAcHJvcGVydHkKICAgIGRlZiBpc19zbW9rZShzZWxmKSAtPiBib29sOgogICAgICAgIHJldHVybiBzZWxmLm1vZGUgPT0gTU9ERV9TTU9LRQoKICAgIGRlZiBjaGFsbGVuZ2VzX3BhdGgoc2VsZiwgc3BsaXQ6IHN0cikgLT4gUGF0aDoKICAgICAgICAiIiJzcGxpdCBpbiB7J3RyYWluaW5nJywgJ2V2YWx1YXRpb24nLCAndGVzdCd9LiIiIgogICAgICAgIHJldHVybiBzZWxmLmRhdGFfZGlyIC8gRklMRV9OQU1FU1tmIntzcGxpdH1fY2hhbGxlbmdlcyJdCgogICAgZGVmIHNvbHV0aW9uc19wYXRoKHNlbGYsIHNwbGl0OiBzdHIpIC0+IFBhdGg6CiAgICAgICAgIiIic3BsaXQgaW4geyd0cmFpbmluZycsICdldmFsdWF0aW9uJ30gKHRlc3Qgc29sdXRpb25zIGFyZSBoaWRkZW4pLiIiIgogICAgICAgIHJldHVybiBzZWxmLmRhdGFfZGlyIC8gRklMRV9OQU1FU1tmIntzcGxpdH1fc29sdXRpb25zIl0KCiAgICBAcHJvcGVydHkKICAgIGRlZiBzYW1wbGVfc3VibWlzc2lvbl9wYXRoKHNlbGYpIC0+IFBhdGg6CiAgICAgICAgcmV0dXJuIHNlbGYuZGF0YV9kaXIgLyBGSUxFX05BTUVTWyJzYW1wbGVfc3VibWlzc2lvbiJdCgogICAgQHByb3BlcnR5CiAgICBkZWYgc3VibWlzc2lvbl9wYXRoKHNlbGYpIC0+IFBhdGg6CiAgICAgICAgcmV0dXJuIHNlbGYub3V0cHV0X2RpciAvICJzdWJtaXNzaW9uLmpzb24iCgoKZGVmIGdldF9jb25maWcoKSAtPiBDb25maWc6CiAgICAiIiJSZXNvbHZlIHRoZSBhY3RpdmUgY29uZmlndXJhdGlvbiBmcm9tIHRoZSBlbnZpcm9ubWVudC4iIiIKICAgIGNmZyA9IENvbmZpZygKICAgICAgICBtb2RlPV9kZXRlY3RfbW9kZSgpLAogICAgICAgIGRhdGFfZGlyPV9kZXRlY3RfZGF0YV9kaXIoKSwKICAgICAgICBvdXRwdXRfZGlyPV9kZXRlY3Rfb3V0cHV0X2RpcigpLAogICAgKQogICAgY2ZnLm91dHB1dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcmV0dXJuIGNmZwoKCiMgLS0tLSBHbG9iYWwgaW5mZXJlbmNlIGJ1ZGdldCAodXNlZCBieSB0aGUgdGltZSB3YXRjaGRvZyBpbiBwaXBlbGluZS5weSkgLS0tLQojIDI0MCB0ZXN0IHRhc2tzIC8gMTIgaCDiiYggMyBtaW4vdGFzay4gS2VlcCBtYXJnaW4gZm9yIG1vZGVsIGxvYWQgKyBJL08uClRPVEFMX1JVTlRJTUVfQlVER0VUX1MgPSAxMS4wICogMzYwMCAgIyBsZWF2ZSB+MWggaGVhZHJvb20gdW5kZXIgdGhlIDEyaCBjYXAKREVGQVVMVF9QRVJfVEFTS19CVURHRVRfUyA9IDE1MC4wICAgICAjIDIuNSBtaW4gdGFyZ2V0IHBlciB0YXNrCg==",
"src/arc/eval/__init__.py": "",
"src/arc/eval/harness.py": "IiIiRXZhbHVhdGlvbiBoYXJuZXNzOiBsb2FkIGEgc3BsaXQsIHJ1biB0aGUgcGlwZWxpbmUsIHNjb3JlIGV4YWN0LW1hdGNoIHRvcC0yLgoKVXNlZCBib3RoIGZvciBsb2NhbCBkZXZlbG9wbWVudCAoc21va2UgcnVucyBvdmVyIHRoZSBwdWJsaWMgZXZhbCBzcGxpdCkgYW5kIGFzCnRoZSBwZXItbWlsZXN0b25lIHJlZ3Jlc3Npb24gY2hlY2suIFNjb3JpbmcgcmVxdWlyZXMgYSBzb2x1dGlvbnMgZmlsZSwgc28gaXQKd29ya3Mgb24gJ3RyYWluaW5nJyBhbmQgJ2V2YWx1YXRpb24nIGJ1dCBub3QgJ3Rlc3QnIChzb2x1dGlvbnMgaGlkZGVuKS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIC4uY29uZmlnIGltcG9ydCBDb25maWcsIGdldF9jb25maWcKZnJvbSAuLmlvLmxvYWRlciBpbXBvcnQgbG9hZF9jaGFsbGVuZ2VzLCBsb2FkX3NvbHV0aW9ucwpmcm9tIC4ucGlwZWxpbmUgaW1wb3J0IHJ1biBhcyBydW5fcGlwZWxpbmUKZnJvbSAuLnNvbHZlcnMuYmFzZSBpbXBvcnQgU29sdmVyCmZyb20gLm1ldHJpY3MgaW1wb3J0IHNjb3JlX3ByZWRpY3Rpb25zCgoKZGVmIGxvYWRfc3BsaXQoc3BsaXQ6IHN0ciwgY2ZnOiBDb25maWcgfCBOb25lID0gTm9uZSwgbGltaXQ6IGludCB8IE5vbmUgPSBOb25lKToKICAgICIiIkxvYWQgKHRhc2tzLCBzb2x1dGlvbnN8Tm9uZSkgZm9yIGEgc3BsaXQsIG9wdGlvbmFsbHkgdHJ1bmNhdGVkIHRvIGBsaW1pdGAuIiIiCiAgICBjZmcgPSBjZmcgb3IgZ2V0X2NvbmZpZygpCiAgICB0YXNrcyA9IGxvYWRfY2hhbGxlbmdlcyhjZmcuY2hhbGxlbmdlc19wYXRoKHNwbGl0KSkKICAgIGlmIGxpbWl0IGlzIG5vdCBOb25lOgogICAgICAgIHRhc2tzID0gZGljdChsaXN0KHRhc2tzLml0ZW1zKCkpWzpsaW1pdF0pCiAgICBzb2x1dGlvbnMgPSBOb25lCiAgICBpZiBzcGxpdCBpbiAoInRyYWluaW5nIiwgImV2YWx1YXRpb24iKToKICAgICAgICBzb2wgPSBsb2FkX3NvbHV0aW9ucyhjZmcuc29sdXRpb25zX3BhdGgoc3BsaXQpKQogICAgICAgIHNvbHV0aW9ucyA9IHtrOiB2IGZvciBrLCB2IGluIHNvbC5pdGVtcygpIGlmIGsgaW4gdGFza3N9CiAgICByZXR1cm4gdGFza3MsIHNvbHV0aW9ucwoKCmRlZiBldmFsdWF0ZSgKICAgIHNwbGl0OiBzdHIgPSAiZXZhbHVhdGlvbiIsCiAgICBzb2x2ZXJzOiBsaXN0W1NvbHZlcl0gfCBOb25lID0gTm9uZSwKICAgIGxpbWl0OiBpbnQgfCBOb25lID0gTm9uZSwKICAgIHBlcl90YXNrX2J1ZGdldF9zOiBmbG9hdCA9IDEwLjAsCiAgICB2ZXJib3NlOiBib29sID0gRmFsc2UsCikgLT4gZGljdDoKICAgICIiIlJ1biB0aGUgcGlwZWxpbmUgb3ZlciBhIHNwbGl0IGFuZCByZXR1cm4gYSBzY29yZSBzdW1tYXJ5LiIiIgogICAgdGFza3MsIHNvbHV0aW9ucyA9IGxvYWRfc3BsaXQoc3BsaXQsIGxpbWl0PWxpbWl0KQogICAgcHJlZGljdGlvbnMgPSBydW5fcGlwZWxpbmUoCiAgICAgICAgdGFza3MsCiAgICAgICAgc29sdmVycz1zb2x2ZXJzLAogICAgICAgIG91dHB1dF9wYXRoPU5vbmUsCiAgICAgICAgcGVyX3Rhc2tfYnVkZ2V0X3M9cGVyX3Rhc2tfYnVkZ2V0X3MsCiAgICAgICAgdmVyYm9zZT12ZXJib3NlLAogICAgKQogICAgaWYgc29sdXRpb25zIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIHsic3BsaXQiOiBzcGxpdCwgIm51bV90YXNrcyI6IGxlbih0YXNrcyksICJzY29yZWQiOiBGYWxzZX0KCiAgICBzdW1tYXJ5ID0gc2NvcmVfcHJlZGljdGlvbnMocHJlZGljdGlvbnMsIHNvbHV0aW9ucykKICAgIHN1bW1hcnkudXBkYXRlKHsic3BsaXQiOiBzcGxpdCwgInNjb3JlZCI6IFRydWV9KQogICAgcmV0dXJuIHN1bW1hcnkK",
"src/arc/eval/metrics.py": "IiIiVGhlIGNvbXBldGl0aW9uIG1ldHJpYzogdG9wLTIgZXhhY3QtbWF0Y2gsIGF2ZXJhZ2VkIG92ZXIgdGVzdCBvdXRwdXRzLgoKRm9yIGVhY2ggdGVzdCBvdXRwdXQgdGhlcmUgaXMgb25lIGdyb3VuZC10cnV0aCBncmlkLiBBIHRhc2sgb3V0cHV0IHNjb3JlcyAxIGlmCkVJVEhFUiBvZiB0aGUgdHdvIGF0dGVtcHRzIG1hdGNoZXMgdGhlIHRydXRoIGV4YWN0bHksIGVsc2UgMC4gVGhlIGZpbmFsIHNjb3JlIGlzCnRoZSBtZWFuIG92ZXIgYWxsIHRlc3Qgb3V0cHV0cyBhY3Jvc3MgYWxsIHRhc2tzLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gLi5pby5ncmlkIGltcG9ydCBHcmlkLCBncmlkc19lcXVhbApmcm9tIC4uaW8uc3VibWlzc2lvbiBpbXBvcnQgQXR0ZW1wdCwgUHJlZGljdGlvbnMKCgpkZWYgc2NvcmVfb3V0cHV0KGF0dGVtcHQ6IEF0dGVtcHQsIHRydXRoOiBHcmlkKSAtPiBpbnQ6CiAgICAiIiIxIGlmIGVpdGhlciBndWVzcyBlcXVhbHMgdGhlIGdyb3VuZCB0cnV0aCBleGFjdGx5LCBlbHNlIDAuIiIiCiAgICByZXR1cm4gaW50KGdyaWRzX2VxdWFsKGF0dGVtcHQuYXR0ZW1wdF8xLCB0cnV0aCkgb3IgZ3JpZHNfZXF1YWwoYXR0ZW1wdC5hdHRlbXB0XzIsIHRydXRoKSkKCgpkZWYgc2NvcmVfcHJlZGljdGlvbnMoCiAgICBwcmVkaWN0aW9uczogUHJlZGljdGlvbnMsCiAgICBzb2x1dGlvbnM6IGRpY3Rbc3RyLCBsaXN0W0dyaWRdXSwKKSAtPiBkaWN0OgogICAgIiIiU2NvcmUgcHJlZGljdGlvbnMgYWdhaW5zdCBncm91bmQtdHJ1dGggc29sdXRpb25zLgoKICAgIFJldHVybnMgYSBzdW1tYXJ5IGRpY3Qgd2l0aCB0aGUgb3ZlcmFsbCBtZWFuIHBsdXMgcmF3IGNvdW50cy4gT25seSB0YXNrCiAgICBvdXRwdXRzIHRoYXQgaGF2ZSBhIGdyb3VuZC10cnV0aCBlbnRyeSBhcmUgc2NvcmVkIChzbyB0aGlzIHdvcmtzIG9uIGFueQogICAgc3Vic2V0LCBlLmcuIGEgc21va2UtdGVzdCBzbGljZSBvZiB0aGUgZXZhbCBzcGxpdCkuCiAgICAiIiIKICAgIHRvdGFsID0gMAogICAgY29ycmVjdCA9IDAKICAgIHBlcl90YXNrOiBkaWN0W3N0ciwgZmxvYXRdID0ge30KICAgIGZvciB0YXNrX2lkLCB0cnV0aHMgaW4gc29sdXRpb25zLml0ZW1zKCk6CiAgICAgICAgYXR0ZW1wdHMgPSBwcmVkaWN0aW9ucy5nZXQodGFza19pZCkKICAgICAgICBpZiBhdHRlbXB0cyBpcyBOb25lOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHRhc2tfY29ycmVjdCA9IDAKICAgICAgICBuID0gbWluKGxlbihhdHRlbXB0cyksIGxlbih0cnV0aHMpKQogICAgICAgIGZvciBpIGluIHJhbmdlKG4pOgogICAgICAgICAgICBzID0gc2NvcmVfb3V0cHV0KGF0dGVtcHRzW2ldLCB0cnV0aHNbaV0pCiAgICAgICAgICAgIHRhc2tfY29ycmVjdCArPSBzCiAgICAgICAgICAgIGNvcnJlY3QgKz0gcwogICAgICAgICAgICB0b3RhbCArPSAxCiAgICAgICAgcGVyX3Rhc2tbdGFza19pZF0gPSB0YXNrX2NvcnJlY3QgLyBuIGlmIG4gZWxzZSAwLjAKCiAgICByZXR1cm4gewogICAgICAgICJzY29yZSI6IGNvcnJlY3QgLyB0b3RhbCBpZiB0b3RhbCBlbHNlIDAuMCwKICAgICAgICAiY29ycmVjdCI6IGNvcnJlY3QsCiAgICAgICAgInRvdGFsIjogdG90YWwsCiAgICAgICAgIm51bV90YXNrcyI6IGxlbihwZXJfdGFzayksCiAgICAgICAgInBlcl90YXNrIjogcGVyX3Rhc2ssCiAgICB9Cg==",
"src/arc/io/__init__.py": "",
"src/arc/io/grid.py": "IiIiVGhlIGNhbm9uaWNhbCBncmlkIHR5cGUgYW5kIGNvbnZlcnNpb25zLgoKQSBncmlkIGlzIGFuIGltbXV0YWJsZSwgaGFzaGFibGUgYHR1cGxlW3R1cGxlW2ludCwgLi4uXSwgLi4uXWAgb2Ygc3ltYm9scyAwLTkuCkltbXV0YWJpbGl0eSBtYXRjaGVzIHRoZSBwcm9qZWN0J3Mgbm8tbXV0YXRpb24gcnVsZTsgaGFzaGFiaWxpdHkgaXMgcmVxdWlyZWQgYnkKdGhlIGNhbmRpZGF0ZS12b3Rpbmcgc2VsZWN0aW9uIHN0YWdlIChncmlkcyBhcmUgdXNlZCBhcyBkaWN0IGtleXMpLgoKU29sdmVycyBtYXkgY29tcHV0ZSBpbiBudW1weSBmb3IgY29udmVuaWVuY2UgYW5kIGNvbnZlcnQgYmFjayB3aXRoIGBmcm9tX251bXB5YC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgU2VxdWVuY2UKCmltcG9ydCBudW1weSBhcyBucAoKR3JpZCA9IHR1cGxlW3R1cGxlW2ludCwgLi4uXSwgLi4uXQoKTlVNX0NPTE9SUyA9IDEwICAgICAgICMgc3ltYm9scyAwLTkKTUFYX0RJTSA9IDMwICAgICAgICAgICMgY29tcGV0aXRpb24gZ3JpZHMgYXJlIGF0IG1vc3QgMzB4MzAKTUlOX0RJTSA9IDEKCgpkZWYgZnJvbV9saXN0cyhyb3dzOiBTZXF1ZW5jZVtTZXF1ZW5jZVtpbnRdXSkgLT4gR3JpZDoKICAgICIiIkNvbnZlcnQgYSBKU09OIGxpc3Qtb2YtbGlzdHMgaW50byB0aGUgY2Fub25pY2FsIGltbXV0YWJsZSBncmlkLiIiIgogICAgcmV0dXJuIHR1cGxlKHR1cGxlKGludChjKSBmb3IgYyBpbiByb3cpIGZvciByb3cgaW4gcm93cykKCgpkZWYgdG9fbGlzdHMoZ3JpZDogR3JpZCkgLT4gbGlzdFtsaXN0W2ludF1dOgogICAgIiIiQ29udmVydCBhIGNhbm9uaWNhbCBncmlkIGJhY2sgdG8gSlNPTi1zZXJpYWxpc2FibGUgbGlzdC1vZi1saXN0cy4iIiIKICAgIHJldHVybiBbbGlzdChyb3cpIGZvciByb3cgaW4gZ3JpZF0KCgpkZWYgZnJvbV9udW1weShhcnI6IG5wLm5kYXJyYXkpIC0+IEdyaWQ6CiAgICAiIiJDb252ZXJ0IGEgMi1EIGludGVnZXIgbnVtcHkgYXJyYXkgaW50byBhIGNhbm9uaWNhbCBncmlkLiIiIgogICAgaWYgYXJyLm5kaW0gIT0gMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXhwZWN0ZWQgYSAyLUQgYXJyYXksIGdvdCBzaGFwZSB7YXJyLnNoYXBlfSIpCiAgICByZXR1cm4gdHVwbGUodHVwbGUoaW50KGMpIGZvciBjIGluIHJvdykgZm9yIHJvdyBpbiBhcnIudG9saXN0KCkpCgoKZGVmIHRvX251bXB5KGdyaWQ6IEdyaWQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJDb252ZXJ0IGEgY2Fub25pY2FsIGdyaWQgaW50byBhIDItRCBpbnQ4IG51bXB5IGFycmF5LiIiIgogICAgcmV0dXJuIG5wLmFycmF5KGdyaWQsIGR0eXBlPW5wLmludDgpCgoKZGVmIHNoYXBlKGdyaWQ6IEdyaWQpIC0+IHR1cGxlW2ludCwgaW50XToKICAgICIiIlJldHVybiAoaGVpZ2h0LCB3aWR0aCkuIEFzc3VtZXMgYSByZWN0YW5ndWxhciBncmlkLiIiIgogICAgaCA9IGxlbihncmlkKQogICAgdyA9IGxlbihncmlkWzBdKSBpZiBoIGVsc2UgMAogICAgcmV0dXJuIGgsIHcKCgpkZWYgaXNfdmFsaWRfZ3JpZChncmlkOiBHcmlkKSAtPiBib29sOgogICAgIiIiVHJ1ZSBpZmYgYGdyaWRgIGlzIHJlY3Rhbmd1bGFyLCB3aXRoaW4gc2l6ZSBsaW1pdHMsIGFuZCBhbGwgY2VsbHMgMC05LiIiIgogICAgaWYgbm90IGlzaW5zdGFuY2UoZ3JpZCwgdHVwbGUpIG9yIGxlbihncmlkKSA8IE1JTl9ESU0gb3IgbGVuKGdyaWQpID4gTUFYX0RJTToKICAgICAgICByZXR1cm4gRmFsc2UKICAgIHdpZHRoID0gTm9uZQogICAgZm9yIHJvdyBpbiBncmlkOgogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJvdywgdHVwbGUpOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICBpZiB3aWR0aCBpcyBOb25lOgogICAgICAgICAgICB3aWR0aCA9IGxlbihyb3cpCiAgICAgICAgICAgIGlmIHdpZHRoIDwgTUlOX0RJTSBvciB3aWR0aCA+IE1BWF9ESU06CiAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICBlbGlmIGxlbihyb3cpICE9IHdpZHRoOiAgIyByYWdnZWQg4oaSIGludmFsaWQKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgZm9yIGNlbGwgaW4gcm93OgogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShjZWxsLCBpbnQpIG9yIGNlbGwgPCAwIG9yIGNlbGwgPj0gTlVNX0NPTE9SUzoKICAgICAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgcmV0dXJuIFRydWUKCgpkZWYgZ3JpZHNfZXF1YWwoYTogR3JpZCwgYjogR3JpZCkgLT4gYm9vbDoKICAgICIiIkV4YWN0IGNlbGwtZm9yLWNlbGwgZXF1YWxpdHkgKHRoZSBjb21wZXRpdGlvbidzIGNvcnJlY3RuZXNzIGNyaXRlcmlvbikuIiIiCiAgICByZXR1cm4gYSA9PSBiCgoKZGVmIGNvbG9yX2NvdW50cyhncmlkOiBHcmlkKSAtPiBkaWN0W2ludCwgaW50XToKICAgICIiIkhpc3RvZ3JhbSBvZiBzeW1ib2wgLT4gY291bnQgYWNyb3NzIGFsbCBjZWxscy4iIiIKICAgIGNvdW50czogZGljdFtpbnQsIGludF0gPSB7fQogICAgZm9yIHJvdyBpbiBncmlkOgogICAgICAgIGZvciBjZWxsIGluIHJvdzoKICAgICAgICAgICAgY291bnRzW2NlbGxdID0gY291bnRzLmdldChjZWxsLCAwKSArIDEKICAgIHJldHVybiBjb3VudHMKCgpkZWYgYmFja2dyb3VuZF9jb2xvcihncmlkOiBHcmlkKSAtPiBpbnQ6CiAgICAiIiJIZXVyaXN0aWMgYmFja2dyb3VuZCA9IG1vc3QgZnJlcXVlbnQgc3ltYm9sICh0aWVzIOKGkiBsb3dlc3Qgc3ltYm9sKS4iIiIKICAgIGNvdW50cyA9IGNvbG9yX2NvdW50cyhncmlkKQogICAgaWYgbm90IGNvdW50czoKICAgICAgICByZXR1cm4gMAogICAgcmV0dXJuIG1heChzb3J0ZWQoY291bnRzKSwga2V5PWxhbWJkYSBjOiBjb3VudHNbY10pCg==",
"src/arc/io/loader.py": "IiIiTG9hZCBBUkMtQUdJLTIgY2hhbGxlbmdlL3NvbHV0aW9uIEpTT04gaW50byB0eXBlZCBvYmplY3RzLgoKSlNPTiBzaGFwZSAoY29uZmlybWVkIGZyb20gdGhlIGNvbXBldGl0aW9uIGRhdGEpOgogICAgY2hhbGxlbmdlczoge3Rhc2tfaWQ6IHsidHJhaW4iOiBbeyJpbnB1dCI6IGdyaWQsICJvdXRwdXQiOiBncmlkfSwgLi4uXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ0ZXN0IjogIFt7ImlucHV0IjogZ3JpZH0sIC4uLl19fQogICAgc29sdXRpb25zOiAge3Rhc2tfaWQ6IFtncmlkLCAuLi5dfSAgICMgb25lIG91dHB1dCBncmlkIHBlciB0ZXN0IGlucHV0LCBpbiBvcmRlcgoKVGVzdCBjaGFsbGVuZ2VzIGNhcnJ5IHRyYWluIHBhaXJzICsgdGVzdCBpbnB1dHMgb25seSAobm8gb3V0cHV0cykuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKZnJvbSAuZ3JpZCBpbXBvcnQgR3JpZCwgZnJvbV9saXN0cwoKX2xvZyA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQoKCmNsYXNzIE1hbGZvcm1lZFRhc2tFcnJvcihWYWx1ZUVycm9yKToKICAgICIiIkEgY2hhbGxlbmdlcyBmaWxlIGRvZXMgbm90IG1hdGNoIHRoZSBleHBlY3RlZCBzY2hlbWEuCgogICAgUmFpc2VkIHdpdGggdGhlIG9mZmVuZGluZyBgYHRhc2tfaWRgYCBhbmQgYSBjb25jcmV0ZSByZWFzb24gc28gYSBiYWQgcmVydW4KICAgIGZpbGUgc3VyZmFjZXMgYXMgYW4gYWN0aW9uYWJsZSBtZXNzYWdlIGluc3RlYWQgb2YgYSBiYXJlIGBgS2V5RXJyb3JgYCBkZWVwIGluCiAgICBwYXJzaW5nLgogICAgIiIiCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgUGFpcjoKICAgICIiIkEgc2luZ2xlIGRlbW9uc3RyYXRpb24gb3IgdGVzdCBwYWlyLiBgb3V0cHV0YCBpcyBOb25lIGZvciB0ZXN0IGlucHV0cy4iIiIKCiAgICBpbnB1dDogR3JpZAogICAgb3V0cHV0OiBHcmlkIHwgTm9uZSA9IE5vbmUKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBUYXNrOgogICAgIiIiT25lIEFSQyB0YXNrOiBkZW1vbnN0cmF0aW9uIHBhaXJzIHBsdXMgdGVzdCBpbnB1dChzKSB0byBzb2x2ZS4iIiIKCiAgICB0YXNrX2lkOiBzdHIKICAgIHRyYWluOiB0dXBsZVtQYWlyLCAuLi5dCiAgICB0ZXN0OiB0dXBsZVtQYWlyLCAuLi5dCgogICAgQHByb3BlcnR5CiAgICBkZWYgbnVtX3Rlc3Qoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi50ZXN0KQoKCmRlZiBfcGFyc2VfZ3JpZCh2YWx1ZSwgd2hlcmU6IHN0cikgLT4gR3JpZDoKICAgIGlmIG5vdCBpc2luc3RhbmNlKHZhbHVlLCBsaXN0KSBvciBub3QgdmFsdWU6CiAgICAgICAgcmFpc2UgTWFsZm9ybWVkVGFza0Vycm9yKGYie3doZXJlfTogZXhwZWN0ZWQgYSBub24tZW1wdHkgZ3JpZCAobGlzdCBvZiByb3dzKSIpCiAgICB0cnk6CiAgICAgICAgcmV0dXJuIGZyb21fbGlzdHModmFsdWUpCiAgICBleGNlcHQgKFR5cGVFcnJvciwgVmFsdWVFcnJvcikgYXMgZXhjOgogICAgICAgIHJhaXNlIE1hbGZvcm1lZFRhc2tFcnJvcihmInt3aGVyZX06IGludmFsaWQgZ3JpZCAoe2V4Y30pIikgZnJvbSBleGMKCgpkZWYgX3BhcnNlX3BhaXIocmF3OiBkaWN0LCB3aGVyZTogc3RyKSAtPiBQYWlyOgogICAgaWYgbm90IGlzaW5zdGFuY2UocmF3LCBkaWN0KSBvciAiaW5wdXQiIG5vdCBpbiByYXc6CiAgICAgICAgcmFpc2UgTWFsZm9ybWVkVGFza0Vycm9yKGYie3doZXJlfTogbWlzc2luZyAnaW5wdXQnIikKICAgIG91dCA9IHJhdy5nZXQoIm91dHB1dCIpCiAgICByZXR1cm4gUGFpcigKICAgICAgICBpbnB1dD1fcGFyc2VfZ3JpZChyYXdbImlucHV0Il0sIGYie3doZXJlfS5pbnB1dCIpLAogICAgICAgIG91dHB1dD1fcGFyc2VfZ3JpZChvdXQsIGYie3doZXJlfS5vdXRwdXQiKSBpZiBvdXQgaXMgbm90IE5vbmUgZWxzZSBOb25lLAogICAgKQoKCmRlZiBfcGFyc2VfdGFzayh0YXNrX2lkOiBzdHIsIGJvZHkpIC0+IFRhc2s6CiAgICBpZiBub3QgaXNpbnN0YW5jZShib2R5LCBkaWN0KSBvciAidHJhaW4iIG5vdCBpbiBib2R5IG9yICJ0ZXN0IiBub3QgaW4gYm9keToKICAgICAgICByYWlzZSBNYWxmb3JtZWRUYXNrRXJyb3IoZiJ0YXNrIHt0YXNrX2lkfTogbWlzc2luZyAndHJhaW4nLyd0ZXN0JyIpCiAgICB0cmFpbiwgdGVzdCA9IGJvZHlbInRyYWluIl0sIGJvZHlbInRlc3QiXQogICAgaWYgbm90IGlzaW5zdGFuY2UodHJhaW4sIGxpc3QpIG9yIG5vdCBpc2luc3RhbmNlKHRlc3QsIGxpc3QpOgogICAgICAgIHJhaXNlIE1hbGZvcm1lZFRhc2tFcnJvcihmInRhc2sge3Rhc2tfaWR9OiAndHJhaW4nLyd0ZXN0JyBtdXN0IGJlIGxpc3RzIikKICAgIGlmIG5vdCB0ZXN0OgogICAgICAgIHJhaXNlIE1hbGZvcm1lZFRhc2tFcnJvcihmInRhc2sge3Rhc2tfaWR9OiAndGVzdCcgbXVzdCBiZSBub24tZW1wdHkiKQogICAgcmV0dXJuIFRhc2soCiAgICAgICAgdGFza19pZD10YXNrX2lkLAogICAgICAgIHRyYWluPXR1cGxlKAogICAgICAgICAgICBfcGFyc2VfcGFpcihwLCBmInRhc2sge3Rhc2tfaWR9IHRyYWluW3tpfV0iKSBmb3IgaSwgcCBpbiBlbnVtZXJhdGUodHJhaW4pCiAgICAgICAgKSwKICAgICAgICB0ZXN0PXR1cGxlKAogICAgICAgICAgICBfcGFyc2VfcGFpcihwLCBmInRhc2sge3Rhc2tfaWR9IHRlc3Rbe2l9XSIpIGZvciBpLCBwIGluIGVudW1lcmF0ZSh0ZXN0KQogICAgICAgICksCiAgICApCgoKZGVmIGxvYWRfY2hhbGxlbmdlcyhwYXRoOiBzdHIgfCBQYXRoLCAqLCBza2lwX2ludmFsaWQ6IGJvb2wgPSBGYWxzZSkgLT4gZGljdFtzdHIsIFRhc2tdOgogICAgIiIiTG9hZCBhICpfY2hhbGxlbmdlcy5qc29uIGZpbGUgaW50byB7dGFza19pZDogVGFza30uCgogICAgVmFsaWRhdGVzIHN0cnVjdHVyZSBhbmQgcmFpc2VzIGEgY2xlYXIgOmNsYXNzOmBNYWxmb3JtZWRUYXNrRXJyb3JgIChuYW1pbmcgdGhlCiAgICB0YXNrIGFuZCByZWFzb24pIG9uIGEgYmFkIGZpbGUsIGluc3RlYWQgb2YgYSBiYXJlIGBgS2V5RXJyb3JgYC4gV2l0aAogICAgYGBza2lwX2ludmFsaWQ9VHJ1ZWBgIG1hbGZvcm1lZCB0YXNrcyBhcmUgbG9nZ2VkIGFuZCBkcm9wcGVkIHJhdGhlciB0aGFuCiAgICBhYm9ydGluZyB0aGUgbG9hZCDigJQgY2FsbGVycyB0aGF0IHJlcXVpcmUgZXZlcnkgdGFza19pZCBwcmVzZW50IChhIHN1Ym1pc3Npb24pCiAgICBzaG91bGQga2VlcCB0aGUgZGVmYXVsdCBhbmQgcmVseSBvbiB0aGUgZW50cnlwb2ludCdzIGZhbGxiYWNrIHNhZmV0eSBuZXQuCiAgICAiIiIKICAgIHdpdGggb3BlbihwYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHJhdyA9IGpzb24ubG9hZChmKQogICAgaWYgbm90IGlzaW5zdGFuY2UocmF3LCBkaWN0KToKICAgICAgICByYWlzZSBNYWxmb3JtZWRUYXNrRXJyb3IoZiJ7cGF0aH06IHRvcC1sZXZlbCBKU09OIG11c3QgYmUgYW4gb2JqZWN0IG9mIHRhc2tzIikKICAgIHRhc2tzOiBkaWN0W3N0ciwgVGFza10gPSB7fQogICAgZHJvcHBlZDogbGlzdFtzdHJdID0gW10KICAgIGZvciB0YXNrX2lkLCBib2R5IGluIHJhdy5pdGVtcygpOgogICAgICAgIHRyeToKICAgICAgICAgICAgdGFza3NbdGFza19pZF0gPSBfcGFyc2VfdGFzayh0YXNrX2lkLCBib2R5KQogICAgICAgIGV4Y2VwdCBNYWxmb3JtZWRUYXNrRXJyb3IgYXMgZXhjOgogICAgICAgICAgICBpZiBub3Qgc2tpcF9pbnZhbGlkOgogICAgICAgICAgICAgICAgcmFpc2UKICAgICAgICAgICAgZHJvcHBlZC5hcHBlbmQoc3RyKGV4YykpCiAgICBpZiBkcm9wcGVkOgogICAgICAgIF9sb2cud2FybmluZygKICAgICAgICAgICAgInNraXBwZWQgJWQgbWFsZm9ybWVkIHRhc2socyk6ICVzIiwgbGVuKGRyb3BwZWQpLCAiOyAiLmpvaW4oZHJvcHBlZFs6NV0pCiAgICAgICAgKQogICAgcmV0dXJuIHRhc2tzCgoKZGVmIGxvYWRfc29sdXRpb25zKHBhdGg6IHN0ciB8IFBhdGgpIC0+IGRpY3Rbc3RyLCBsaXN0W0dyaWRdXToKICAgICIiIkxvYWQgYSAqX3NvbHV0aW9ucy5qc29uIGZpbGUgaW50byB7dGFza19pZDogW291dHB1dF9ncmlkLCAuLi5dfS4iIiIKICAgIHdpdGggb3BlbihwYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHJhdyA9IGpzb24ubG9hZChmKQogICAgcmV0dXJuIHsKICAgICAgICB0YXNrX2lkOiBbZnJvbV9saXN0cyhnKSBmb3IgZyBpbiBncmlkc10gZm9yIHRhc2tfaWQsIGdyaWRzIGluIHJhdy5pdGVtcygpCiAgICB9Cg==",
"src/arc/io/submission.py": "IiIiQnVpbGQsIHZhbGlkYXRlIGFuZCB3cml0ZSB0aGUgY29tcGV0aXRpb24gc3VibWlzc2lvbi5qc29uLgoKUmVxdWlyZWQgc2NoZW1hICh2YWxpZGF0ZWQgYWdhaW5zdCB0aGUgdGVzdCBjaGFsbGVuZ2VzKToKICAgIHt0YXNrX2lkOiBbeyJhdHRlbXB0XzEiOiBncmlkLCAiYXR0ZW1wdF8yIjogZ3JpZH0sIC4uLl19CiAgLSBvbmUgZGljdCBwZXIgdGVzdCBpbnB1dCwgaW4gdGhlIFNBTUUgb3JkZXIgYXMgdGhlIHRhc2sncyB0ZXN0IGlucHV0czsKICAtIEJPVEggYXR0ZW1wdF8xIGFuZCBhdHRlbXB0XzIgbXVzdCBiZSBwcmVzZW50IGZvciBldmVyeSB0ZXN0IG91dHB1dDsKICAtIEVWRVJZIHRhc2tfaWQgaW4gdGhlIGNoYWxsZW5nZXMgZmlsZSBtdXN0IGFwcGVhciBpbiB0aGUgc3VibWlzc2lvbi4KCldlIG1vZGVsIGEgc2luZ2xlIHRlc3Qgb3V0cHV0J3MgdHdvIGd1ZXNzZXMgYXMgYW4gYEF0dGVtcHQoYXR0ZW1wdF8xLCBhdHRlbXB0XzIpYC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIC5ncmlkIGltcG9ydCBHcmlkLCBpc192YWxpZF9ncmlkLCB0b19saXN0cwpmcm9tIC5sb2FkZXIgaW1wb3J0IFRhc2sKCiMgRmFsbGJhY2sgZ3JpZCB1c2VkIHdoZW5ldmVyIGEgc29sdmVyIHByb2R1Y2VkIG5vdGhpbmcg4oCUIGd1YXJhbnRlZXMgdGhlIHNjaGVtYQojIGlzIGFsd2F5cyBzYXRpc2ZpYWJsZS4gQSAxeDEgemVybyBncmlkIGlzIHRoZSBjaGVhcGVzdCB2YWxpZCBwbGFjZWhvbGRlci4KRkFMTEJBQ0tfR1JJRDogR3JpZCA9ICgoMCwpLCkKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBBdHRlbXB0OgogICAgIiIiVGhlIHR3byBndWVzc2VzIGZvciBvbmUgdGVzdCBvdXRwdXQuIiIiCgogICAgYXR0ZW1wdF8xOiBHcmlkCiAgICBhdHRlbXB0XzI6IEdyaWQKCgojIEEgZnVsbCBwcmVkaWN0aW9uIHNldDoge3Rhc2tfaWQ6IFtBdHRlbXB0IHBlciB0ZXN0IGlucHV0LCBpbiBvcmRlcl19ClByZWRpY3Rpb25zID0gZGljdFtzdHIsIGxpc3RbQXR0ZW1wdF1dCgoKZGVmIGJ1aWxkX3N1Ym1pc3Npb24ocHJlZGljdGlvbnM6IFByZWRpY3Rpb25zKSAtPiBkaWN0OgogICAgIiIiU2VyaWFsaXNlIHByZWRpY3Rpb25zIGludG8gdGhlIGNvbXBldGl0aW9uJ3MgSlNPTiBzdHJ1Y3R1cmUuIiIiCiAgICBzdWJtaXNzaW9uOiBkaWN0W3N0ciwgbGlzdFtkaWN0XV0gPSB7fQogICAgZm9yIHRhc2tfaWQsIGF0dGVtcHRzIGluIHByZWRpY3Rpb25zLml0ZW1zKCk6CiAgICAgICAgc3VibWlzc2lvblt0YXNrX2lkXSA9IFsKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImF0dGVtcHRfMSI6IHRvX2xpc3RzKGEuYXR0ZW1wdF8xKSwKICAgICAgICAgICAgICAgICJhdHRlbXB0XzIiOiB0b19saXN0cyhhLmF0dGVtcHRfMiksCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZm9yIGEgaW4gYXR0ZW1wdHMKICAgICAgICBdCiAgICByZXR1cm4gc3VibWlzc2lvbgoKCmRlZiBlbXB0eV9wcmVkaWN0aW9ucyh0YXNrczogZGljdFtzdHIsIFRhc2tdKSAtPiBQcmVkaWN0aW9uczoKICAgICIiIkEgY29tcGxldGUsIHNjaGVtYS12YWxpZCBmYWxsYmFjayBwcmVkaWN0aW9uIChhbGwgMXgxIHplcm9zKS4KCiAgICBUaGUgcGlwZWxpbmUgd3JpdGVzIHRoaXMgZmlyc3Qgc28gYSB2YWxpZCBzdWJtaXNzaW9uIGFsd2F5cyBleGlzdHMsIHRoZW4KICAgIG92ZXJ3cml0ZXMgZW50cmllcyBhcyByZWFsIGFuc3dlcnMgYXJyaXZlICh0aW1lLXdhdGNoZG9nIHNhZmV0eSBuZXQpLgogICAgIiIiCiAgICByZXR1cm4gewogICAgICAgIHRhc2tfaWQ6IFtBdHRlbXB0KEZBTExCQUNLX0dSSUQsIEZBTExCQUNLX0dSSUQpIGZvciBfIGluIHRhc2sudGVzdF0KICAgICAgICBmb3IgdGFza19pZCwgdGFzayBpbiB0YXNrcy5pdGVtcygpCiAgICB9CgoKZGVmIGZhbGxiYWNrX2Zyb21fcmF3KHJhdzogb2JqZWN0KSAtPiBQcmVkaWN0aW9uczoKICAgICIiIkJlc3QtZWZmb3J0IHNjaGVtYS12YWxpZCBmYWxsYmFjayBmcm9tICpyYXcqIChwb3NzaWJseSBtYWxmb3JtZWQpIGNoYWxsZW5nZQogICAgSlNPTjogb25lIDF4MS16ZXJvIEF0dGVtcHQgcGVyIHRlc3QgaW5wdXQsIGRlZmF1bHRpbmcgdG8gYSBzaW5nbGUgb3V0cHV0IHdoZW4KICAgIGEgdGFzaydzIHRlc3QgY291bnQgY2Fubm90IGJlIGRldGVybWluZWQuCgogICAgVXNlZCBieSB0aGUgS2FnZ2xlIGVudHJ5cG9pbnQgdG8gZ3VhcmFudGVlIGEgc2NvcmVhYmxlIHN1Ym1pc3Npb24gZXhpc3RzIG9uCiAgICBkaXNrICpiZWZvcmUqIGFueSBwYXJzaW5nL21vZGVsIHdvcmssIHNvIGEgY3Jhc2ggb3IgT09NIGFueXdoZXJlIGRvd25zdHJlYW0KICAgIGNhbm5vdCBsZWF2ZSBhbiBlbXB0eSBgL2thZ2dsZS93b3JraW5nYC4KICAgICIiIgogICAgcHJlZHM6IFByZWRpY3Rpb25zID0ge30KICAgIGlmIG5vdCBpc2luc3RhbmNlKHJhdywgZGljdCk6CiAgICAgICAgcmV0dXJuIHByZWRzCiAgICBmb3IgdGFza19pZCwgYm9keSBpbiByYXcuaXRlbXMoKToKICAgICAgICBuID0gMQogICAgICAgIGlmIGlzaW5zdGFuY2UoYm9keSwgZGljdCk6CiAgICAgICAgICAgIHRlc3QgPSBib2R5LmdldCgidGVzdCIpCiAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodGVzdCwgbGlzdCkgYW5kIHRlc3Q6CiAgICAgICAgICAgICAgICBuID0gbGVuKHRlc3QpCiAgICAgICAgcHJlZHNbc3RyKHRhc2tfaWQpXSA9IFtBdHRlbXB0KEZBTExCQUNLX0dSSUQsIEZBTExCQUNLX0dSSUQpIGZvciBfIGluIHJhbmdlKG4pXQogICAgcmV0dXJuIHByZWRzCgoKZGVmIHZhbGlkYXRlX3N1Ym1pc3Npb24oc3VibWlzc2lvbjogZGljdCwgdGFza3M6IGRpY3Rbc3RyLCBUYXNrXSkgLT4gbGlzdFtzdHJdOgogICAgIiIiUmV0dXJuIGEgbGlzdCBvZiBzY2hlbWEgcHJvYmxlbXM7IGVtcHR5IGxpc3QgbWVhbnMgdGhlIHN1Ym1pc3Npb24gaXMgdmFsaWQuIiIiCiAgICBwcm9ibGVtczogbGlzdFtzdHJdID0gW10KCiAgICBtaXNzaW5nID0gc2V0KHRhc2tzKSAtIHNldChzdWJtaXNzaW9uKQogICAgaWYgbWlzc2luZzoKICAgICAgICBwcm9ibGVtcy5hcHBlbmQoZiJtaXNzaW5nIHtsZW4obWlzc2luZyl9IHRhc2tfaWRzLCBlLmcuIHtzb3J0ZWQobWlzc2luZylbOjNdfSIpCiAgICBleHRyYSA9IHNldChzdWJtaXNzaW9uKSAtIHNldCh0YXNrcykKICAgIGlmIGV4dHJhOgogICAgICAgIHByb2JsZW1zLmFwcGVuZChmInVuZXhwZWN0ZWQge2xlbihleHRyYSl9IHRhc2tfaWRzLCBlLmcuIHtzb3J0ZWQoZXh0cmEpWzozXX0iKQoKICAgIGZvciB0YXNrX2lkLCB0YXNrIGluIHRhc2tzLml0ZW1zKCk6CiAgICAgICAgZW50cnkgPSBzdWJtaXNzaW9uLmdldCh0YXNrX2lkKQogICAgICAgIGlmIGVudHJ5IGlzIE5vbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UoZW50cnksIGxpc3QpIG9yIGxlbihlbnRyeSkgIT0gdGFzay5udW1fdGVzdDoKICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKAogICAgICAgICAgICAgICAgZiJ7dGFza19pZH06IGV4cGVjdGVkIHt0YXNrLm51bV90ZXN0fSBvdXRwdXRzLCBnb3QgIgogICAgICAgICAgICAgICAgZiJ7bGVuKGVudHJ5KSBpZiBpc2luc3RhbmNlKGVudHJ5LCBsaXN0KSBlbHNlIHR5cGUoZW50cnkpLl9fbmFtZV9ffSIKICAgICAgICAgICAgKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZvciBpLCBvdXQgaW4gZW51bWVyYXRlKGVudHJ5KToKICAgICAgICAgICAgZm9yIGtleSBpbiAoImF0dGVtcHRfMSIsICJhdHRlbXB0XzIiKToKICAgICAgICAgICAgICAgIGlmIGtleSBub3QgaW4gb3V0OgogICAgICAgICAgICAgICAgICAgIHByb2JsZW1zLmFwcGVuZChmInt0YXNrX2lkfVt7aX1dOiBtaXNzaW5nIHtrZXl9IikKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgZ3JpZCA9IF9jb2VyY2VfZ3JpZChvdXRba2V5XSkKICAgICAgICAgICAgICAgIGlmIGdyaWQgaXMgTm9uZSBvciBub3QgaXNfdmFsaWRfZ3JpZChncmlkKToKICAgICAgICAgICAgICAgICAgICBwcm9ibGVtcy5hcHBlbmQoZiJ7dGFza19pZH1be2l9XS57a2V5fTogaW52YWxpZCBncmlkIikKICAgIHJldHVybiBwcm9ibGVtcwoKCmRlZiBfY29lcmNlX2dyaWQodmFsdWUpIC0+IEdyaWQgfCBOb25lOgogICAgIiIiQmVzdC1lZmZvcnQgY29udmVydCBhIEpTT04gbGlzdC1vZi1saXN0cyBpbnRvIGEgR3JpZCBmb3IgdmFsaWRhdGlvbi4iIiIKICAgIGlmIG5vdCBpc2luc3RhbmNlKHZhbHVlLCBsaXN0KSBvciBub3QgdmFsdWU6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHRyeToKICAgICAgICByZXR1cm4gdHVwbGUodHVwbGUoaW50KGMpIGZvciBjIGluIHJvdykgZm9yIHJvdyBpbiB2YWx1ZSkKICAgIGV4Y2VwdCAoVHlwZUVycm9yLCBWYWx1ZUVycm9yKToKICAgICAgICByZXR1cm4gTm9uZQoKCmRlZiB3cml0ZV9zdWJtaXNzaW9uKHN1Ym1pc3Npb246IGRpY3QsIHBhdGg6IHN0ciB8IFBhdGgpIC0+IFBhdGg6CiAgICAiIiJXcml0ZSBzdWJtaXNzaW9uIEpTT04gdG8gYHBhdGhgLCByZXR1cm5pbmcgdGhlIHBhdGguIiIiCiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCBvcGVuKHBhdGgsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAoc3VibWlzc2lvbiwgZikKICAgIHJldHVybiBwYXRoCg==",
"src/arc/pipeline.py": "IiIiRW5zZW1ibGUgb3JjaGVzdHJhdGlvbjogcnVuIHNvbHZlcnMsIHZvdGUgY2FuZGlkYXRlcyBpbnRvIHR3byBhdHRlbXB0cyBwZXIKdGVzdCBvdXRwdXQsIGFuZCBndWFyYW50ZWUgYSBjb21wbGV0ZSB2YWxpZCBzdWJtaXNzaW9uIHVuZGVyIGEgaGFyZCB0aW1lIGJ1ZGdldC4KCkRlc2lnbiBub3RlczoKICAqIEEgY29tcGxldGUgRkFMTEJBQ0sgc3VibWlzc2lvbiBpcyB3cml0dGVuIGJlZm9yZSBhbnkgc29sdmluZyBiZWdpbnMsIHRoZW4KICAgIG92ZXJ3cml0dGVuIGFzIHJlYWwgYW5zd2VycyBhcnJpdmUgYW5kIHJlLWNoZWNrcG9pbnRlZCBwZXJpb2RpY2FsbHkuIElmIHRoZQogICAgcHJvY2VzcyBpcyBraWxsZWQgbWlkLXJ1biwgYSB2YWxpZCBmaWxlIGlzIGFscmVhZHkgb24gZGlzay4KICAqIENhbmRpZGF0ZXMgZnJvbSBhbGwgc29sdmVycyBhcmUgbWVyZ2VkIGJ5IHdlaWdodGVkIHZvdGluZzsgdGhlIHNhbWUgbWVyZ2UKICAgIHdpbGwgYWJzb3JiIHRoZSBMTE0tVFRUIHNvbHZlcidzIGNhbmRpZGF0ZXMgaW4gbGF0ZXIgbWlsZXN0b25lcy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbG9nZ2luZwppbXBvcnQgdGltZQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZWZhdWx0ZGljdAoKZnJvbSAuY29uZmlnIGltcG9ydCBERUZBVUxUX1BFUl9UQVNLX0JVREdFVF9TLCBUT1RBTF9SVU5USU1FX0JVREdFVF9TCmZyb20gLmlvLmdyaWQgaW1wb3J0IEdyaWQKZnJvbSAuaW8ubG9hZGVyIGltcG9ydCBUYXNrCmZyb20gLmlvLnN1Ym1pc3Npb24gaW1wb3J0ICgKICAgIEZBTExCQUNLX0dSSUQsCiAgICBBdHRlbXB0LAogICAgUHJlZGljdGlvbnMsCiAgICBidWlsZF9zdWJtaXNzaW9uLAogICAgZW1wdHlfcHJlZGljdGlvbnMsCiAgICB3cml0ZV9zdWJtaXNzaW9uLAopCmZyb20gLnNvbHZlcnMuYmFzZSBpbXBvcnQgQ2FuZGlkYXRlcywgU29sdmVyCmZyb20gLnNvbHZlcnMuZHNsLnNvbHZlciBpbXBvcnQgRFNMU29sdmVyCmZyb20gLnNvbHZlcnMuaWRlbnRpdHkgaW1wb3J0IENIRUFQX1NPTFZFUlMKCiMgQmFzZSB2b3RlIHdlaWdodCBwZXIgc29sdmVyICh2ZXJpZmllZCBzb2x2ZXJzIGRvbWluYXRlIGhldXJpc3RpYyBwcmlvcnMpLgpTT0xWRVJfV0VJR0hUUzogZGljdFtzdHIsIGZsb2F0XSA9IHsKICAgICJkc2wiOiAxMC4wLAogICAgImlkZW50aXR5IjogOC4wLAogICAgImNvbnN0YW50X291dHB1dCI6IDYuMCwKICAgICJsbG1fdHR0IjogOS4wLCAgICMgYXJyaXZlcyBpbiBNMgogICAgImxsbSI6IDQuMCwgICAgICAgICMgYXJyaXZlcyBpbiBNMQogICAgIm1ham9yaXR5X3NoYXBlIjogMC41LAp9Cl9ERUZBVUxUX1dFSUdIVCA9IDEuMApfQ0hFQ0tQT0lOVF9FVkVSWSA9IDI1ICAjIHJld3JpdGUgc3VibWlzc2lvbiBhdCBsZWFzdCBldmVyeSBOIHRhc2tzCl9DSEVDS1BPSU5UX1NFQ09ORFMgPSAzMDAuMCAgIyAuLi5hbmQgYXQgbGVhc3QgZXZlcnkgTiBzZWNvbmRzIG9mIHdhbGwtY2xvY2sKCl9sb2cgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykKCgpkZWYgZGVmYXVsdF9zb2x2ZXJzKGxsbV9tb2RlbD1Ob25lLCBsbG1fa3dhcmdzOiBkaWN0IHwgTm9uZSA9IE5vbmUpIC0+IGxpc3RbU29sdmVyXToKICAgICIiIlRoZSBDUFUgZW5zZW1ibGUgKERTTCArIGNoZWFwIGhldXJpc3RpY3MpLiBJZiBgbGxtX21vZGVsYCBpcyBzdXBwbGllZCwgYW4KICAgIGBMTE1Tb2x2ZXJgIGlzIGFwcGVuZGVkIOKAlCB3b3JrcyB3aXRoIHRoZSByZWFsIEhGTW9kZWwgb24gS2FnZ2xlIG9yIGEgTW9ja01vZGVsCiAgICBsb2NhbGx5LCBzbyB0aGUgZnVsbCBlbnNlbWJsZSBpcyB0ZXN0YWJsZSB3aXRob3V0IGEgR1BVLiIiIgogICAgc29sdmVyczogbGlzdFtTb2x2ZXJdID0gW0RTTFNvbHZlcigpLCAqQ0hFQVBfU09MVkVSU10KICAgIGlmIGxsbV9tb2RlbCBpcyBub3QgTm9uZToKICAgICAgICBmcm9tIC5zb2x2ZXJzLmxsbS5zb2x2ZXIgaW1wb3J0IExMTVNvbHZlcgoKICAgICAgICBzb2x2ZXJzLmFwcGVuZChMTE1Tb2x2ZXIobGxtX21vZGVsLCAqKihsbG1fa3dhcmdzIG9yIHt9KSkpCiAgICByZXR1cm4gc29sdmVycwoKCmRlZiBzb2x2ZV90YXNrKHRhc2s6IFRhc2ssIHNvbHZlcnM6IGxpc3RbU29sdmVyXSwgYnVkZ2V0X3M6IGZsb2F0KSAtPiBsaXN0W0F0dGVtcHRdOgogICAgIiIiUnVuIGFsbCBzb2x2ZXJzIG9uIGEgdGFzayBhbmQgdm90ZSBjYW5kaWRhdGVzIGludG8gdHdvIGF0dGVtcHRzIHBlciB0ZXN0CiAgICBvdXRwdXQuIFNvbHZlcnMgc2hhcmUgb25lIHBlci10YXNrIGJ1ZGdldDogZWFjaCByZWNlaXZlcyB0aGUgdGltZSByZW1haW5pbmcsCiAgICBzbyBjaGVhcCBzb2x2ZXJzIChsaXN0ZWQgZmlyc3QpIHJ1biBmcmVlIGFuZCB0aGUgZXhwZW5zaXZlIExMTS9UVFQgZ2V0cyB0aGUKICAgIHJlc3QuIiIiCiAgICBkZWFkbGluZSA9IHRpbWUubW9ub3RvbmljKCkgKyBidWRnZXRfcwogICAgdm90ZXM6IGxpc3RbZGVmYXVsdGRpY3RbR3JpZCwgZmxvYXRdXSA9IFtkZWZhdWx0ZGljdChmbG9hdCkgZm9yIF8gaW4gdGFzay50ZXN0XQogICAgZm9yIHNvbHZlciBpbiBzb2x2ZXJzOgogICAgICAgIHJlbWFpbmluZyA9IG1heCgwLjAsIGRlYWRsaW5lIC0gdGltZS5tb25vdG9uaWMoKSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIGNhbmRpZGF0ZXM6IENhbmRpZGF0ZXMgPSBzb2x2ZXIuc29sdmUodGFzaywgcmVtYWluaW5nKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICMgTmV2ZXIgbGV0IG9uZSBzb2x2ZXIgc2luayB0aGUgdGFzazogbG9nIGZvciBkaWFnbm9zaXMgKHNpbGVudAogICAgICAgICAgICAjIGRlZ3JhZGF0aW9uIHRvIHRoZSByZW1haW5pbmcgc29sdmVycyBpcyBvdGhlcndpc2UgaW52aXNpYmxlIGluIHRoZQogICAgICAgICAgICAjIEthZ2dsZSBydW4gbG9nKSBhbmQgZmFsbCB0aHJvdWdoIHRvIHdoYXRldmVyIGVsc2Ugdm90ZWQuCiAgICAgICAgICAgIF9sb2cuZXhjZXB0aW9uKCJzb2x2ZXIgJXMgZmFpbGVkIG9uIHRhc2sgJXMiLCBzb2x2ZXIubmFtZSwgdGFzay50YXNrX2lkKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHdlaWdodCA9IFNPTFZFUl9XRUlHSFRTLmdldChzb2x2ZXIubmFtZSwgX0RFRkFVTFRfV0VJR0hUKQogICAgICAgIGZvciBpLCByYW5rZWQgaW4gZW51bWVyYXRlKGNhbmRpZGF0ZXMpOgogICAgICAgICAgICBpZiBpID49IGxlbih2b3Rlcyk6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBmb3IgcmFuaywgZ3JpZCBpbiBlbnVtZXJhdGUocmFua2VkKToKICAgICAgICAgICAgICAgIHZvdGVzW2ldW2dyaWRdICs9IHdlaWdodCAvIChyYW5rICsgMSkKCiAgICByZXR1cm4gW190b3AyKHYpIGZvciB2IGluIHZvdGVzXQoKCmRlZiBfdG9wMih2b3RlOiBkZWZhdWx0ZGljdFtHcmlkLCBmbG9hdF0pIC0+IEF0dGVtcHQ6CiAgICAiIiJQaWNrIHRoZSB0d28gaGlnaGVzdC13ZWlnaHRlZCBkaXN0aW5jdCBncmlkcyAod2l0aCBzYWZlIGZhbGxiYWNrcykuIiIiCiAgICByYW5rZWQgPSBzb3J0ZWQodm90ZS5pdGVtcygpLCBrZXk9bGFtYmRhIGt2OiAoLWt2WzFdLCBfZ3JpZF9zaXplKGt2WzBdKSkpCiAgICBhMSA9IHJhbmtlZFswXVswXSBpZiByYW5rZWQgZWxzZSBGQUxMQkFDS19HUklECiAgICBhMiA9IHJhbmtlZFsxXVswXSBpZiBsZW4ocmFua2VkKSA+IDEgZWxzZSBhMQogICAgcmV0dXJuIEF0dGVtcHQoYTEsIGEyKQoKCmRlZiBfZ3JpZF9zaXplKGdyaWQ6IEdyaWQpIC0+IGludDoKICAgIHJldHVybiBsZW4oZ3JpZCkgKiAobGVuKGdyaWRbMF0pIGlmIGdyaWQgZWxzZSAwKQoKCmRlZiBydW4oCiAgICB0YXNrczogZGljdFtzdHIsIFRhc2tdLAogICAgc29sdmVyczogbGlzdFtTb2x2ZXJdIHwgTm9uZSA9IE5vbmUsCiAgICBvdXRwdXRfcGF0aD1Ob25lLAogICAgdG90YWxfYnVkZ2V0X3M6IGZsb2F0ID0gVE9UQUxfUlVOVElNRV9CVURHRVRfUywKICAgIHBlcl90YXNrX2J1ZGdldF9zOiBmbG9hdCA9IERFRkFVTFRfUEVSX1RBU0tfQlVER0VUX1MsCiAgICB2ZXJib3NlOiBib29sID0gRmFsc2UsCikgLT4gUHJlZGljdGlvbnM6CiAgICAiIiJTb2x2ZSBhbGwgdGFza3MgdW5kZXIgYSBnbG9iYWwgdGltZSBidWRnZXQsIGNoZWNrcG9pbnRpbmcgdGhlIHN1Ym1pc3Npb24uIiIiCiAgICBzb2x2ZXJzID0gc29sdmVycyBpZiBzb2x2ZXJzIGlzIG5vdCBOb25lIGVsc2UgZGVmYXVsdF9zb2x2ZXJzKCkKICAgIGZvciBzb2x2ZXIgaW4gc29sdmVyczoKICAgICAgICBpZiBzb2x2ZXIubmFtZSBub3QgaW4gU09MVkVSX1dFSUdIVFM6CiAgICAgICAgICAgIF9sb2cud2FybmluZygKICAgICAgICAgICAgICAgICJzb2x2ZXIgJXIgaGFzIG5vIHJlZ2lzdGVyZWQgdm90ZSB3ZWlnaHQ7IHVzaW5nIGRlZmF1bHQgJS4xZiIsCiAgICAgICAgICAgICAgICBzb2x2ZXIubmFtZSwKICAgICAgICAgICAgICAgIF9ERUZBVUxUX1dFSUdIVCwKICAgICAgICAgICAgKQogICAgcHJlZGljdGlvbnMgPSBlbXB0eV9wcmVkaWN0aW9ucyh0YXNrcykgICMgY29tcGxldGUgdmFsaWQgZmFsbGJhY2sgdXAgZnJvbnQKICAgIGlmIG91dHB1dF9wYXRoIGlzIG5vdCBOb25lOgogICAgICAgIHdyaXRlX3N1Ym1pc3Npb24oYnVpbGRfc3VibWlzc2lvbihwcmVkaWN0aW9ucyksIG91dHB1dF9wYXRoKQoKICAgIHN0YXJ0ID0gdGltZS5tb25vdG9uaWMoKQogICAgbGFzdF9jaGVja3BvaW50ID0gc3RhcnQKICAgIGZvciBuLCAodGFza19pZCwgdGFzaykgaW4gZW51bWVyYXRlKHRhc2tzLml0ZW1zKCksIDEpOgogICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgLSBzdGFydCA+IHRvdGFsX2J1ZGdldF9zOgogICAgICAgICAgICBpZiB2ZXJib3NlOgogICAgICAgICAgICAgICAgcHJpbnQoZiJbd2F0Y2hkb2ddIGJ1ZGdldCBleGhhdXN0ZWQgYWZ0ZXIge24gLSAxfSB0YXNrcyIpCiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgcHJlZGljdGlvbnNbdGFza19pZF0gPSBzb2x2ZV90YXNrKHRhc2ssIHNvbHZlcnMsIHBlcl90YXNrX2J1ZGdldF9zKQogICAgICAgIG5vdyA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICAjIENoZWNrcG9pbnQgb24gdGFzayBjb3VudCBPUiBlbGFwc2VkIHRpbWUg4oCUIGEgc2xvdyBydW4gKGZldyB0YXNrcywgbG9uZwogICAgICAgICMgZWFjaCkgc3RpbGwgZmx1c2hlcyByZWFsIGFuc3dlcnMgaW5zdGVhZCBvZiBsb3NpbmcgdXAgdG8gYW4gaG91ciB0byBhCiAgICAgICAgIyBjcmFzaCBiZXR3ZWVuIHRoZSBjb3VudC1iYXNlZCBjaGVja3BvaW50cy4KICAgICAgICBkdWUgPSBuICUgX0NIRUNLUE9JTlRfRVZFUlkgPT0gMCBvciBub3cgLSBsYXN0X2NoZWNrcG9pbnQgPj0gX0NIRUNLUE9JTlRfU0VDT05EUwogICAgICAgIGlmIG91dHB1dF9wYXRoIGlzIG5vdCBOb25lIGFuZCBkdWU6CiAgICAgICAgICAgIHdyaXRlX3N1Ym1pc3Npb24oYnVpbGRfc3VibWlzc2lvbihwcmVkaWN0aW9ucyksIG91dHB1dF9wYXRoKQogICAgICAgICAgICBsYXN0X2NoZWNrcG9pbnQgPSBub3cKICAgICAgICBpZiB2ZXJib3NlIGFuZCBuICUgX0NIRUNLUE9JTlRfRVZFUlkgPT0gMDoKICAgICAgICAgICAgcHJpbnQoZiJbcGlwZWxpbmVdIHNvbHZlZCB7bn0ve2xlbih0YXNrcyl9IHRhc2tzIikKCiAgICBpZiBvdXRwdXRfcGF0aCBpcyBub3QgTm9uZToKICAgICAgICB3cml0ZV9zdWJtaXNzaW9uKGJ1aWxkX3N1Ym1pc3Npb24ocHJlZGljdGlvbnMpLCBvdXRwdXRfcGF0aCkKICAgIHJldHVybiBwcmVkaWN0aW9ucwo=",
"src/arc/serialize/__init__.py": "",
"src/arc/serialize/prompt.py": "IiIiQnVpbGQgdHJhbnNkdWN0aW9uIHByb21wdHMgZnJvbSBhIHRhc2sncyBkZW1vbnN0cmF0aW9uIHBhaXJzICsgdGVzdCBpbnB1dC4KCkZvcm1hdCAoZmV3LXNob3QsIGNvbXBsZXRpb24gc3R5bGUpOgoKICAgIElucHV0OgogICAgPGdyaWQ+CiAgICBPdXRwdXQ6CiAgICA8Z3JpZD4KCiAgICBJbnB1dDoKICAgIDxncmlkPgogICAgT3V0cHV0OgogICAgPGdyaWQ+CgogICAgSW5wdXQ6CiAgICA8Z3JpZD4KICAgIE91dHB1dDoKClRoZSBtb2RlbCBjb21wbGV0ZXMgdGhlIGZpbmFsIGdyaWQuIGBwYXJzZV9jb21wbGV0aW9uYCBleHRyYWN0cyBpdC4gVGhpcyBtb2R1bGUKaXMgbW9kZWwtYWdub3N0aWMgdGV4dCBhc3NlbWJseTsgdG9rZW5pc2F0aW9uL2RlY29kaW5nIGxpdmUgaW4gc29sdmVycy9sbG0uCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IFNlcXVlbmNlCgpmcm9tIC4uaW8uZ3JpZCBpbXBvcnQgR3JpZApmcm9tIC4uaW8ubG9hZGVyIGltcG9ydCBQYWlyCmZyb20gLnRva2VuaXplciBpbXBvcnQgZ3JpZF90b19zdHIsIHN0cl90b19ncmlkCgpJTlBVVF9UQUcgPSAiSW5wdXQ6IgpPVVRQVVRfVEFHID0gIk91dHB1dDoiClBBSVJfU0VQID0gIlxuXG4iCgoKZGVmIF9mb3JtYXRfcGFpcihwYWlyOiBQYWlyLCBpbmNsdWRlX291dHB1dDogYm9vbCkgLT4gc3RyOgogICAgYmxvY2sgPSBmIntJTlBVVF9UQUd9XG57Z3JpZF90b19zdHIocGFpci5pbnB1dCl9XG57T1VUUFVUX1RBR30iCiAgICBpZiBpbmNsdWRlX291dHB1dCBhbmQgcGFpci5vdXRwdXQgaXMgbm90IE5vbmU6CiAgICAgICAgYmxvY2sgKz0gZiJcbntncmlkX3RvX3N0cihwYWlyLm91dHB1dCl9IgogICAgcmV0dXJuIGJsb2NrCgoKZGVmIGJ1aWxkX3Byb21wdCh0cmFpbjogU2VxdWVuY2VbUGFpcl0sIHRlc3RfaW5wdXQ6IEdyaWQpIC0+IHN0cjoKICAgICIiIkFzc2VtYmxlIGEgZmV3LXNob3QgcHJvbXB0IGVuZGluZyB3aXRoIGFuIG9wZW4gT3V0cHV0OiBmb3IgdGhlIHRlc3QgaW5wdXQuIiIiCiAgICBibG9ja3MgPSBbX2Zvcm1hdF9wYWlyKHAsIGluY2x1ZGVfb3V0cHV0PVRydWUpIGZvciBwIGluIHRyYWluXQogICAgYmxvY2tzLmFwcGVuZChfZm9ybWF0X3BhaXIoUGFpcihpbnB1dD10ZXN0X2lucHV0KSwgaW5jbHVkZV9vdXRwdXQ9RmFsc2UpKQogICAgcmV0dXJuIFBBSVJfU0VQLmpvaW4oYmxvY2tzKQoKCmRlZiBwYXJzZV9jb21wbGV0aW9uKGNvbXBsZXRpb246IHN0cikgLT4gR3JpZCB8IE5vbmU6CiAgICAiIiJFeHRyYWN0IHRoZSBwcmVkaWN0ZWQgZ3JpZCBmcm9tIHRoZSBtb2RlbCdzIGNvbXBsZXRpb24gdGV4dC4KCiAgICBUaGUgY29tcGxldGlvbiBpcyBldmVyeXRoaW5nIGdlbmVyYXRlZCBhZnRlciB0aGUgdHJhaWxpbmcgYE91dHB1dDpgOyBpZiBhbgogICAgYElucHV0OmAgbWFya2VyIGFwcGVhcnMgKG1vZGVsIHJhbiBvbiksIHdlIGN1dCBhdCBpdCBmaXJzdC4KICAgICIiIgogICAgdGV4dCA9IGNvbXBsZXRpb24KICAgIGN1dCA9IHRleHQuZmluZChJTlBVVF9UQUcpCiAgICBpZiBjdXQgIT0gLTE6CiAgICAgICAgdGV4dCA9IHRleHRbOmN1dF0KICAgIHJldHVybiBzdHJfdG9fZ3JpZCh0ZXh0KQoKCmRlZiBleHRyYWN0X2xhc3RfaW5wdXQocHJvbXB0OiBzdHIpIC0+IEdyaWQgfCBOb25lOgogICAgIiIiUmV0dXJuIHRoZSBncmlkIG9mIHRoZSBmaW5hbCBgSW5wdXQ6YCBibG9jayBpbiBhIHByb21wdCAodXNlZCBieSBtb2NrcyAvCiAgICBzYW5pdHkgY2hlY2tzKS4gVGV4dCBiZXR3ZWVuIHRoZSBsYXN0IElOUFVUX1RBRyBhbmQgdGhlIGZvbGxvd2luZyBPVVRQVVRfVEFHLgogICAgIiIiCiAgICBpZHggPSBwcm9tcHQucmZpbmQoSU5QVVRfVEFHKQogICAgaWYgaWR4ID09IC0xOgogICAgICAgIHJldHVybiBOb25lCiAgICB0YWlsID0gcHJvbXB0W2lkeCArIGxlbihJTlBVVF9UQUcpIDpdCiAgICBlbmQgPSB0YWlsLmZpbmQoT1VUUFVUX1RBRykKICAgIGlmIGVuZCAhPSAtMToKICAgICAgICB0YWlsID0gdGFpbFs6ZW5kXQogICAgcmV0dXJuIHN0cl90b19ncmlkKHRhaWwpCg==",
"src/arc/serialize/tokenizer.py": "IiIiR3JpZCA8LT4gdGV4dCBzZXJpYWxpc2F0aW9uIGZvciBMTE0gdHJhbnNkdWN0aW9uLgoKRGVmYXVsdCBzY2hlbWU6IG9uZSBncmlkIHJvdyBwZXIgbGluZSwgY2VsbHMgd3JpdHRlbiBhcyBjb25jYXRlbmF0ZWQgZGlnaXRzCihlYWNoIHN5bWJvbCBpcyBhIHNpbmdsZSBjaGFyYWN0ZXIgMC05KS4gQ29tcGFjdCBhbmQgdW5hbWJpZ3VvdXMsIGUuZy4KCiAgICAwMTIKICAgIDM0NQoKUGFyc2luZyBpcyBkZWxpYmVyYXRlbHkgbGVuaWVudCBhYm91dCBtb2RlbCBvdXRwdXQ6IHdlIGtlZXAgb25seSBkaWdpdApjaGFyYWN0ZXJzIHBlciBsaW5lIGFuZCBkcm9wIGJsYW5rL2dhcmJhZ2UgbGluZXMsIHNvIGEgc2xpZ2h0bHkgbWFsZm9ybWVkCmNvbXBsZXRpb24gc3RpbGwgeWllbGRzIGEgdXNhYmxlIGdyaWQgKG9yIE5vbmUgaWYgbm90aGluZyBwYXJzZXMpLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gLi5pby5ncmlkIGltcG9ydCBNQVhfRElNLCBHcmlkLCBpc192YWxpZF9ncmlkCgpST1dfU0VQID0gIlxuIgoKCmRlZiBncmlkX3RvX3N0cihncmlkOiBHcmlkKSAtPiBzdHI6CiAgICAiIiJTZXJpYWxpc2UgYSBncmlkIHRvIHRoZSBjYW5vbmljYWwgY29tcGFjdCB0ZXh0IGZvcm0uIiIiCiAgICByZXR1cm4gUk9XX1NFUC5qb2luKCIiLmpvaW4oc3RyKGMpIGZvciBjIGluIHJvdykgZm9yIHJvdyBpbiBncmlkKQoKCmRlZiBzdHJfdG9fZ3JpZCh0ZXh0OiBzdHIpIC0+IEdyaWQgfCBOb25lOgogICAgIiIiUGFyc2UgdGV4dCAocG9zc2libHkgbm9pc3kgbW9kZWwgb3V0cHV0KSBiYWNrIGludG8gYSBHcmlkLCBvciBOb25lLgoKICAgIExlbmllbnQ6IHBlciBsaW5lLCBrZWVwIG9ubHkgMC05IGNoYXJhY3RlcnM7IGlnbm9yZSBlbXB0eSBsaW5lczsgc3RvcCBhdCB0aGUKICAgIGZpcnN0IHJhZ2dlZCByb3cgdG8gYXZvaWQgc3dhbGxvd2luZyB0cmFpbGluZyBwcm9zZS4gUmV0dXJucyBOb25lIGlmIHRoZQogICAgcmVzdWx0IGlzIG5vdCBhIHZhbGlkIGdyaWQuCiAgICAiIiIKICAgIHJvd3M6IGxpc3RbdHVwbGVbaW50LCAuLi5dXSA9IFtdCiAgICB3aWR0aDogaW50IHwgTm9uZSA9IE5vbmUKICAgIGZvciByYXdfbGluZSBpbiB0ZXh0LnNwbGl0bGluZXMoKToKICAgICAgICBkaWdpdHMgPSBbaW50KGNoKSBmb3IgY2ggaW4gcmF3X2xpbmUgaWYgY2guaXNkaWdpdCgpXQogICAgICAgIGlmIG5vdCBkaWdpdHM6CiAgICAgICAgICAgICMgQWxsb3cgYmxhbmsgc2VwYXJhdG9yIGxpbmVzIGJlZm9yZSB0aGUgZ3JpZDsgb25jZSB0aGUgZ3JpZCBoYXMKICAgICAgICAgICAgIyBzdGFydGVkLCBhIGJsYW5rIGxpbmUgdGVybWluYXRlcyBpdC4KICAgICAgICAgICAgaWYgcm93czoKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgd2lkdGggaXMgTm9uZToKICAgICAgICAgICAgd2lkdGggPSBsZW4oZGlnaXRzKQogICAgICAgIGVsaWYgbGVuKGRpZ2l0cykgIT0gd2lkdGg6CiAgICAgICAgICAgIGJyZWFrICAjIHJhZ2dlZCDihpIgZW5kIG9mIGdyaWQKICAgICAgICByb3dzLmFwcGVuZCh0dXBsZShkaWdpdHMpKQogICAgICAgIGlmIGxlbihyb3dzKSA+IE1BWF9ESU06CiAgICAgICAgICAgIGJyZWFrCiAgICBpZiBub3Qgcm93czoKICAgICAgICByZXR1cm4gTm9uZQogICAgZ3JpZCA9IHR1cGxlKHJvd3MpCiAgICByZXR1cm4gZ3JpZCBpZiBpc192YWxpZF9ncmlkKGdyaWQpIGVsc2UgTm9uZQo=",
"src/arc/solvers/__init__.py": "",
"src/arc/solvers/base.py": "IiIiU29sdmVyIGludGVyZmFjZSBhbmQgdGhlIHRyYWluLXZlcmlmaWNhdGlvbiBjb250cmFjdC4KCkEgU29sdmVyIGV4YW1pbmVzIGEgVGFzayBhbmQgcHJvcG9zZXMsIGZvciBlYWNoIHRlc3QgaW5wdXQgKGluIG9yZGVyKSwgYSByYW5rZWQKbGlzdCBvZiBjYW5kaWRhdGUgb3V0cHV0IGdyaWRzIChiZXN0IGZpcnN0LCBwb3NzaWJseSBlbXB0eSkuIFRoZSBwaXBlbGluZSBtZXJnZXMKY2FuZGlkYXRlcyBmcm9tIHNldmVyYWwgc29sdmVycyBpbnRvIHRoZSBmaW5hbCB0d28gYXR0ZW1wdHMuCgpUaGUgc2hhcmVkIGB2ZXJpZnlfcHJvZ3JhbWAgaGVscGVyIGVuY29kZXMgdGhlIGNlbnRyYWwgc2FmZXR5IHByaW5jaXBsZTogYQpncmlkLT5ncmlkIHRyYW5zZm9ybWF0aW9uIGlzIG9ubHkgdHJ1c3R3b3J0aHkgaWYgaXQgcmVwcm9kdWNlcyBFVkVSWQpkZW1vbnN0cmF0aW9uIG91dHB1dCBleGFjdGx5LiBPbiBhIG5vdmVsLCBsYWJlbC1mcmVlIHRlc3QgdGFzayB0aGUgdHJhaW4gcGFpcnMKYXJlIHRoZSBvbmx5IGdyb3VuZCB0cnV0aCBhdmFpbGFibGUsIHNvIHRoZXkgZG91YmxlIGFzIHRoZSBhY2NlcHRhbmNlIHRlc3QuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFiYwpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgQ2FsbGFibGUKCmZyb20gLi5pby5ncmlkIGltcG9ydCBHcmlkLCBncmlkc19lcXVhbApmcm9tIC4uaW8ubG9hZGVyIGltcG9ydCBUYXNrCgojIEEgcHJvZ3JhbSBpcyBhbnkgZ3JpZCAtPiBncmlkIGZ1bmN0aW9uIChtYXkgcmFpc2U7IGNhbGxlcnMgZ3VhcmQpLgpQcm9ncmFtID0gQ2FsbGFibGVbW0dyaWRdLCBHcmlkXQoKIyBQZXIgdGVzdCBpbnB1dCwgYSByYW5rZWQgbGlzdCBvZiBjYW5kaWRhdGUgZ3JpZHMuCkNhbmRpZGF0ZXMgPSBsaXN0W2xpc3RbR3JpZF1dCgoKZGVmIHZlcmlmeV9wcm9ncmFtKHByb2dyYW06IFByb2dyYW0sIHRhc2s6IFRhc2spIC0+IGJvb2w6CiAgICAiIiJUcnVlIGlmZiBgcHJvZ3JhbWAgbWFwcyBldmVyeSB0cmFpbiBpbnB1dCB0byBpdHMgZXhhY3QgdHJhaW4gb3V0cHV0LiIiIgogICAgaWYgbm90IHRhc2sudHJhaW46CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICBmb3IgcGFpciBpbiB0YXNrLnRyYWluOgogICAgICAgIGlmIHBhaXIub3V0cHV0IGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIHRyeToKICAgICAgICAgICAgcHJlZGljdGVkID0gcHJvZ3JhbShwYWlyLmlucHV0KQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIGlmIG5vdCBncmlkc19lcXVhbChwcmVkaWN0ZWQsIHBhaXIub3V0cHV0KToKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICByZXR1cm4gVHJ1ZQoKCmRlZiBhcHBseV90b190ZXN0cyhwcm9ncmFtOiBQcm9ncmFtLCB0YXNrOiBUYXNrKSAtPiBsaXN0W0dyaWQgfCBOb25lXToKICAgICIiIkFwcGx5IGEgKHZlcmlmaWVkKSBwcm9ncmFtIHRvIGVhY2ggdGVzdCBpbnB1dDsgTm9uZSBvbiBmYWlsdXJlLiIiIgogICAgb3V0OiBsaXN0W0dyaWQgfCBOb25lXSA9IFtdCiAgICBmb3IgcGFpciBpbiB0YXNrLnRlc3Q6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBvdXQuYXBwZW5kKHByb2dyYW0ocGFpci5pbnB1dCkpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgb3V0LmFwcGVuZChOb25lKQogICAgcmV0dXJuIG91dAoKCmNsYXNzIFNvbHZlcihhYmMuQUJDKToKICAgICIiIkJhc2UgY2xhc3MgZm9yIGFsbCBzb2x2ZXJzLiIiIgoKICAgIG5hbWU6IHN0ciA9ICJzb2x2ZXIiCgogICAgQGFiYy5hYnN0cmFjdG1ldGhvZAogICAgZGVmIHNvbHZlKHNlbGYsIHRhc2s6IFRhc2ssIGJ1ZGdldF9zOiBmbG9hdCkgLT4gQ2FuZGlkYXRlczoKICAgICAgICAiIiJSZXR1cm4gcmFua2VkIGNhbmRpZGF0ZSBncmlkcyBwZXIgdGVzdCBpbnB1dCAob3V0ZXIgaW5kZXggPSB0ZXN0IGkpLiIiIgogICAgICAgIHJhaXNlIE5vdEltcGxlbWVudGVkRXJyb3IKCiAgICBkZWYgX2VtcHR5KHNlbGYsIHRhc2s6IFRhc2spIC0+IENhbmRpZGF0ZXM6CiAgICAgICAgcmV0dXJuIFtbXSBmb3IgXyBpbiB0YXNrLnRlc3RdCg==",
"src/arc/solvers/dsl/__init__.py": "",
"src/arc/solvers/dsl/primitives.py": "IiIiR3JpZC0+Z3JpZCBwcmltaXRpdmVzIGZvciB0aGUgRFNMIG1pY3JvLXNvbHZlci4KClR3byBraW5kczoKICAqIFBhcmFtZXRlci1mcmVlIGdlb21ldHJpYyBvcHMgKHJvdGF0aW9ucywgZmxpcHMsIGNyb3AtdG8tY29udGVudCkuCiAgKiBQYXJhbWV0ZXIgb3BzIHdob3NlIHBhcmFtZXRlcnMgYXJlIElORkVSUkVEIGZyb20gdGhlIHRhc2sncyB0cmFpbiBwYWlycwogICAgKGludGVnZXIgc2NhbGluZywgdGlsaW5nLCBhbmQgYSBsZWFybmVkIGNlbGx3aXNlIGNvbG91ciBtYXApLiBJbmZlcnJpbmcKICAgIHBhcmFtZXRlcnMgZnJvbSB0aGUgZGVtb25zdHJhdGlvbnMga2VlcHMgdGhlIHNlYXJjaCBzcGFjZSB0aW55IHdoaWxlIHN0aWxsCiAgICBjb3ZlcmluZyBhIGxhcmdlIHNsaWNlIG9mIGNvbW1vbiBBUkMgdHJhbnNmb3JtYXRpb25zLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuLi5hdWdtZW50IGltcG9ydCBzeW1tZXRyeQpmcm9tIC4uLmlvLmdyaWQgaW1wb3J0IEdyaWQsIGJhY2tncm91bmRfY29sb3IsIGZyb21fbnVtcHksIHNoYXBlLCB0b19udW1weQpmcm9tIC4uLmlvLmxvYWRlciBpbXBvcnQgVGFzawoKIyBBUkMgZ3JpZHMgYXJlIGF0IG1vc3QgMzB4MzA7IGEgcHJvZ3JhbSB3aG9zZSBvdXRwdXQgd291bGQgZXhjZWVkIHRoaXMgY2FuIG5ldmVyCiMgYmUgYSBjb3JyZWN0IGFuc3dlciwgc28gc2NhbGUvdGlsZSByZWZ1c2UgdG8gYWxsb2NhdGUgYmV5b25kIGl0IChib3VuZHMgbWVtb3J5CiMgb24gYSBwYXRob2xvZ2ljYWwgdGVzdCBpbnB1dCBhbmQgbGV0cyB0aGUgb3ZlcnNpemVkIHByb2dyYW0gZmFpbCB2ZXJpZmljYXRpb24pLgpNQVhfR1JJRF9ESU0gPSAzMAoKCiMgLS0tLSBQYXJhbWV0ZXItZnJlZSBnZW9tZXRyaWMgb3BzIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBjcm9wX3RvX2NvbnRlbnQoZ3JpZDogR3JpZCkgLT4gR3JpZDoKICAgICIiIkNyb3AgdG8gdGhlIGJvdW5kaW5nIGJveCBvZiBub24tYmFja2dyb3VuZCBjZWxscyAoYmFja2dyb3VuZCA9IG1vZGUpLiIiIgogICAgYmcgPSBiYWNrZ3JvdW5kX2NvbG9yKGdyaWQpCiAgICBhcnIgPSB0b19udW1weShncmlkKQogICAgbWFzayA9IGFyciAhPSBiZwogICAgaWYgbm90IG1hc2suYW55KCk6CiAgICAgICAgcmV0dXJuIGdyaWQKICAgIHJvd3MgPSBucC5hbnkobWFzaywgYXhpcz0xKQogICAgY29scyA9IG5wLmFueShtYXNrLCBheGlzPTApCiAgICByMCwgcjEgPSBucC53aGVyZShyb3dzKVswXVtbMCwgLTFdXQogICAgYzAsIGMxID0gbnAud2hlcmUoY29scylbMF1bWzAsIC0xXV0KICAgIHJldHVybiBmcm9tX251bXB5KGFycltyMCA6IHIxICsgMSwgYzAgOiBjMSArIDFdKQoKClBBUkFNX0ZSRUU6IGRpY3Rbc3RyLCBjYWxsYWJsZV0gPSB7CiAgICAiaWRlbnRpdHkiOiBsYW1iZGEgZzogZywKICAgICJyb3Q5MCI6IGxhbWJkYSBnOiBzeW1tZXRyeS5hcHBseSgicm90OTAiLCBnKSwKICAgICJyb3QxODAiOiBsYW1iZGEgZzogc3ltbWV0cnkuYXBwbHkoInJvdDE4MCIsIGcpLAogICAgInJvdDI3MCI6IGxhbWJkYSBnOiBzeW1tZXRyeS5hcHBseSgicm90MjcwIiwgZyksCiAgICAiZmxpcF9oIjogbGFtYmRhIGc6IHN5bW1ldHJ5LmFwcGx5KCJmbGlwX2giLCBnKSwKICAgICJmbGlwX3YiOiBsYW1iZGEgZzogc3ltbWV0cnkuYXBwbHkoImZsaXBfdiIsIGcpLAogICAgInRyYW5zcG9zZSI6IGxhbWJkYSBnOiBzeW1tZXRyeS5hcHBseSgidHJhbnNwb3NlIiwgZyksCiAgICAiYW50aV90cmFuc3Bvc2UiOiBsYW1iZGEgZzogc3ltbWV0cnkuYXBwbHkoImFudGlfdHJhbnNwb3NlIiwgZyksCiAgICAiY3JvcF90b19jb250ZW50IjogY3JvcF90b19jb250ZW50LAp9CgoKIyAtLS0tIEluZmVycmVkIHBhcmFtZXRlciBvcHMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KZGVmIF9zaGFwZV9yYXRpbyh0YXNrOiBUYXNrKSAtPiB0dXBsZVtpbnQsIGludF0gfCBOb25lOgogICAgIiIiQ29uc2lzdGVudCBpbnRlZ2VyIChvdXQvaW4pIHNoYXBlIHJhdGlvIGFjcm9zcyB0cmFpbiBwYWlycywgb3IgTm9uZS4iIiIKICAgIHJhdGlvczogc2V0W3R1cGxlW2ludCwgaW50XV0gPSBzZXQoKQogICAgZm9yIHBhaXIgaW4gdGFzay50cmFpbjoKICAgICAgICBpZiBwYWlyLm91dHB1dCBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIGloLCBpdyA9IHNoYXBlKHBhaXIuaW5wdXQpCiAgICAgICAgb2gsIG93ID0gc2hhcGUocGFpci5vdXRwdXQpCiAgICAgICAgaWYgaWggPT0gMCBvciBpdyA9PSAwIG9yIG9oICUgaWggb3Igb3cgJSBpdzoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICByYXRpb3MuYWRkKChvaCAvLyBpaCwgb3cgLy8gaXcpKQogICAgaWYgbGVuKHJhdGlvcykgIT0gMToKICAgICAgICByZXR1cm4gTm9uZQogICAgZnksIGZ4ID0gbmV4dChpdGVyKHJhdGlvcykpCiAgICByZXR1cm4gKGZ5LCBmeCkgaWYgKGZ5LCBmeCkgIT0gKDEsIDEpIGVsc2UgTm9uZQoKCmRlZiBzY2FsZV9wcm9ncmFtKGZ5OiBpbnQsIGZ4OiBpbnQpOgogICAgIiIiRWFjaCBjZWxsIGJlY29tZXMgYW4gZnkgeCBmeCBibG9jayAobnAucmVwZWF0KS4iIiIKCiAgICBkZWYgZihnOiBHcmlkKSAtPiBHcmlkOgogICAgICAgIGFyciA9IHRvX251bXB5KGcpCiAgICAgICAgaWYgYXJyLnNoYXBlWzBdICogZnkgPiBNQVhfR1JJRF9ESU0gb3IgYXJyLnNoYXBlWzFdICogZnggPiBNQVhfR1JJRF9ESU06CiAgICAgICAgICAgIHJldHVybiBnICAjIG92ZXJzaXplZCAtPiBwYXNzIHRocm91Z2ggc28gdGhlIHByb2dyYW0gZmFpbHMgdG8gdmVyaWZ5CiAgICAgICAgcmV0dXJuIGZyb21fbnVtcHkobnAucmVwZWF0KG5wLnJlcGVhdChhcnIsIGZ5LCBheGlzPTApLCBmeCwgYXhpcz0xKSkKCiAgICByZXR1cm4gZgoKCmRlZiB0aWxlX3Byb2dyYW0obnk6IGludCwgbng6IGludCk6CiAgICAiIiJSZXBlYXQgdGhlIHdob2xlIGdyaWQgbnkgeCBueCB0aW1lcyAobnAudGlsZSkuIiIiCgogICAgZGVmIGYoZzogR3JpZCkgLT4gR3JpZDoKICAgICAgICBhcnIgPSB0b19udW1weShnKQogICAgICAgIGlmIGFyci5zaGFwZVswXSAqIG55ID4gTUFYX0dSSURfRElNIG9yIGFyci5zaGFwZVsxXSAqIG54ID4gTUFYX0dSSURfRElNOgogICAgICAgICAgICByZXR1cm4gZyAgIyBvdmVyc2l6ZWQgLT4gcGFzcyB0aHJvdWdoIHNvIHRoZSBwcm9ncmFtIGZhaWxzIHRvIHZlcmlmeQogICAgICAgIHJldHVybiBmcm9tX251bXB5KG5wLnRpbGUoYXJyLCAobnksIG54KSkpCgogICAgcmV0dXJuIGYKCgpkZWYgbGVhcm5fY29sb3JtYXAodGFzazogVGFzaykgLT4gZGljdFtpbnQsIGludF0gfCBOb25lOgogICAgIiIiTGVhcm4gYSBjb25zaXN0ZW50IGNlbGx3aXNlIHN5bWJvbCBtYXAgaWYgZXZlcnkgdHJhaW4gcGFpciBpcyBhIHB1cmUKICAgIHNhbWUtc2hhcGUgcmVjb2xvdXI7IGVsc2UgTm9uZS4iIiIKICAgIG1hcHBpbmc6IGRpY3RbaW50LCBpbnRdID0ge30KICAgIGZvciBwYWlyIGluIHRhc2sudHJhaW46CiAgICAgICAgaWYgcGFpci5vdXRwdXQgaXMgTm9uZSBvciBzaGFwZShwYWlyLmlucHV0KSAhPSBzaGFwZShwYWlyLm91dHB1dCk6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgZm9yIGluX3Jvdywgb3V0X3JvdyBpbiB6aXAocGFpci5pbnB1dCwgcGFpci5vdXRwdXQsIHN0cmljdD1GYWxzZSk6CiAgICAgICAgICAgIGZvciBzLCBkIGluIHppcChpbl9yb3csIG91dF9yb3csIHN0cmljdD1GYWxzZSk6CiAgICAgICAgICAgICAgICBpZiBzIGluIG1hcHBpbmcgYW5kIG1hcHBpbmdbc10gIT0gZDoKICAgICAgICAgICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgICAgICAgICAgbWFwcGluZ1tzXSA9IGQKICAgIHJldHVybiBtYXBwaW5nIG9yIE5vbmUKCgpkZWYgY29sb3JtYXBfcHJvZ3JhbShtYXBwaW5nOiBkaWN0W2ludCwgaW50XSk6CiAgICAiIiJBcHBseSBhIGxlYXJuZWQgY29sb3VyIG1hcDsgdW5rbm93biBzeW1ib2xzIHBhc3MgdGhyb3VnaCB1bmNoYW5nZWQuIiIiCgogICAgZGVmIGYoZzogR3JpZCkgLT4gR3JpZDoKICAgICAgICByZXR1cm4gdHVwbGUodHVwbGUobWFwcGluZy5nZXQoYywgYykgZm9yIGMgaW4gcm93KSBmb3Igcm93IGluIGcpCgogICAgcmV0dXJuIGYK",
"src/arc/solvers/dsl/search.py": "IiIiQm91bmRlZC1kZXB0aCBjb21wb3NpdGlvbiBzZWFyY2ggb3ZlciBEU0wgcHJpbWl0aXZlcy4KClN0cmF0ZWd5OiBidWlsZCBhIHNtYWxsIGJhc2Ugdm9jYWJ1bGFyeSAocGFyYW1ldGVyLWZyZWUgZ2VvbWV0cmljIG9wcyBwbHVzIGFueQpwYXJhbWV0ZXJzIGluZmVycmVkIGZyb20gdGhlIHRyYWluIHBhaXJzIOKAlCBzY2FsZSwgdGlsZSwgY29sb3VyIG1hcCksIGVudW1lcmF0ZQpzaW5nbGUgb3BzIGFuZCBkZXB0aC0yIGNvbXBvc2l0aW9ucywga2VlcCBvbmx5IHByb2dyYW1zIHRoYXQgcmVwcm9kdWNlIEFMTCB0cmFpbgpvdXRwdXRzLCBhbmQgcmFuayBzdXJ2aXZvcnMgYnkgc2ltcGxpY2l0eS4gVGlueSBieSBkZXNpZ24gc28gaXQgcnVucyBpbgptaWxsaXNlY29uZHMgYW5kIG5ldmVyIGJsb3dzIHRoZSBwZXItdGFzayBidWRnZXQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRpbWUKCmZyb20gLi4uaW8ubG9hZGVyIGltcG9ydCBUYXNrCmZyb20gLi5iYXNlIGltcG9ydCBQcm9ncmFtLCB2ZXJpZnlfcHJvZ3JhbQpmcm9tIC4gaW1wb3J0IHByaW1pdGl2ZXMgYXMgUAoKTGFiZWxlZFByb2dyYW0gPSB0dXBsZVtpbnQsIHN0ciwgUHJvZ3JhbV0gICMgKGNvbXBsZXhpdHksIGxhYmVsLCBwcm9ncmFtKQoKCmRlZiBfY29tcG9zZShmOiBQcm9ncmFtLCBnOiBQcm9ncmFtKSAtPiBQcm9ncmFtOgogICAgIiIiUmV0dXJuIHRoZSBwcm9ncmFtIHggLT4gZyhmKHgpKS4iIiIKICAgIHJldHVybiBsYW1iZGEgeDogZyhmKHgpKQoKCmRlZiBfYmFzZV92b2NhYnVsYXJ5KHRhc2s6IFRhc2spIC0+IGxpc3RbdHVwbGVbc3RyLCBQcm9ncmFtXV06CiAgICAiIiJQYXJhbWV0ZXItZnJlZSBvcHMgcGx1cyB0YXNrLWluZmVycmVkIHBhcmFtZXRlciBvcHMuIiIiCiAgICB2b2NhYjogbGlzdFt0dXBsZVtzdHIsIFByb2dyYW1dXSA9IGxpc3QoUC5QQVJBTV9GUkVFLml0ZW1zKCkpCgogICAgcmF0aW8gPSBQLl9zaGFwZV9yYXRpbyh0YXNrKQogICAgaWYgcmF0aW8gaXMgbm90IE5vbmU6CiAgICAgICAgZnksIGZ4ID0gcmF0aW8KICAgICAgICB2b2NhYi5hcHBlbmQoKGYic2NhbGV7Znl9eHtmeH0iLCBQLnNjYWxlX3Byb2dyYW0oZnksIGZ4KSkpCiAgICAgICAgdm9jYWIuYXBwZW5kKChmInRpbGV7Znl9eHtmeH0iLCBQLnRpbGVfcHJvZ3JhbShmeSwgZngpKSkKCiAgICBjbWFwID0gUC5sZWFybl9jb2xvcm1hcCh0YXNrKQogICAgaWYgY21hcCBpcyBub3QgTm9uZToKICAgICAgICB2b2NhYi5hcHBlbmQoKCJjb2xvcm1hcCIsIFAuY29sb3JtYXBfcHJvZ3JhbShjbWFwKSkpCgogICAgcmV0dXJuIHZvY2FiCgoKZGVmIGdlbmVyYXRlX3Byb2dyYW1zKHRhc2s6IFRhc2spIC0+IGxpc3RbdHVwbGVbc3RyLCBQcm9ncmFtXV06CiAgICAiIiJTaW5nbGUgb3BzIGFuZCBkZXB0aC0yIGNvbXBvc2l0aW9ucyBvdmVyIHRoZSBiYXNlIHZvY2FidWxhcnkuIiIiCiAgICB2b2NhYiA9IF9iYXNlX3ZvY2FidWxhcnkodGFzaykKICAgIHByb2dyYW1zOiBsaXN0W3R1cGxlW3N0ciwgUHJvZ3JhbV1dID0gbGlzdCh2b2NhYikgICMgZGVwdGggMQogICAgZm9yIG4xLCBmMSBpbiB2b2NhYjoKICAgICAgICBpZiBuMSA9PSAiaWRlbnRpdHkiOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZvciBuMiwgZjIgaW4gdm9jYWI6CiAgICAgICAgICAgIGlmIG4yID09ICJpZGVudGl0eSI6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBwcm9ncmFtcy5hcHBlbmQoKGYie24xfXx7bjJ9IiwgX2NvbXBvc2UoZjEsIGYyKSkpICAjIGRlcHRoIDIKICAgIHJldHVybiBwcm9ncmFtcwoKCmRlZiBzZWFyY2godGFzazogVGFzaywgYnVkZ2V0X3M6IGZsb2F0ID0gNS4wKSAtPiBsaXN0W3R1cGxlW3N0ciwgUHJvZ3JhbV1dOgogICAgIiIiUmV0dXJuIHZlcmlmaWVkIHByb2dyYW1zLCBzaW1wbGVzdCAoZmV3ZXN0IG9wcykgZmlyc3QsIGRlZHVwZWQgYnkgbGFiZWwuIiIiCiAgICBkZWFkbGluZSA9IHRpbWUubW9ub3RvbmljKCkgKyBidWRnZXRfcwogICAgdmVyaWZpZWQ6IGxpc3RbdHVwbGVbaW50LCBzdHIsIFByb2dyYW1dXSA9IFtdCiAgICBzZWVuX2xhYmVsczogc2V0W3N0cl0gPSBzZXQoKQogICAgZm9yIGxhYmVsLCBwcm9ncmFtIGluIGdlbmVyYXRlX3Byb2dyYW1zKHRhc2spOgogICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgPiBkZWFkbGluZToKICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBsYWJlbCBpbiBzZWVuX2xhYmVsczoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiB2ZXJpZnlfcHJvZ3JhbShwcm9ncmFtLCB0YXNrKToKICAgICAgICAgICAgY29tcGxleGl0eSA9IGxhYmVsLmNvdW50KCJ8IikgKyAxCiAgICAgICAgICAgIHZlcmlmaWVkLmFwcGVuZCgoY29tcGxleGl0eSwgbGFiZWwsIHByb2dyYW0pKQogICAgICAgICAgICBzZWVuX2xhYmVscy5hZGQobGFiZWwpCiAgICB2ZXJpZmllZC5zb3J0KGtleT1sYW1iZGEgdDogKHRbMF0sIHRbMV0pKQogICAgcmV0dXJuIFsobGFiZWwsIHByb2dyYW0pIGZvciBfLCBsYWJlbCwgcHJvZ3JhbSBpbiB2ZXJpZmllZF0K",
"src/arc/solvers/dsl/solver.py": "IiIiRFNMU29sdmVyIOKAlCB3cmFwcyB0aGUgcHJvZ3JhbSBzZWFyY2ggYmVoaW5kIHRoZSBTb2x2ZXIgaW50ZXJmYWNlLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSAuLi5pby5sb2FkZXIgaW1wb3J0IFRhc2sKZnJvbSAuLmJhc2UgaW1wb3J0IENhbmRpZGF0ZXMsIFNvbHZlciwgYXBwbHlfdG9fdGVzdHMKZnJvbSAuc2VhcmNoIGltcG9ydCBzZWFyY2gKCgpjbGFzcyBEU0xTb2x2ZXIoU29sdmVyKToKICAgICIiIlNlYXJjaCBmb3IgZ3JpZCBwcm9ncmFtcyB0aGF0IHJlcHJvZHVjZSBldmVyeSB0cmFpbiBwYWlyLCB0aGVuIGFwcGx5IHRoZQogICAgc2ltcGxlc3Qgc3Vydml2b3JzIHRvIGVhY2ggdGVzdCBpbnB1dCBhcyByYW5rZWQgY2FuZGlkYXRlcy4iIiIKCiAgICBuYW1lID0gImRzbCIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgbWF4X2NhbmRpZGF0ZXM6IGludCA9IDQpOgogICAgICAgIHNlbGYubWF4X2NhbmRpZGF0ZXMgPSBtYXhfY2FuZGlkYXRlcwoKICAgIGRlZiBzb2x2ZShzZWxmLCB0YXNrOiBUYXNrLCBidWRnZXRfczogZmxvYXQpIC0+IENhbmRpZGF0ZXM6CiAgICAgICAgcHJvZ3JhbXMgPSBzZWFyY2godGFzaywgYnVkZ2V0X3M9YnVkZ2V0X3MpCiAgICAgICAgaWYgbm90IHByb2dyYW1zOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fZW1wdHkodGFzaykKCiAgICAgICAgcGVyX3Rlc3Q6IENhbmRpZGF0ZXMgPSBbW10gZm9yIF8gaW4gdGFzay50ZXN0XQogICAgICAgIGZvciBfbGFiZWwsIHByb2dyYW0gaW4gcHJvZ3JhbXM6CiAgICAgICAgICAgIHByZWRzID0gYXBwbHlfdG9fdGVzdHMocHJvZ3JhbSwgdGFzaykKICAgICAgICAgICAgZm9yIGksIHByZWQgaW4gZW51bWVyYXRlKHByZWRzKToKICAgICAgICAgICAgICAgIGlmIHByZWQgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgYnVja2V0ID0gcGVyX3Rlc3RbaV0KICAgICAgICAgICAgICAgIGlmIHByZWQgbm90IGluIGJ1Y2tldCBhbmQgbGVuKGJ1Y2tldCkgPCBzZWxmLm1heF9jYW5kaWRhdGVzOgogICAgICAgICAgICAgICAgICAgIGJ1Y2tldC5hcHBlbmQocHJlZCkKICAgICAgICByZXR1cm4gcGVyX3Rlc3QK",
"src/arc/solvers/identity.py": "IiIiQ2hlYXAsIGFsd2F5cy1hdmFpbGFibGUgaGV1cmlzdGljIHNvbHZlcnMgKG1pY3Jvc2Vjb25kIGNvc3QpLgoKVGhlc2UgZXhpc3QgYXMgYW4gZW5zZW1ibGUgZmxvb3IgYW5kIGEgZmFsbGJhY2sgd2hlbiBleHBlbnNpdmUgc29sdmVycyB0aW1lIG91dApvciBPT00uIEVhY2ggc3RpbGwgcmVzcGVjdHMgdGhlIHZlcmlmeS1vbi10cmFpbiBjb250cmFjdCB3aGVyZSBhcHBsaWNhYmxlLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKCmZyb20gLi5pby5ncmlkIGltcG9ydCBHcmlkCmZyb20gLi5pby5sb2FkZXIgaW1wb3J0IFRhc2sKZnJvbSAuYmFzZSBpbXBvcnQgQ2FuZGlkYXRlcywgU29sdmVyLCBhcHBseV90b190ZXN0cywgdmVyaWZ5X3Byb2dyYW0KCgpjbGFzcyBJZGVudGl0eVNvbHZlcihTb2x2ZXIpOgogICAgIiIiUHJlZGljdCBvdXRwdXQgPT0gaW5wdXQuIFdpbnMgdGhlIChyYXJlKSB0YXNrcyB3aGVyZSB0aGUgZ3JpZCBpcyB1bmNoYW5nZWQuIiIiCgogICAgbmFtZSA9ICJpZGVudGl0eSIKCiAgICBkZWYgc29sdmUoc2VsZiwgdGFzazogVGFzaywgYnVkZ2V0X3M6IGZsb2F0KSAtPiBDYW5kaWRhdGVzOgogICAgICAgIGlmIG5vdCB2ZXJpZnlfcHJvZ3JhbShsYW1iZGEgZzogZywgdGFzayk6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9lbXB0eSh0YXNrKQogICAgICAgIHByZWRzID0gYXBwbHlfdG9fdGVzdHMobGFtYmRhIGc6IGcsIHRhc2spCiAgICAgICAgcmV0dXJuIFtbcF0gaWYgcCBpcyBub3QgTm9uZSBlbHNlIFtdIGZvciBwIGluIHByZWRzXQoKCmNsYXNzIENvbnN0YW50T3V0cHV0U29sdmVyKFNvbHZlcik6CiAgICAiIiJJZiBhbGwgdHJhaW4gb3V0cHV0cyBhcmUgaWRlbnRpY2FsLCBwcmVkaWN0IHRoYXQgY29uc3RhbnQgZ3JpZC4KCiAgICBDYXB0dXJlcyB0YXNrcyB3aG9zZSBhbnN3ZXIgaWdub3JlcyB0aGUgaW5wdXQgKGUuZy4gYSBmaXhlZCBsZWdlbmQva2V5KS4KICAgICIiIgoKICAgIG5hbWUgPSAiY29uc3RhbnRfb3V0cHV0IgoKICAgIGRlZiBzb2x2ZShzZWxmLCB0YXNrOiBUYXNrLCBidWRnZXRfczogZmxvYXQpIC0+IENhbmRpZGF0ZXM6CiAgICAgICAgb3V0cHV0cyA9IFtwLm91dHB1dCBmb3IgcCBpbiB0YXNrLnRyYWluIGlmIHAub3V0cHV0IGlzIG5vdCBOb25lXQogICAgICAgIGlmIG5vdCBvdXRwdXRzOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fZW1wdHkodGFzaykKICAgICAgICBmaXJzdCA9IG91dHB1dHNbMF0KICAgICAgICBpZiBhbGwobyA9PSBmaXJzdCBmb3IgbyBpbiBvdXRwdXRzKToKICAgICAgICAgICAgcmV0dXJuIFtbZmlyc3RdIGZvciBfIGluIHRhc2sudGVzdF0KICAgICAgICByZXR1cm4gc2VsZi5fZW1wdHkodGFzaykKCgpjbGFzcyBNYWpvcml0eVNoYXBlU29sdmVyKFNvbHZlcik6CiAgICAiIiJXZWFrIHByaW9yOiBlbWl0IGEgYmFja2dyb3VuZC1maWxsZWQgZ3JpZCBhdCB0aGUgbW9zdCBjb21tb24gb3V0cHV0IHNoYXBlLgoKICAgIE5ldmVyIHZlcmlmaWVkIGFnYWluc3QgdHJhaW4sIHNvIHRoZSBwaXBlbGluZSBtdXN0IHJhbmsgaXQgYmVsb3cgdmVyaWZpZWQKICAgIGNhbmRpZGF0ZXMuIFVzZWZ1bCBvbmx5IGFzIGEgbGFzdC1yZXNvcnQsIG5vbi10cml2aWFsIGZhbGxiYWNrIHNoYXBlLgogICAgIiIiCgogICAgbmFtZSA9ICJtYWpvcml0eV9zaGFwZSIKCiAgICBkZWYgc29sdmUoc2VsZiwgdGFzazogVGFzaywgYnVkZ2V0X3M6IGZsb2F0KSAtPiBDYW5kaWRhdGVzOgogICAgICAgIHNoYXBlcyA9IFsKICAgICAgICAgICAgKGxlbihwLm91dHB1dCksIGxlbihwLm91dHB1dFswXSkpCiAgICAgICAgICAgIGZvciBwIGluIHRhc2sudHJhaW4KICAgICAgICAgICAgaWYgcC5vdXRwdXQgaXMgbm90IE5vbmUgYW5kIHAub3V0cHV0CiAgICAgICAgXQogICAgICAgIGlmIG5vdCBzaGFwZXM6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9lbXB0eSh0YXNrKQogICAgICAgIChoLCB3KSwgXyA9IENvdW50ZXIoc2hhcGVzKS5tb3N0X2NvbW1vbigxKVswXQogICAgICAgICMgTW9zdCBmcmVxdWVudCBzeW1ib2wgYWNyb3NzIHRyYWluIG91dHB1dHMgYXMgdGhlIGZpbGwgY29sb3VyLgogICAgICAgIGZpbGwgPSBfbW9zdF9jb21tb25fc3ltYm9sKHRhc2spCiAgICAgICAgZ3JpZDogR3JpZCA9IHR1cGxlKHR1cGxlKGZpbGwgZm9yIF8gaW4gcmFuZ2UodykpIGZvciBfIGluIHJhbmdlKGgpKQogICAgICAgIHJldHVybiBbW2dyaWRdIGZvciBfIGluIHRhc2sudGVzdF0KCgpkZWYgX21vc3RfY29tbW9uX3N5bWJvbCh0YXNrOiBUYXNrKSAtPiBpbnQ6CiAgICBjb3VudGVyOiBDb3VudGVyW2ludF0gPSBDb3VudGVyKCkKICAgIGZvciBwYWlyIGluIHRhc2sudHJhaW46CiAgICAgICAgaWYgcGFpci5vdXRwdXQgaXMgTm9uZToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmb3Igcm93IGluIHBhaXIub3V0cHV0OgogICAgICAgICAgICBjb3VudGVyLnVwZGF0ZShyb3cpCiAgICByZXR1cm4gY291bnRlci5tb3N0X2NvbW1vbigxKVswXVswXSBpZiBjb3VudGVyIGVsc2UgMAoKCkNIRUFQX1NPTFZFUlM6IHR1cGxlW1NvbHZlciwgLi4uXSA9ICgKICAgIElkZW50aXR5U29sdmVyKCksCiAgICBDb25zdGFudE91dHB1dFNvbHZlcigpLAogICAgTWFqb3JpdHlTaGFwZVNvbHZlcigpLAopCg==",
"src/arc/solvers/llm/__init__.py": "IiIiTExNIHRyYW5zZHVjdGlvbiArIHRlc3QtdGltZS10cmFpbmluZyBzb2x2ZXJzIChNMSspLiIiIgoKZnJvbSAubW9kZWwgaW1wb3J0IEhGTW9kZWwsIExhbmd1YWdlTW9kZWwsIE1vY2tNb2RlbApmcm9tIC5zb2x2ZXIgaW1wb3J0IExMTVNvbHZlcgpmcm9tIC50dHQgaW1wb3J0ICgKICAgIExvcmFUVFRSdW5uZXIsCiAgICBNb2NrVFRUUnVubmVyLAogICAgVFRUQ29uZmlnLAogICAgVFRUUnVubmVyLAogICAgVFRUU29sdmVyLAopCmZyb20gLnR0dF9kYXRhIGltcG9ydCBUcmFpbkV4YW1wbGUsIGJ1aWxkX3R0dF9leGFtcGxlcwoKX19hbGxfXyA9IFsKICAgICJIRk1vZGVsIiwKICAgICJMYW5ndWFnZU1vZGVsIiwKICAgICJNb2NrTW9kZWwiLAogICAgIkxMTVNvbHZlciIsCiAgICAiVFRUU29sdmVyIiwKICAgICJUVFRSdW5uZXIiLAogICAgIk1vY2tUVFRSdW5uZXIiLAogICAgIkxvcmFUVFRSdW5uZXIiLAogICAgIlRUVENvbmZpZyIsCiAgICAiVHJhaW5FeGFtcGxlIiwKICAgICJidWlsZF90dHRfZXhhbXBsZXMiLApdCg==",
"src/arc/solvers/llm/infer.py": "IiIiUnVuIGEgbGFuZ3VhZ2UgbW9kZWwgb24gb25lICh0cmFpbiwgdGVzdF9pbnB1dCkgYW5kIHBhcnNlIGdyaWQgY2FuZGlkYXRlcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBTZXF1ZW5jZQoKZnJvbSAuLi5pby5ncmlkIGltcG9ydCBHcmlkCmZyb20gLi4uaW8ubG9hZGVyIGltcG9ydCBQYWlyCmZyb20gLi4uc2VyaWFsaXplLnByb21wdCBpbXBvcnQgYnVpbGRfcHJvbXB0LCBwYXJzZV9jb21wbGV0aW9uCmZyb20gLm1vZGVsIGltcG9ydCBMYW5ndWFnZU1vZGVsCgoKZGVmIGdlbmVyYXRlX2NhbmRpZGF0ZXMoCiAgICBtb2RlbDogTGFuZ3VhZ2VNb2RlbCwKICAgIHRyYWluOiBTZXF1ZW5jZVtQYWlyXSwKICAgIHRlc3RfaW5wdXQ6IEdyaWQsCiAgICAqLAogICAgbnVtX3NhbXBsZXM6IGludCA9IDEsCiAgICBtYXhfbmV3X3Rva2VuczogaW50ID0gMTAyNCwKICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9IDAuMCwKICAgIG1heF90aW1lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtHcmlkXToKICAgICIiIlByb21wdCB0aGUgbW9kZWwgYW5kIHJldHVybiB0aGUgdmFsaWQgZ3JpZHMgcGFyc2VkIGZyb20gaXRzIGNvbXBsZXRpb25zLiIiIgogICAgcHJvbXB0ID0gYnVpbGRfcHJvbXB0KHRyYWluLCB0ZXN0X2lucHV0KQogICAgY29tcGxldGlvbnMgPSBtb2RlbC5nZW5lcmF0ZSgKICAgICAgICBwcm9tcHQsCiAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgbnVtX3NhbXBsZXM9bnVtX3NhbXBsZXMsCiAgICAgICAgdGVtcGVyYXR1cmU9dGVtcGVyYXR1cmUsCiAgICAgICAgbWF4X3RpbWVfcz1tYXhfdGltZV9zLAogICAgKQogICAgZ3JpZHM6IGxpc3RbR3JpZF0gPSBbXQogICAgZm9yIGNvbXBsZXRpb24gaW4gY29tcGxldGlvbnM6CiAgICAgICAgZ3JpZCA9IHBhcnNlX2NvbXBsZXRpb24oY29tcGxldGlvbikKICAgICAgICBpZiBncmlkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBncmlkcy5hcHBlbmQoZ3JpZCkKICAgIHJldHVybiBncmlkcwo=",
"src/arc/solvers/llm/model.py": "IiIiTGFuZ3VhZ2UtbW9kZWwgYWJzdHJhY3Rpb24gZm9yIHRyYW5zZHVjdGlvbi4KCkRlZmluZXMgYSBtaW5pbWFsIGBMYW5ndWFnZU1vZGVsYCBwcm90b2NvbCAoZ2VuZXJhdGUgY29tcGxldGlvbnM7IHNjb3JlIGEKY29tcGxldGlvbidzIGxvZy1saWtlbGlob29kKSBwbHVzIHR3byBpbXBsZW1lbnRhdGlvbnM6CgogICogYE1vY2tNb2RlbGAg4oCUIHB1cmUtUHl0aG9uLCBDUFUsIG5vIHRvcmNoLiBEZXRlcm1pbmlzdGljOyBhcHBsaWVzIGEKICAgIGNvbmZpZ3VyYWJsZSBncmlkIGB0cmFuc2Zvcm1gIHRvIHRoZSBwcm9tcHQncyBmaW5hbCB0ZXN0IGlucHV0IHNvIHRoZSBlbnRpcmUKICAgIGF1Z21lbnQgLT4gaW5mZXIgLT4gaW52ZXJ0IC0+IHZvdGUgbG9vcCBpcyB1bml0LXRlc3RhYmxlIHdpdGhvdXQgYSBHUFUuCiAgKiBgSEZNb2RlbGAg4oCUIHRoZSByZWFsIG1vZGVsIChRd2VuMi41IGJ5IGRlZmF1bHQpLCBsb2FkZWQgZnJvbSBhIGxvY2FsIHBhdGgKICAgICh0aGUgS2FnZ2xlIGRhdGFzZXQgbW91bnQpLiB0b3JjaC90cmFuc2Zvcm1lcnMgYXJlIGltcG9ydGVkIGxhemlseSBzbyB0aGUKICAgIENQVSBkZXYgZW52aXJvbm1lbnQgbmV2ZXIgbmVlZHMgdGhlbS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgQ2FsbGFibGUKZnJvbSB0eXBpbmcgaW1wb3J0IFByb3RvY29sLCBydW50aW1lX2NoZWNrYWJsZQoKZnJvbSAuLi5pby5ncmlkIGltcG9ydCBHcmlkCmZyb20gLi4uc2VyaWFsaXplLnByb21wdCBpbXBvcnQgZXh0cmFjdF9sYXN0X2lucHV0CmZyb20gLi4uc2VyaWFsaXplLnRva2VuaXplciBpbXBvcnQgZ3JpZF90b19zdHIKCgpAcnVudGltZV9jaGVja2FibGUKY2xhc3MgTGFuZ3VhZ2VNb2RlbChQcm90b2NvbCk6CiAgICAiIiJXaGF0IHRoZSBMTE0gc29sdmVyIG5lZWRzIGZyb20gYW55IG1vZGVsIGJhY2tlbmQuIiIiCgogICAgZGVmIGdlbmVyYXRlKAogICAgICAgIHNlbGYsCiAgICAgICAgcHJvbXB0OiBzdHIsCiAgICAgICAgbWF4X25ld190b2tlbnM6IGludCA9IDEwMjQsCiAgICAgICAgbnVtX3NhbXBsZXM6IGludCA9IDEsCiAgICAgICAgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMC4wLAogICAgICAgIG1heF90aW1lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCiAgICApIC0+IGxpc3Rbc3RyXToKICAgICAgICAiIiJSZXR1cm4gYG51bV9zYW1wbGVzYCBjb21wbGV0aW9uIHN0cmluZ3MgZm9yIGBwcm9tcHRgLiBgbWF4X3RpbWVfc2AsIGlmCiAgICAgICAgZ2l2ZW4sIGNhcHMgZGVjb2RlIHdhbGwtY2xvY2sgc28gYSBzdGFsbGVkIGdlbmVyYXRpb24gY2Fubm90IGJsb3cgdGhlCiAgICAgICAgcGVyLXRhc2sgYnVkZ2V0LiIiIgogICAgICAgIC4uLgoKICAgIGRlZiBzY29yZShzZWxmLCBwcm9tcHQ6IHN0ciwgY29tcGxldGlvbjogc3RyKSAtPiBmbG9hdDoKICAgICAgICAiIiJSZXR1cm4gdGhlIG1lYW4gcGVyLXRva2VuIGxvZy1wcm9iYWJpbGl0eSBvZiBgY29tcGxldGlvbmAgZ2l2ZW4KICAgICAgICBgcHJvbXB0YCAoaGlnaGVyID0gbW9yZSBsaWtlbHkpLiBVc2VkIGZvciBjYW5kaWRhdGUgc2VsZWN0aW9uLiIiIgogICAgICAgIC4uLgoKCmNsYXNzIE1vY2tNb2RlbDoKICAgICIiIkRldGVybWluaXN0aWMgQ1BVIHN0YW5kLWluLiBQcmVkaWN0cyBgdHJhbnNmb3JtKHRlc3RfaW5wdXQpYCBmb3IgdGhlCiAgICBwcm9tcHQncyBmaW5hbCBJbnB1dCBibG9jayAoaWRlbnRpdHkgYnkgZGVmYXVsdCkuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHRyYW5zZm9ybTogQ2FsbGFibGVbW0dyaWRdLCBHcmlkXSB8IE5vbmUgPSBOb25lKToKICAgICAgICBzZWxmLnRyYW5zZm9ybSA9IHRyYW5zZm9ybSBvciAobGFtYmRhIGc6IGcpCgogICAgZGVmIGdlbmVyYXRlKAogICAgICAgIHNlbGYsCiAgICAgICAgcHJvbXB0OiBzdHIsCiAgICAgICAgbWF4X25ld190b2tlbnM6IGludCA9IDEwMjQsCiAgICAgICAgbnVtX3NhbXBsZXM6IGludCA9IDEsCiAgICAgICAgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMC4wLAogICAgICAgIG1heF90aW1lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCiAgICApIC0+IGxpc3Rbc3RyXToKICAgICAgICBncmlkID0gZXh0cmFjdF9sYXN0X2lucHV0KHByb21wdCkKICAgICAgICBpZiBncmlkIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBbIiJdICogbnVtX3NhbXBsZXMKICAgICAgICB0cnk6CiAgICAgICAgICAgIHByZWRpY3RlZCA9IHNlbGYudHJhbnNmb3JtKGdyaWQpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcHJlZGljdGVkID0gZ3JpZAogICAgICAgIHJldHVybiBbZ3JpZF90b19zdHIocHJlZGljdGVkKV0gKiBudW1fc2FtcGxlcwoKICAgIGRlZiBzY29yZShzZWxmLCBwcm9tcHQ6IHN0ciwgY29tcGxldGlvbjogc3RyKSAtPiBmbG9hdDoKICAgICAgICAjIERldGVybWluaXN0aWMgcHNldWRvLXNjb3JlOiBwcmVmZXIgY29tcGxldGlvbnMgdGhhdCBwYXJzZSB0byB0aGUKICAgICAgICAjIHRyYW5zZm9ybSBvZiB0aGUgcHJvbXB0J3MgaW5wdXQgKGV4YWN0IG1hdGNoIC0+IDAuMCwgZWxzZSAtMS4wKS4KICAgICAgICBncmlkID0gZXh0cmFjdF9sYXN0X2lucHV0KHByb21wdCkKICAgICAgICBpZiBncmlkIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiAtMS4wCiAgICAgICAgdHJ5OgogICAgICAgICAgICB0YXJnZXQgPSBncmlkX3RvX3N0cihzZWxmLnRyYW5zZm9ybShncmlkKSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICByZXR1cm4gLTEuMAogICAgICAgIHJldHVybiAwLjAgaWYgY29tcGxldGlvbi5zdHJpcCgpID09IHRhcmdldC5zdHJpcCgpIGVsc2UgLTEuMAoKCmNsYXNzIEhGTW9kZWw6CiAgICAiIiJIdWdnaW5nRmFjZSBjYXVzYWwtTE0gYmFja2VuZCAoS2FnZ2xlL0dQVSkuIExhenkgdG9yY2ggaW1wb3J0LiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIG1vZGVsX3BhdGg6IHN0ciwKICAgICAgICBkZXZpY2U6IHN0ciA9ICJjdWRhIiwKICAgICAgICBkdHlwZTogc3RyID0gImJmbG9hdDE2IiwKICAgICAgICBhZGFwdGVyX3BhdGg6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgKToKICAgICAgICBpbXBvcnQgdG9yY2ggICMgbm9xYTogUExDMDQxNSDigJQgbGF6eTogb25seSBwcmVzZW50IGluIHRoZSBHUFUgZW52CiAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Nb2RlbEZvckNhdXNhbExNLCBBdXRvVG9rZW5pemVyICAjIG5vcWE6IFBMQzA0MTUKCiAgICAgICAgc2VsZi5fdG9yY2ggPSB0b3JjaAogICAgICAgIHNlbGYudG9rZW5pemVyID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQobW9kZWxfcGF0aCkKICAgICAgICBpZiBzZWxmLnRva2VuaXplci5wYWRfdG9rZW5faWQgaXMgTm9uZToKICAgICAgICAgICAgc2VsZi50b2tlbml6ZXIucGFkX3Rva2VuID0gc2VsZi50b2tlbml6ZXIuZW9zX3Rva2VuCiAgICAgICAgc2VsZi5tb2RlbCA9IEF1dG9Nb2RlbEZvckNhdXNhbExNLmZyb21fcHJldHJhaW5lZCgKICAgICAgICAgICAgbW9kZWxfcGF0aCwKICAgICAgICAgICAgdG9yY2hfZHR5cGU9Z2V0YXR0cih0b3JjaCwgZHR5cGUpLAogICAgICAgICAgICBkZXZpY2VfbWFwPWRldmljZSwKICAgICAgICAgICAgdXNlX3NhZmV0ZW5zb3JzPVRydWUsICAjIHJlZnVzZSBwaWNrbGUgLmJpbiBjaGVja3BvaW50cyAoUkNFIHN1cmZhY2UpCiAgICAgICAgKQogICAgICAgIGlmIGFkYXB0ZXJfcGF0aCBpcyBub3QgTm9uZToKICAgICAgICAgICAgZnJvbSBwZWZ0IGltcG9ydCBQZWZ0TW9kZWwgICMgbm9xYTogUExDMDQxNQoKICAgICAgICAgICAgc2VsZi5tb2RlbCA9IFBlZnRNb2RlbC5mcm9tX3ByZXRyYWluZWQoc2VsZi5tb2RlbCwgYWRhcHRlcl9wYXRoKQogICAgICAgIHNlbGYubW9kZWwuZXZhbCgpCiAgICAgICAgc2VsZi5kZXZpY2UgPSBkZXZpY2UKCiAgICBkZWYgZ2VuZXJhdGUoCiAgICAgICAgc2VsZiwKICAgICAgICBwcm9tcHQ6IHN0ciwKICAgICAgICBtYXhfbmV3X3Rva2VuczogaW50ID0gMTAyNCwKICAgICAgICBudW1fc2FtcGxlczogaW50ID0gMSwKICAgICAgICB0ZW1wZXJhdHVyZTogZmxvYXQgPSAwLjAsCiAgICAgICAgbWF4X3RpbWVfczogZmxvYXQgfCBOb25lID0gTm9uZSwKICAgICkgLT4gbGlzdFtzdHJdOgogICAgICAgIHRvcmNoID0gc2VsZi5fdG9yY2gKICAgICAgICBpbnB1dHMgPSBzZWxmLnRva2VuaXplcihwcm9tcHQsIHJldHVybl90ZW5zb3JzPSJwdCIpLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIGRvX3NhbXBsZSA9IHRlbXBlcmF0dXJlID4gMC4wCiAgICAgICAgZ2VuX2t3YXJncyA9IHt9CiAgICAgICAgaWYgbWF4X3RpbWVfcyBpcyBub3QgTm9uZSBhbmQgbWF4X3RpbWVfcyA+IDA6CiAgICAgICAgICAgICMgdHJhbnNmb3JtZXJzIHN0b3BzIGdlbmVyYXRpbmcgb25jZSB0aGlzIHdhbGwtY2xvY2sgZWxhcHNlcywgc28gYQogICAgICAgICAgICAjIHN0YWxsZWQgZGVjb2RlIGNhbm5vdCBvdmVycnVuIHRoZSBwZXItdGFzayB0aW1lIGJ1ZGdldC4KICAgICAgICAgICAgZ2VuX2t3YXJnc1sibWF4X3RpbWUiXSA9IG1heF90aW1lX3MKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgb3V0ID0gc2VsZi5tb2RlbC5nZW5lcmF0ZSgKICAgICAgICAgICAgICAgICoqaW5wdXRzLAogICAgICAgICAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgICAgICAgICBkb19zYW1wbGU9ZG9fc2FtcGxlLAogICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU9dGVtcGVyYXR1cmUgaWYgZG9fc2FtcGxlIGVsc2UgTm9uZSwKICAgICAgICAgICAgICAgIG51bV9yZXR1cm5fc2VxdWVuY2VzPW51bV9zYW1wbGVzIGlmIGRvX3NhbXBsZSBlbHNlIDEsCiAgICAgICAgICAgICAgICBwYWRfdG9rZW5faWQ9c2VsZi50b2tlbml6ZXIucGFkX3Rva2VuX2lkLAogICAgICAgICAgICAgICAgKipnZW5fa3dhcmdzLAogICAgICAgICAgICApCiAgICAgICAgcHJvbXB0X2xlbiA9IGlucHV0c1siaW5wdXRfaWRzIl0uc2hhcGVbMV0KICAgICAgICBjb21wbGV0aW9ucyA9IFsKICAgICAgICAgICAgc2VsZi50b2tlbml6ZXIuZGVjb2RlKHNlcVtwcm9tcHRfbGVuOl0sIHNraXBfc3BlY2lhbF90b2tlbnM9VHJ1ZSkKICAgICAgICAgICAgZm9yIHNlcSBpbiBvdXQKICAgICAgICBdCiAgICAgICAgIyBHcmVlZHkgeWllbGRzIG9uZSBzZXF1ZW5jZTsgcGFkIHVwIHRvIG51bV9zYW1wbGVzIGZvciBhIHVuaWZvcm0gQVBJLgogICAgICAgIGlmIG5vdCBkb19zYW1wbGUgYW5kIG51bV9zYW1wbGVzID4gMToKICAgICAgICAgICAgY29tcGxldGlvbnMgPSBjb21wbGV0aW9ucyAqIG51bV9zYW1wbGVzCiAgICAgICAgcmV0dXJuIGNvbXBsZXRpb25zCgogICAgZGVmIHNjb3JlKHNlbGYsIHByb21wdDogc3RyLCBjb21wbGV0aW9uOiBzdHIpIC0+IGZsb2F0OgogICAgICAgIHRvcmNoID0gc2VsZi5fdG9yY2gKICAgICAgICBmdWxsID0gcHJvbXB0ICsgY29tcGxldGlvbgogICAgICAgIGVuYyA9IHNlbGYudG9rZW5pemVyKGZ1bGwsIHJldHVybl90ZW5zb3JzPSJwdCIpLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIHByb21wdF9sZW4gPSBzZWxmLnRva2VuaXplcihwcm9tcHQsIHJldHVybl90ZW5zb3JzPSJwdCIpWyJpbnB1dF9pZHMiXS5zaGFwZVsxXQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBsb2dpdHMgPSBzZWxmLm1vZGVsKCoqZW5jKS5sb2dpdHMKICAgICAgICBsb2dfcHJvYnMgPSB0b3JjaC5sb2dfc29mdG1heChsb2dpdHNbMCwgOi0xXSwgZGltPS0xKQogICAgICAgIHRhcmdldF9pZHMgPSBlbmNbImlucHV0X2lkcyJdWzAsIDE6XQogICAgICAgIHRva2VuX2xwID0gbG9nX3Byb2JzW3JhbmdlKHRhcmdldF9pZHMuc2hhcGVbMF0pLCB0YXJnZXRfaWRzXQogICAgICAgIGNvbXBsZXRpb25fbHAgPSB0b2tlbl9scFtwcm9tcHRfbGVuIC0gMSA6XQogICAgICAgIGlmIGNvbXBsZXRpb25fbHAubnVtZWwoKSA9PSAwOgogICAgICAgICAgICByZXR1cm4gZmxvYXQoIi1pbmYiKQogICAgICAgIHJldHVybiBmbG9hdChjb21wbGV0aW9uX2xwLm1lYW4oKSkK",
"src/arc/solvers/llm/select.py": "IiIiQ2FuZGlkYXRlIHNlbGVjdGlvbjogYWdncmVnYXRlIHdlaWdodGVkIGdyaWQgdm90ZXMgaW50byBhIHJhbmtlZCBsaXN0LgoKSW4gTTEgdGhlIHdlaWdodCBpcyBhIHNpbXBsZSBjb3VudCBhY3Jvc3MgYXVnbWVudGF0aW9ucyAoYSBncmlkIHRoYXQgc3Vydml2ZXMKbWFueSBpbmRlcGVuZGVudCBhdWdtZW50YXRpb25zIGlzIG1vcmUgdHJ1c3R3b3J0aHkpLiBNMiB3aWxsIGFkZCBtb2RlbApsb2ctbGlrZWxpaG9vZCBhcyBhbiBhZGRpdGlvbmFsIHdlaWdodCBzaWduYWwg4oCUIHNhbWUgYWdncmVnYXRpb24sIHJpY2hlciB3ZWlnaHRzLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBJdGVyYWJsZSwgU2VxdWVuY2UKCmZyb20gLi4uaW8uZ3JpZCBpbXBvcnQgR3JpZApmcm9tIC4uLmlvLmxvYWRlciBpbXBvcnQgUGFpcgpmcm9tIC4uLnNlcmlhbGl6ZS5wcm9tcHQgaW1wb3J0IGJ1aWxkX3Byb21wdApmcm9tIC4uLnNlcmlhbGl6ZS50b2tlbml6ZXIgaW1wb3J0IGdyaWRfdG9fc3RyCmZyb20gLm1vZGVsIGltcG9ydCBMYW5ndWFnZU1vZGVsCmZyb20gLnR0dF9kYXRhIGltcG9ydCBDT01QTEVUSU9OX1BSRUZJWAoKCmRlZiBzY29yZV9jYW5kaWRhdGVzKAogICAgbW9kZWw6IExhbmd1YWdlTW9kZWwsCiAgICB0cmFpbjogU2VxdWVuY2VbUGFpcl0sCiAgICB0ZXN0X2lucHV0OiBHcmlkLAogICAgY2FuZGlkYXRlczogU2VxdWVuY2VbR3JpZF0sCikgLT4gbGlzdFt0dXBsZVtHcmlkLCBmbG9hdF1dOgogICAgIiIiUmFuayBjYW5kaWRhdGVzIGJ5IHRoZSBtb2RlbCdzIGxvZy1saWtlbGlob29kIHVuZGVyIHRoZSBjYW5vbmljYWwgcHJvbXB0LgoKICAgIEVhY2ggY2FuZGlkYXRlIGdyaWQgaXMgc2NvcmVkIGFzIHRoZSBjb21wbGV0aW9uIHRoZSBtb2RlbCB3b3VsZCBhc3NpZ24gYWZ0ZXIKICAgIHRoZSBjbGVhbiAodW4tYXVnbWVudGVkKSBmZXctc2hvdCBwcm9tcHQg4oCUIGEgY29uZmlkZW5jZSBzaWduYWwgaW5kZXBlbmRlbnQgb2YKICAgIGhvdyB0aGUgY2FuZGlkYXRlIHdhcyBnZW5lcmF0ZWQuIEJlc3QgKGhpZ2hlc3QgbG9nLXByb2IpIGZpcnN0LgogICAgIiIiCiAgICBwcm9tcHQgPSBidWlsZF9wcm9tcHQodHJhaW4sIHRlc3RfaW5wdXQpCiAgICBzY29yZWQgPSBbCiAgICAgICAgKGdyaWQsIG1vZGVsLnNjb3JlKHByb21wdCwgQ09NUExFVElPTl9QUkVGSVggKyBncmlkX3RvX3N0cihncmlkKSkpCiAgICAgICAgZm9yIGdyaWQgaW4gY2FuZGlkYXRlcwogICAgXQogICAgc2NvcmVkLnNvcnQoa2V5PWxhbWJkYSBrdjogKC1rdlsxXSwgX3NpemUoa3ZbMF0pLCBrdlswXSkpCiAgICByZXR1cm4gc2NvcmVkCgoKZGVmIHJhbmtfYnlfdm90ZXMod2VpZ2h0ZWQ6IEl0ZXJhYmxlW3R1cGxlW0dyaWQsIGZsb2F0XV0pIC0+IGxpc3RbdHVwbGVbR3JpZCwgZmxvYXRdXToKICAgICIiIlN1bSB3ZWlnaHRzIHBlciBkaXN0aW5jdCBncmlkIGFuZCByZXR1cm4gdGhlbSBiZXN0LWZpcnN0LgoKICAgIFRpZXMgYnJlYWsgdG93YXJkIHRoZSBzbWFsbGVyIGdyaWQgKGEgbWlsZCBPY2NhbSBwcmlvciksIHRoZW4gbGV4aWNvZ3JhcGhpY2FsbHkKICAgIGZvciBkZXRlcm1pbmlzbS4KICAgICIiIgogICAgYWdnOiBkaWN0W0dyaWQsIGZsb2F0XSA9IGRlZmF1bHRkaWN0KGZsb2F0KQogICAgZm9yIGdyaWQsIHdlaWdodCBpbiB3ZWlnaHRlZDoKICAgICAgICBhZ2dbZ3JpZF0gKz0gd2VpZ2h0CiAgICByZXR1cm4gc29ydGVkKGFnZy5pdGVtcygpLCBrZXk9bGFtYmRhIGt2OiAoLWt2WzFdLCBfc2l6ZShrdlswXSksIGt2WzBdKSkKCgpkZWYgX3NpemUoZ3JpZDogR3JpZCkgLT4gaW50OgogICAgcmV0dXJuIGxlbihncmlkKSAqIChsZW4oZ3JpZFswXSkgaWYgZ3JpZCBlbHNlIDApCg==",
"src/arc/solvers/llm/solver.py": "IiIiTExNU29sdmVyIOKAlCB0cmFuc2R1Y3Rpb24gd2l0aCBhdWdtZW50YXRpb24tYmFzZWQgY2FuZGlkYXRlIHNlbGVjdGlvbi4KCkZvciBlYWNoIHRlc3QgaW5wdXQsIHRoZSB0YXNrIGlzIHJlLWV4cHJlc3NlZCB1bmRlciBzZXZlcmFsIGludmVydGlibGUKYXVnbWVudGF0aW9ucyAoRDQgc3ltbWV0cnkgKyBjb2xvdXIgcGVybXV0YXRpb24pLiBUaGUgbW9kZWwgcHJlZGljdHMgaW4gZWFjaAphdWdtZW50ZWQgZnJhbWU7IHByZWRpY3Rpb25zIGFyZSBpbnZlcnRlZCBiYWNrIHRvIHRoZSBjYW5vbmljYWwgZnJhbWUgYW5kIHZvdGVkLgpBdWdtZW50aW5nIGJvdGggZGl2ZXJzaWZpZXMgdGhlIG1vZGVsJ3MgaW5wdXRzIGFuZCBwcm92aWRlcyBhIGNvbnNlbnN1cyBzaWduYWwKdGhhdCBpcyBmYXIgbW9yZSByZWxpYWJsZSB0aGFuIGEgc2luZ2xlIGdyZWVkeSBkZWNvZGUuCgpUaGlzIGlzIHRoZSBNMSBiYXNlbGluZSAobm8gdGVzdC10aW1lIHRyYWluaW5nKTsgTTIgcGx1Z3MgYSBwZXItdGFzayBmaW5lLXR1bmVkCm1vZGVsIGludG8gdGhlIHNhbWUgc29sdmVyIHZpYSB0aGUgYG1vZGVsYCBhcmd1bWVudC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdGltZQoKZnJvbSAuLi5hdWdtZW50LnRhc2tfYXVnIGltcG9ydCBkaXN0aW5jdF9hdWdzCmZyb20gLi4uaW8ubG9hZGVyIGltcG9ydCBUYXNrCmZyb20gLi5iYXNlIGltcG9ydCBDYW5kaWRhdGVzLCBTb2x2ZXIKZnJvbSAuaW5mZXIgaW1wb3J0IGdlbmVyYXRlX2NhbmRpZGF0ZXMKZnJvbSAubW9kZWwgaW1wb3J0IExhbmd1YWdlTW9kZWwKZnJvbSAuc2VsZWN0IGltcG9ydCByYW5rX2J5X3ZvdGVzLCBzY29yZV9jYW5kaWRhdGVzCgoKY2xhc3MgTExNU29sdmVyKFNvbHZlcik6CiAgICBuYW1lID0gImxsbSIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBtb2RlbDogTGFuZ3VhZ2VNb2RlbCwKICAgICAgICBudW1fYXVnczogaW50ID0gNCwKICAgICAgICBudW1fc2FtcGxlczogaW50ID0gMSwKICAgICAgICBtYXhfbmV3X3Rva2VuczogaW50ID0gMTAyNCwKICAgICAgICB0ZW1wZXJhdHVyZTogZmxvYXQgPSAwLjAsCiAgICAgICAga2VlcF96ZXJvOiBib29sID0gRmFsc2UsCiAgICAgICAgbWF4X2NhbmRpZGF0ZXM6IGludCA9IDQsCiAgICAgICAgYXVnX3NlZWQ6IGludCA9IDAsCiAgICAgICAgdXNlX2xpa2VsaWhvb2Q6IGJvb2wgPSBGYWxzZSwKICAgICk6CiAgICAgICAgc2VsZi5tb2RlbCA9IG1vZGVsCiAgICAgICAgc2VsZi5udW1fYXVncyA9IG51bV9hdWdzCiAgICAgICAgc2VsZi5udW1fc2FtcGxlcyA9IG51bV9zYW1wbGVzCiAgICAgICAgc2VsZi5tYXhfbmV3X3Rva2VucyA9IG1heF9uZXdfdG9rZW5zCiAgICAgICAgc2VsZi50ZW1wZXJhdHVyZSA9IHRlbXBlcmF0dXJlCiAgICAgICAgc2VsZi5rZWVwX3plcm8gPSBrZWVwX3plcm8KICAgICAgICBzZWxmLm1heF9jYW5kaWRhdGVzID0gbWF4X2NhbmRpZGF0ZXMKICAgICAgICBzZWxmLmF1Z19zZWVkID0gYXVnX3NlZWQKICAgICAgICBzZWxmLnVzZV9saWtlbGlob29kID0gdXNlX2xpa2VsaWhvb2QKCiAgICBkZWYgc29sdmUoc2VsZiwgdGFzazogVGFzaywgYnVkZ2V0X3M6IGZsb2F0KSAtPiBDYW5kaWRhdGVzOgogICAgICAgIGRlYWRsaW5lID0gdGltZS5tb25vdG9uaWMoKSArIGJ1ZGdldF9zCiAgICAgICAgYXVncyA9IGRpc3RpbmN0X2F1Z3Moc2VsZi5udW1fYXVncywgc2VlZD1zZWxmLmF1Z19zZWVkLCBrZWVwX3plcm89c2VsZi5rZWVwX3plcm8pCiAgICAgICAgcGVyX3Rlc3Q6IENhbmRpZGF0ZXMgPSBbXQoKICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4odGFzay50ZXN0KSk6CiAgICAgICAgICAgIHdlaWdodGVkOiBsaXN0W3R1cGxlXSA9IFtdCiAgICAgICAgICAgIGZvciBqLCBhdWcgaW4gZW51bWVyYXRlKGF1Z3MpOgogICAgICAgICAgICAgICAgIyBBbHdheXMgcnVuIHRoZSBmaXJzdCAoaWRlbnRpdHkpIGF1Z21lbnRhdGlvbjsgc3RvcCBhZGRpbmcgbW9yZQogICAgICAgICAgICAgICAgIyBvbmNlIHRoZSBwZXItdGFzayBidWRnZXQgaXMgc3BlbnQuCiAgICAgICAgICAgICAgICBpZiBqID4gMCBhbmQgdGltZS5tb25vdG9uaWMoKSA+IGRlYWRsaW5lOgogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBhdGFzayA9IGF1Zy5hcHBseV90YXNrKHRhc2spCiAgICAgICAgICAgICAgICBncmlkcyA9IGdlbmVyYXRlX2NhbmRpZGF0ZXMoCiAgICAgICAgICAgICAgICAgICAgc2VsZi5tb2RlbCwKICAgICAgICAgICAgICAgICAgICBhdGFzay50cmFpbiwKICAgICAgICAgICAgICAgICAgICBhdGFzay50ZXN0W2ldLmlucHV0LAogICAgICAgICAgICAgICAgICAgIG51bV9zYW1wbGVzPXNlbGYubnVtX3NhbXBsZXMsCiAgICAgICAgICAgICAgICAgICAgbWF4X25ld190b2tlbnM9c2VsZi5tYXhfbmV3X3Rva2VucywKICAgICAgICAgICAgICAgICAgICB0ZW1wZXJhdHVyZT1zZWxmLnRlbXBlcmF0dXJlLAogICAgICAgICAgICAgICAgICAgICMgQ2FwIHRoaXMgZGVjb2RlIGF0IHRoZSB0aW1lIHN0aWxsIGxlZnQgaW4gdGhlIHBlci10YXNrCiAgICAgICAgICAgICAgICAgICAgIyBidWRnZXQgc28gb25lIHNsb3cgZ2VuZXJhdGlvbiBjYW4ndCBvdmVycnVuIGl0LgogICAgICAgICAgICAgICAgICAgIG1heF90aW1lX3M9bWF4KDAuMCwgZGVhZGxpbmUgLSB0aW1lLm1vbm90b25pYygpKSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGZvciBncmlkIGluIGdyaWRzOgogICAgICAgICAgICAgICAgICAgIHdlaWdodGVkLmFwcGVuZCgoYXVnLmludmVydF9ncmlkKGdyaWQpLCAxLjApKQoKICAgICAgICAgICAgcmFua2VkID0gcmFua19ieV92b3Rlcyh3ZWlnaHRlZCkKICAgICAgICAgICAgdm90ZWQgPSBbZyBmb3IgZywgXyBpbiByYW5rZWRdCiAgICAgICAgICAgIGlmIHNlbGYudXNlX2xpa2VsaWhvb2QgYW5kIHZvdGVkOgogICAgICAgICAgICAgICAgIyBSZS1yYW5rIHRoZSB2b3RlZCBjYW5kaWRhdGVzIGJ5IHRoZSBtb2RlbCdzIG93biBjb25maWRlbmNlCiAgICAgICAgICAgICAgICAjIHVuZGVyIHRoZSBjYW5vbmljYWwgKHVuLWF1Z21lbnRlZCkgcHJvbXB0LgogICAgICAgICAgICAgICAgc2NvcmVkID0gc2NvcmVfY2FuZGlkYXRlcygKICAgICAgICAgICAgICAgICAgICBzZWxmLm1vZGVsLCB0YXNrLnRyYWluLCB0YXNrLnRlc3RbaV0uaW5wdXQsIHZvdGVkCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICB2b3RlZCA9IFtnIGZvciBnLCBfIGluIHNjb3JlZF0KICAgICAgICAgICAgcGVyX3Rlc3QuYXBwZW5kKHZvdGVkWzogc2VsZi5tYXhfY2FuZGlkYXRlc10pCgogICAgICAgIHJldHVybiBwZXJfdGVzdAo=",
"src/arc/solvers/llm/ttt.py": "IiIiVGVzdC10aW1lIHRyYWluaW5nOiBwZXItdGFzayBMb1JBIGFkYXB0YXRpb24sIHRoZW4gdHJhbnNkdWN0aW9uLgoKYFRUVFJ1bm5lcmAgaXMgdGhlIHNlYW0gYmV0d2VlbiB0aGUgKENQVS10ZXN0YWJsZSkgb3JjaGVzdHJhdGlvbiBhbmQgdGhlCihHUFUtb25seSkgZ3JhZGllbnQgc3RlcDoKCiAgKiBgTW9ja1RUVFJ1bm5lcmAgcGVyZm9ybXMgbm8gdHJhaW5pbmcgYW5kIHJldHVybnMgdGhlIGJhc2UgbW9kZWwg4oCUIGxldHMgdGhlCiAgICB3aG9sZSBhZGFwdCAtPiBpbmZlciAtPiB2b3RlIGZsb3cgcnVuIGFuZCBiZSB1bml0LXRlc3RlZCBvbiBDUFUuCiAgKiBgTG9yYVRUVFJ1bm5lcmAgZmluZS10dW5lcyBhIGZyZXNoIExvUkEgYWRhcHRlciBvbiB0aGUgdGFzaydzIGNvcnB1cywgdGhlbgogICAgc2VydmVzIGluZmVyZW5jZSB0aHJvdWdoIHRoZSBhZGFwdGVkIG1vZGVsLiB0b3JjaC9wZWZ0IGltcG9ydCBsYXppbHk7IHRoaXMKICAgIHBhdGggaXMgdmFsaWRhdGVkIG9uIEthZ2dsZS4KCmBUVFRTb2x2ZXJgIHRpZXMgaXQgdG9nZXRoZXI6IGJ1aWxkIHRoZSBjb3JwdXMgKENQVSksIGFkYXB0IChydW5uZXIpLCB0aGVuCmRlbGVnYXRlIHRvIHRoZSBleGlzdGluZyBgTExNU29sdmVyYCBmb3IgYXVnbWVudGF0aW9uLWJhc2VkIGluZmVyZW5jZSArIHZvdGluZy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdGltZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IFByb3RvY29sCgpmcm9tIC4uLmlvLmxvYWRlciBpbXBvcnQgVGFzawpmcm9tIC4uYmFzZSBpbXBvcnQgQ2FuZGlkYXRlcywgU29sdmVyCmZyb20gLm1vZGVsIGltcG9ydCBMYW5ndWFnZU1vZGVsCmZyb20gLnNvbHZlciBpbXBvcnQgTExNU29sdmVyCmZyb20gLnR0dF9kYXRhIGltcG9ydCBUcmFpbkV4YW1wbGUsIGJ1aWxkX3R0dF9leGFtcGxlcwoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIFRUVENvbmZpZzoKICAgICIiIkxvUkEgKyB0cmFpbmluZyBoeXBlcnBhcmFtZXRlcnMgZm9yIHBlci10YXNrIGFkYXB0YXRpb24uIiIiCgogICAgbG9yYV9yOiBpbnQgPSAxNgogICAgbG9yYV9hbHBoYTogaW50ID0gMzIKICAgIGxvcmFfZHJvcG91dDogZmxvYXQgPSAwLjAKICAgIHRhcmdldF9tb2R1bGVzOiB0dXBsZVtzdHIsIC4uLl0gPSAoCiAgICAgICAgInFfcHJvaiIsCiAgICAgICAgImtfcHJvaiIsCiAgICAgICAgInZfcHJvaiIsCiAgICAgICAgIm9fcHJvaiIsCiAgICAgICAgImdhdGVfcHJvaiIsCiAgICAgICAgInVwX3Byb2oiLAogICAgICAgICJkb3duX3Byb2oiLAogICAgKQogICAgbGVhcm5pbmdfcmF0ZTogZmxvYXQgPSAxZS00CiAgICBtYXhfc3RlcHM6IGludCA9IDY0CiAgICBiYXRjaF9zaXplOiBpbnQgPSAyCiAgICBtYXhfc2VxX2xlbjogaW50ID0gMjA0OAogICAgc2VlZDogaW50ID0gMAoKCmNsYXNzIFRUVFJ1bm5lcihQcm90b2NvbCk6CiAgICAiIiJBZGFwdHMgYSBiYXNlIG1vZGVsIHRvIGEgdGFzaydzIGNvcnB1cyBhbmQgc2VydmVzIGluZmVyZW5jZS4iIiIKCiAgICBkZWYgYWRhcHQoc2VsZiwgZXhhbXBsZXM6IGxpc3RbVHJhaW5FeGFtcGxlXSkgLT4gTGFuZ3VhZ2VNb2RlbDoKICAgICAgICAiIiJUcmFpbiBvbiBgZXhhbXBsZXNgIGFuZCByZXR1cm4gYSBtb2RlbCByZWFkeSBmb3IgaW5mZXJlbmNlLiIiIgogICAgICAgIC4uLgoKICAgIGRlZiByZXNldChzZWxmKSAtPiBOb25lOgogICAgICAgICIiIlJlc3RvcmUgdGhlIGJhc2UgbW9kZWwgZm9yIHRoZSBuZXh0IHRhc2suIiIiCiAgICAgICAgLi4uCgoKY2xhc3MgTW9ja1RUVFJ1bm5lcjoKICAgICIiIk5vLW9wIHJ1bm5lciBmb3IgQ1BVIHRlc3RzOiByZWNvcmRzIHRoZSBjb3JwdXMsIHJldHVybnMgdGhlIGJhc2UgbW9kZWwuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGJhc2VfbW9kZWw6IExhbmd1YWdlTW9kZWwpOgogICAgICAgIHNlbGYuYmFzZV9tb2RlbCA9IGJhc2VfbW9kZWwKICAgICAgICBzZWxmLmxhc3RfZXhhbXBsZXM6IGxpc3RbVHJhaW5FeGFtcGxlXSA9IFtdCiAgICAgICAgc2VsZi5hZGFwdF9jYWxscyA9IDAKICAgICAgICBzZWxmLnJlc2V0X2NhbGxzID0gMAoKICAgIGRlZiBhZGFwdChzZWxmLCBleGFtcGxlczogbGlzdFtUcmFpbkV4YW1wbGVdKSAtPiBMYW5ndWFnZU1vZGVsOgogICAgICAgIHNlbGYubGFzdF9leGFtcGxlcyA9IGV4YW1wbGVzCiAgICAgICAgc2VsZi5hZGFwdF9jYWxscyArPSAxCiAgICAgICAgcmV0dXJuIHNlbGYuYmFzZV9tb2RlbAoKICAgIGRlZiByZXNldChzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYucmVzZXRfY2FsbHMgKz0gMQoKCmNsYXNzIFRUVFNvbHZlcihTb2x2ZXIpOgogICAgIiIiUGVyLXRhc2sgdGVzdC10aW1lIHRyYWluaW5nICsgdHJhbnNkdWN0aW9uLiIiIgoKICAgIG5hbWUgPSAibGxtX3R0dCIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBydW5uZXI6IFRUVFJ1bm5lciwKICAgICAgICBsbG1fa3dhcmdzOiBkaWN0IHwgTm9uZSA9IE5vbmUsCiAgICAgICAgdHR0X2RhdGFfa3dhcmdzOiBkaWN0IHwgTm9uZSA9IE5vbmUsCiAgICApOgogICAgICAgIHNlbGYucnVubmVyID0gcnVubmVyCiAgICAgICAgc2VsZi5sbG1fa3dhcmdzID0gbGxtX2t3YXJncyBvciB7fQogICAgICAgIHNlbGYudHR0X2RhdGFfa3dhcmdzID0gdHR0X2RhdGFfa3dhcmdzIG9yIHt9CgogICAgZGVmIHNvbHZlKHNlbGYsIHRhc2s6IFRhc2ssIGJ1ZGdldF9zOiBmbG9hdCkgLT4gQ2FuZGlkYXRlczoKICAgICAgICB0MCA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICBleGFtcGxlcyA9IGJ1aWxkX3R0dF9leGFtcGxlcyh0YXNrLCAqKnNlbGYudHR0X2RhdGFfa3dhcmdzKQogICAgICAgIHRyeToKICAgICAgICAgICAgYWRhcHRlZCA9IHNlbGYucnVubmVyLmFkYXB0KGV4YW1wbGVzKQogICAgICAgICAgICAjIENoYXJnZSBjb3JwdXMtYnVpbGQgKyBhZGFwdGF0aW9uIHRpbWUgYWdhaW5zdCB0aGlzIHNvbHZlcidzIGJ1ZGdldAogICAgICAgICAgICAjIHNvIHRoZSBpbm5lciB0cmFuc2R1Y3Rpb24gZ2V0cyB0aGUgdGltZSB0aGF0IGlzICphY3R1YWxseSogbGVmdCwKICAgICAgICAgICAgIyBub3QgYSBmcmVzaCBmdWxsIGJ1ZGdldCAod2hpY2ggc2lsZW50bHkgb3ZlcnJhbiB0aGUgcGVyLXRhc2sgY2FwKS4KICAgICAgICAgICAgcmVtYWluaW5nID0gbWF4KDAuMCwgYnVkZ2V0X3MgLSAodGltZS5tb25vdG9uaWMoKSAtIHQwKSkKICAgICAgICAgICAgcmV0dXJuIExMTVNvbHZlcihhZGFwdGVkLCAqKnNlbGYubGxtX2t3YXJncykuc29sdmUodGFzaywgcmVtYWluaW5nKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgICMgYGFkYXB0KClgIGlzIGluc2lkZSB0aGUgdHJ5IHNvIHJlc2V0KCkgcnVucyBldmVuIGlmIGFkYXB0YXRpb24KICAgICAgICAgICAgIyByYWlzZXMgKGUuZy4gQ1VEQSBPT00pLCByZXN0b3JpbmcgdGhlIHNoYXJlZCBiYXNlIG1vZGVsIGZvciB0aGUKICAgICAgICAgICAgIyBuZXh0IHRhc2sgaW5zdGVhZCBvZiBsZWF2aW5nIGl0IHdyYXBwZWQgaW4gYSBwYXJ0aWFsIGFkYXB0ZXIuCiAgICAgICAgICAgIHNlbGYucnVubmVyLnJlc2V0KCkKCgpjbGFzcyBMb3JhVFRUUnVubmVyOgogICAgIiIiR1BVIExvUkEgZmluZS10dW5lciAodG9yY2gvcGVmdCwgbGF6eSBpbXBvcnQpLiBWYWxpZGF0ZWQgb24gS2FnZ2xlLgoKICAgIFRyYWlucyBhIGZyZXNoIGFkYXB0ZXIgb24gdGhlIHRhc2sgY29ycHVzIGluIHBsYWNlLCBzZXJ2aW5nIGluZmVyZW5jZSB0aHJvdWdoCiAgICB0aGUgc2FtZSBgSEZNb2RlbGAsIHRoZW4gdW5sb2FkcyB0aGUgYWRhcHRlciBvbiBgcmVzZXQoKWAgdG8gcmVzdG9yZSB0aGUgYmFzZQogICAgd2VpZ2h0cyBmb3IgdGhlIG5leHQgdGFzay4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBoZl9tb2RlbCwgY29uZmlnOiBUVFRDb25maWcgfCBOb25lID0gTm9uZSk6CiAgICAgICAgc2VsZi5oZl9tb2RlbCA9IGhmX21vZGVsCiAgICAgICAgc2VsZi5jb25maWcgPSBjb25maWcgb3IgVFRUQ29uZmlnKCkKICAgICAgICBzZWxmLl9iYXNlID0gaGZfbW9kZWwubW9kZWwgICMgb3JpZ2luYWwgKHVuLWFkYXB0ZWQpIG1vZHVsZQoKICAgIGRlZiBhZGFwdChzZWxmLCBleGFtcGxlczogbGlzdFtUcmFpbkV4YW1wbGVdKSAtPiBMYW5ndWFnZU1vZGVsOgogICAgICAgIGltcG9ydCB0b3JjaCAgIyBub3FhOiBQTEMwNDE1CiAgICAgICAgZnJvbSBwZWZ0IGltcG9ydCBMb3JhQ29uZmlnLCBnZXRfcGVmdF9tb2RlbCAgIyBub3FhOiBQTEMwNDE1CgogICAgICAgIGNmZyA9IHNlbGYuY29uZmlnCiAgICAgICAgdG9yY2gubWFudWFsX3NlZWQoY2ZnLnNlZWQpCiAgICAgICAgdG9rZW5pemVyID0gc2VsZi5oZl9tb2RlbC50b2tlbml6ZXIKCiAgICAgICAgbG9yYSA9IExvcmFDb25maWcoCiAgICAgICAgICAgIHI9Y2ZnLmxvcmFfciwKICAgICAgICAgICAgbG9yYV9hbHBoYT1jZmcubG9yYV9hbHBoYSwKICAgICAgICAgICAgbG9yYV9kcm9wb3V0PWNmZy5sb3JhX2Ryb3BvdXQsCiAgICAgICAgICAgIHRhcmdldF9tb2R1bGVzPWxpc3QoY2ZnLnRhcmdldF9tb2R1bGVzKSwKICAgICAgICAgICAgdGFza190eXBlPSJDQVVTQUxfTE0iLAogICAgICAgICkKICAgICAgICAjIGBnZXRfcGVmdF9tb2RlbGAgaW5qZWN0cyBMb1JBIGxheWVycyBpbnRvIGBzZWxmLl9iYXNlYCdzIG1vZHVsZSB0cmVlIGJ5CiAgICAgICAgIyByZWZlcmVuY2UsIHNvIGEgZmFpbHVyZSBwYXJ0d2F5IHRocm91Z2ggdHJhaW5pbmcgbXVzdCB1bmxvYWQgdGhlIHBhcnRpYWwKICAgICAgICAjIGFkYXB0ZXIg4oCUIG90aGVyd2lzZSB0aGUgc2hhcmVkIG1vZGVsIHN0YXlzIGNvcnJ1cHRlZCBmb3IgbGF0ZXIgdGFza3MuCiAgICAgICAgbW9kZWwgPSBnZXRfcGVmdF9tb2RlbChzZWxmLl9iYXNlLCBsb3JhKQogICAgICAgIHRyeToKICAgICAgICAgICAgbW9kZWwudHJhaW4oKQogICAgICAgICAgICBpZiBoYXNhdHRyKG1vZGVsLCAiZ3JhZGllbnRfY2hlY2twb2ludGluZ19lbmFibGUiKToKICAgICAgICAgICAgICAgIG1vZGVsLmdyYWRpZW50X2NoZWNrcG9pbnRpbmdfZW5hYmxlKCkKCiAgICAgICAgICAgIGJhdGNoZXMgPSBzZWxmLl90b2tlbml6ZSh0b2tlbml6ZXIsIGV4YW1wbGVzLCBjZmcubWF4X3NlcV9sZW4pCiAgICAgICAgICAgIG9wdGltID0gdG9yY2gub3B0aW0uQWRhbVcoCiAgICAgICAgICAgICAgICAocCBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJlc19ncmFkKSwKICAgICAgICAgICAgICAgIGxyPWNmZy5sZWFybmluZ19yYXRlLAogICAgICAgICAgICApCiAgICAgICAgICAgIGRldmljZSA9IHNlbGYuaGZfbW9kZWwuZGV2aWNlCiAgICAgICAgICAgIHJuZyA9IHRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKGNmZy5zZWVkKQogICAgICAgICAgICBmb3IgX3N0ZXAgaW4gcmFuZ2UoY2ZnLm1heF9zdGVwcyk6CiAgICAgICAgICAgICAgICBiYXRjaCA9IHNlbGYuX3NhbXBsZV9iYXRjaChiYXRjaGVzLCBjZmcuYmF0Y2hfc2l6ZSwgcm5nKQogICAgICAgICAgICAgICAgaW5wdXRfaWRzID0gYmF0Y2hbImlucHV0X2lkcyJdLnRvKGRldmljZSkKICAgICAgICAgICAgICAgIGxhYmVscyA9IGJhdGNoWyJsYWJlbHMiXS50byhkZXZpY2UpCiAgICAgICAgICAgICAgICBhdHRuID0gYmF0Y2hbImF0dGVudGlvbl9tYXNrIl0udG8oZGV2aWNlKQogICAgICAgICAgICAgICAgb3V0ID0gbW9kZWwoaW5wdXRfaWRzPWlucHV0X2lkcywgYXR0ZW50aW9uX21hc2s9YXR0biwgbGFiZWxzPWxhYmVscykKICAgICAgICAgICAgICAgIG91dC5sb3NzLmJhY2t3YXJkKCkKICAgICAgICAgICAgICAgIG9wdGltLnN0ZXAoKQogICAgICAgICAgICAgICAgb3B0aW0uemVyb19ncmFkKCkKCiAgICAgICAgICAgIG1vZGVsLmV2YWwoKQogICAgICAgICAgICBzZWxmLmhmX21vZGVsLm1vZGVsID0gbW9kZWwKICAgICAgICAgICAgcmV0dXJuIHNlbGYuaGZfbW9kZWwKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBzZWxmLmhmX21vZGVsLm1vZGVsID0gKAogICAgICAgICAgICAgICAgbW9kZWwudW5sb2FkKCkgaWYgaGFzYXR0cihtb2RlbCwgInVubG9hZCIpIGVsc2Ugc2VsZi5fYmFzZQogICAgICAgICAgICApCiAgICAgICAgICAgIHJhaXNlCgogICAgZGVmIHJlc2V0KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgbW9kZWwgPSBzZWxmLmhmX21vZGVsLm1vZGVsCiAgICAgICAgaWYgaGFzYXR0cihtb2RlbCwgInVubG9hZCIpOgogICAgICAgICAgICBzZWxmLmhmX21vZGVsLm1vZGVsID0gbW9kZWwudW5sb2FkKCkgICMgZHJvcCBhZGFwdGVyLCByZXN0b3JlIGJhc2UKICAgICAgICBlbHNlOiAgIyBwcmFnbWE6IG5vIGNvdmVyCiAgICAgICAgICAgIHNlbGYuaGZfbW9kZWwubW9kZWwgPSBzZWxmLl9iYXNlCgogICAgZGVmIF90b2tlbml6ZShzZWxmLCB0b2tlbml6ZXIsIGV4YW1wbGVzLCBtYXhfc2VxX2xlbik6CiAgICAgICAgZW9zID0gdG9rZW5pemVyLmVvc190b2tlbiBvciAiIgogICAgICAgIHJvd3MgPSBbXQogICAgICAgIGZvciBleCBpbiBleGFtcGxlczoKICAgICAgICAgICAgcHJvbXB0X2lkcyA9IHRva2VuaXplcihleC5wcm9tcHQsIGFkZF9zcGVjaWFsX3Rva2Vucz1GYWxzZSlbImlucHV0X2lkcyJdCiAgICAgICAgICAgIGNvbXBfaWRzID0gdG9rZW5pemVyKGV4LmNvbXBsZXRpb24gKyBlb3MsIGFkZF9zcGVjaWFsX3Rva2Vucz1GYWxzZSlbImlucHV0X2lkcyJdCiAgICAgICAgICAgIGlkcyA9IChwcm9tcHRfaWRzICsgY29tcF9pZHMpWzptYXhfc2VxX2xlbl0KICAgICAgICAgICAgbGFiZWxzID0gKFstMTAwXSAqIGxlbihwcm9tcHRfaWRzKSArIGNvbXBfaWRzKVs6bWF4X3NlcV9sZW5dCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKChpZHMsIGxhYmVscykpCiAgICAgICAgcmV0dXJuIHJvd3MKCiAgICBkZWYgX3NhbXBsZV9iYXRjaChzZWxmLCByb3dzLCBiYXRjaF9zaXplLCBybmcpOgogICAgICAgIGltcG9ydCB0b3JjaCAgIyBub3FhOiBQTEMwNDE1CgogICAgICAgIG4gPSBsZW4ocm93cykKICAgICAgICBpZHggPSB0b3JjaC5yYW5kaW50KDAsIG4sIChtaW4oYmF0Y2hfc2l6ZSwgbiksKSwgZ2VuZXJhdG9yPXJuZykudG9saXN0KCkKICAgICAgICBjaG9zZW4gPSBbcm93c1tpXSBmb3IgaSBpbiBpZHhdCiAgICAgICAgbWF4X2xlbiA9IG1heChsZW4oaWRzKSBmb3IgaWRzLCBfIGluIGNob3NlbikKICAgICAgICBwYWRfaWQgPSAwCiAgICAgICAgaW5wdXRfaWRzLCBsYWJlbHMsIGF0dG4gPSBbXSwgW10sIFtdCiAgICAgICAgZm9yIGlkcywgbGFiIGluIGNob3NlbjoKICAgICAgICAgICAgcGFkID0gbWF4X2xlbiAtIGxlbihpZHMpCiAgICAgICAgICAgIGlucHV0X2lkcy5hcHBlbmQoaWRzICsgW3BhZF9pZF0gKiBwYWQpCiAgICAgICAgICAgIGxhYmVscy5hcHBlbmQobGFiICsgWy0xMDBdICogcGFkKQogICAgICAgICAgICBhdHRuLmFwcGVuZChbMV0gKiBsZW4oaWRzKSArIFswXSAqIHBhZCkKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAiaW5wdXRfaWRzIjogdG9yY2gudGVuc29yKGlucHV0X2lkcyksCiAgICAgICAgICAgICJsYWJlbHMiOiB0b3JjaC50ZW5zb3IobGFiZWxzKSwKICAgICAgICAgICAgImF0dGVudGlvbl9tYXNrIjogdG9yY2gudGVuc29yKGF0dG4pLAogICAgICAgIH0K",
"src/arc/solvers/llm/ttt_data.py": "IiIiQnVpbGQgYSBwZXItdGFzayBmaW5lLXR1bmluZyBjb3JwdXMgZm9yIHRlc3QtdGltZSB0cmFpbmluZy4KCkEgdGFzayBjYXJyaWVzIG9ubHkgYSBoYW5kZnVsIG9mIGRlbW9uc3RyYXRpb24gcGFpcnMg4oCUIHRvbyBmZXcgdG8gZmluZS10dW5lIG9uCmRpcmVjdGx5LiBXZSBleHBhbmQgdGhlbSB0d28gd2F5czoKCiAgKiBsZWF2ZS1vbmUtb3V0OiBlYWNoIGRlbW8gcGFpciBiZWNvbWVzIGEgKHN1cHBvcnQgLT4gcXVlcnkpIHByZWRpY3Rpb24KICAgIHByb2JsZW0sIHNvIE4gcGFpcnMgeWllbGQgTiBzZWxmLXN1cGVydmlzZWQgZXhhbXBsZXM7CiAgKiBhdWdtZW50YXRpb246IGV2ZXJ5IHZpZXcgaXMgcmVwbGljYXRlZCB1bmRlciBpbnZlcnRpYmxlIEQ0IHggY29sb3VyCiAgICBhdWdtZW50YXRpb25zLCB0dXJuaW5nIE4gcGFpcnMgaW50byBodW5kcmVkcyBvZiB0cmFpbmluZyBleGFtcGxlcy4KCkVhY2ggZXhhbXBsZSBpcyBhIChwcm9tcHQsIGNvbXBsZXRpb24pIHRleHQgcGFpciB3aGVyZSB0aGUgcHJvbXB0IGVuZHMgd2l0aCB0aGUKb3BlbiBgT3V0cHV0OmAgdGFnIGFuZCB0aGUgY29tcGxldGlvbiBpcyB0aGUgdGFyZ2V0IGdyaWQuIFB1cmUgc3RyaW5nIGFzc2VtYmx5IOKAlApubyBtb2RlbCByZXF1aXJlZCDigJQgc28gdGhlIHdob2xlIGJ1aWxkZXIgaXMgdW5pdC10ZXN0YWJsZSBvbiBDUFUuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJhbmRvbQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmZyb20gLi4uYXVnbWVudC50YXNrX2F1ZyBpbXBvcnQgZGlzdGluY3RfYXVncywgbGVhdmVfb25lX291dApmcm9tIC4uLmlvLmxvYWRlciBpbXBvcnQgUGFpciwgVGFzawpmcm9tIC4uLnNlcmlhbGl6ZS5wcm9tcHQgaW1wb3J0IGJ1aWxkX3Byb21wdApmcm9tIC4uLnNlcmlhbGl6ZS50b2tlbml6ZXIgaW1wb3J0IGdyaWRfdG9fc3RyCgojIFRoZSBjb21wbGV0aW9uIHN0YXJ0cyBvbiB0aGUgbGluZSBhZnRlciAiT3V0cHV0OiIgKG1pcnJvcnMgdGhlIGRlbW8gZm9ybWF0KS4KQ09NUExFVElPTl9QUkVGSVggPSAiXG4iCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgVHJhaW5FeGFtcGxlOgogICAgIiIiT25lIHN1cGVydmlzZWQgZmluZS10dW5pbmcgZXhhbXBsZSBmb3IgVFRULiIiIgoKICAgIHByb21wdDogc3RyCiAgICBjb21wbGV0aW9uOiBzdHIKCgpkZWYgX2V4YW1wbGUoc3VwcG9ydDogdHVwbGVbUGFpciwgLi4uXSwgcXVlcnk6IFBhaXIpIC0+IFRyYWluRXhhbXBsZToKICAgIHByb21wdCA9IGJ1aWxkX3Byb21wdChzdXBwb3J0LCBxdWVyeS5pbnB1dCkKICAgIGNvbXBsZXRpb24gPSBDT01QTEVUSU9OX1BSRUZJWCArIGdyaWRfdG9fc3RyKHF1ZXJ5Lm91dHB1dCkKICAgIHJldHVybiBUcmFpbkV4YW1wbGUocHJvbXB0PXByb21wdCwgY29tcGxldGlvbj1jb21wbGV0aW9uKQoKCmRlZiBfdmlld3ModGFzazogVGFzaykgLT4gbGlzdFt0dXBsZVt0dXBsZVtQYWlyLCAuLi5dLCBQYWlyXV06CiAgICAiIiJMZWF2ZS1vbmUtb3V0IHZpZXdzOyBmYWxsIGJhY2sgdG8gc2VsZi12aWV3cyBpZiA8MiBkZW1vIHBhaXJzLiIiIgogICAgdmlld3MgPSBsZWF2ZV9vbmVfb3V0KHRhc2spCiAgICBpZiB2aWV3czoKICAgICAgICByZXR1cm4gdmlld3MKICAgIHJldHVybiBbKHRhc2sudHJhaW4sIHApIGZvciBwIGluIHRhc2sudHJhaW5dCgoKZGVmIGJ1aWxkX3R0dF9leGFtcGxlcygKICAgIHRhc2s6IFRhc2ssCiAgICBudW1fYXVnczogaW50ID0gMTYsCiAgICAqLAogICAgc2VlZDogaW50ID0gMCwKICAgIGtlZXBfemVybzogYm9vbCA9IEZhbHNlLAogICAgbWF4X2V4YW1wbGVzOiBpbnQgfCBOb25lID0gNTAwLAopIC0+IGxpc3RbVHJhaW5FeGFtcGxlXToKICAgICIiIkFzc2VtYmxlIHRoZSBUVFQgY29ycHVzIGZvciBhIHRhc2s6IGxlYXZlLW9uZS1vdXQgdmlld3Mgb3ZlciBtYW55CiAgICBhdWdtZW50YXRpb25zLCBkZWR1cGVkIGFuZCBjYXBwZWQuIiIiCiAgICBhdWdzID0gZGlzdGluY3RfYXVncyhudW1fYXVncywgc2VlZD1zZWVkLCBrZWVwX3plcm89a2VlcF96ZXJvKQogICAgZXhhbXBsZXM6IGxpc3RbVHJhaW5FeGFtcGxlXSA9IFtdCiAgICBzZWVuOiBzZXRbdHVwbGVbc3RyLCBzdHJdXSA9IHNldCgpCiAgICBmb3IgYXVnIGluIGF1Z3M6CiAgICAgICAgYXRhc2sgPSBhdWcuYXBwbHlfdGFzayh0YXNrKQogICAgICAgIGZvciBzdXBwb3J0LCBxdWVyeSBpbiBfdmlld3MoYXRhc2spOgogICAgICAgICAgICBpZiBxdWVyeS5vdXRwdXQgaXMgTm9uZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGV4ID0gX2V4YW1wbGUoc3VwcG9ydCwgcXVlcnkpCiAgICAgICAgICAgIGtleSA9IChleC5wcm9tcHQsIGV4LmNvbXBsZXRpb24pCiAgICAgICAgICAgIGlmIGtleSBpbiBzZWVuOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgc2Vlbi5hZGQoa2V5KQogICAgICAgICAgICBleGFtcGxlcy5hcHBlbmQoZXgpCgogICAgaWYgbWF4X2V4YW1wbGVzIGlzIG5vdCBOb25lIGFuZCBsZW4oZXhhbXBsZXMpID4gbWF4X2V4YW1wbGVzOgogICAgICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgICAgICBybmcuc2h1ZmZsZShleGFtcGxlcykKICAgICAgICBleGFtcGxlcyA9IGV4YW1wbGVzWzptYXhfZXhhbXBsZXNdCiAgICByZXR1cm4gZXhhbXBsZXMK",
"src/arc/synth/__init__.py": "IiIiU3ludGhldGljIEFSQy1saWtlIHRhc2sgZ2VuZXJhdGlvbiBmb3IgYmFzZSBmaW5lLXR1bmluZyAoTTMpLiIiIgoKZnJvbSAuYnVpbGRfZGF0YXNldCBpbXBvcnQgKAogICAgYnVpbGRfc3ludGhldGljX3Rhc2tzLAogICAgZ2VuZXJhdGVfZXhhbXBsZXMsCiAgICBsb2FkX2V4YW1wbGVzX2pzb25sLAogICAgc2F2ZV9leGFtcGxlc19qc29ubCwKICAgIHRhc2tfdG9fZXhhbXBsZSwKICAgIHRhc2tzX3RvX2V4YW1wbGVzLAopCmZyb20gLmdlbmVyYXRvcnMgaW1wb3J0IEdFTkVSQVRPUlMsIEdlbmVyYXRlZFRhc2ssIGJ1aWxkX3Rhc2ssIGlzX3dlbGxfZm9ybWVkCgpfX2FsbF9fID0gWwogICAgIkdFTkVSQVRPUlMiLAogICAgIkdlbmVyYXRlZFRhc2siLAogICAgImJ1aWxkX3Rhc2siLAogICAgImlzX3dlbGxfZm9ybWVkIiwKICAgICJidWlsZF9zeW50aGV0aWNfdGFza3MiLAogICAgInRhc2tzX3RvX2V4YW1wbGVzIiwKICAgICJ0YXNrX3RvX2V4YW1wbGUiLAogICAgImdlbmVyYXRlX2V4YW1wbGVzIiwKICAgICJzYXZlX2V4YW1wbGVzX2pzb25sIiwKICAgICJsb2FkX2V4YW1wbGVzX2pzb25sIiwKXQo=",
"src/arc/synth/build_dataset.py": "IiIiQXNzZW1ibGUgc3ludGhldGljIHRhc2tzIGludG8gYSBiYXNlLWZpbmUtdHVuaW5nIGNvcnB1cy4KCkVhY2ggZ2VuZXJhdGVkIHRhc2sgYmVjb21lcyBvbmUgc3VwZXJ2aXNlZCBleGFtcGxlOiBwcm9tcHQgPSBkZW1vbnN0cmF0aW9uIHBhaXJzCisgdGVzdCBpbnB1dCAoZW5kaW5nIGF0IHRoZSBvcGVuIGBPdXRwdXQ6YCksIGNvbXBsZXRpb24gPSB0aGUgdGVzdCBvdXRwdXQgZ3JpZC4KVGhlIGNvcnB1cyBpcyB3aGF0IGJhc2UgZmluZS10dW5pbmcgdHJhaW5zIG9uIHNvIHRoZSBtb2RlbCBsZWFybnMgdGhlIEFSQyBJL08KZm9ybWF0IGFuZCBhIGJyb2FkIGxpYnJhcnkgb2YgdHJhbnNmb3JtYXRpb25zIGJlZm9yZSBwZXItdGFzayBUVFQuCgpgc2F2ZS9sb2FkX2V4YW1wbGVzX2pzb25sYCBwZXJzaXN0IHRoZSBjb3JwdXMgc28gaXQgY2FuIGJlIGdlbmVyYXRlZCBvbmNlIGFuZApzdGFnZWQgYXMgYSBLYWdnbGUgRGF0YXNldCBmb3Igb2ZmbGluZSBmaW5lLXR1bmluZy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gLi5pby5sb2FkZXIgaW1wb3J0IFRhc2sKZnJvbSAuLnNlcmlhbGl6ZS5wcm9tcHQgaW1wb3J0IGJ1aWxkX3Byb21wdApmcm9tIC4uc2VyaWFsaXplLnRva2VuaXplciBpbXBvcnQgZ3JpZF90b19zdHIKZnJvbSAuLnNvbHZlcnMubGxtLnR0dF9kYXRhIGltcG9ydCBDT01QTEVUSU9OX1BSRUZJWCwgVHJhaW5FeGFtcGxlCmZyb20gLmdlbmVyYXRvcnMgaW1wb3J0IEdFTkVSQVRPUlMsIEdlbmVyYXRvciwgYnVpbGRfdGFzaywgaXNfd2VsbF9mb3JtZWQKCl9NQVhfUkVUUklFUyA9IDUKCgpkZWYgYnVpbGRfc3ludGhldGljX3Rhc2tzKAogICAgbjogaW50LAogICAgc2VlZDogaW50ID0gMCwKICAgIGdlbmVyYXRvcnM6IHR1cGxlW0dlbmVyYXRvciwgLi4uXSA9IEdFTkVSQVRPUlMsCiAgICBudW1fcGFpcnM6IGludCA9IDMsCikgLT4gbGlzdFtUYXNrXToKICAgICIiIkdlbmVyYXRlIGBuYCB3ZWxsLWZvcm1lZCBzeW50aGV0aWMgdGFza3MsIHJvdW5kLXJvYmluIG92ZXIgZ2VuZXJhdG9ycy4iIiIKICAgIHRhc2tzOiBsaXN0W1Rhc2tdID0gW10KICAgIGkgPSAwCiAgICB3aGlsZSBsZW4odGFza3MpIDwgbjoKICAgICAgICBnZW4gPSBnZW5lcmF0b3JzW2kgJSBsZW4oZ2VuZXJhdG9ycyldCiAgICAgICAgZ3QgPSBOb25lCiAgICAgICAgZm9yIHIgaW4gcmFuZ2UoX01BWF9SRVRSSUVTKToKICAgICAgICAgICAgY2FuZGlkYXRlID0gYnVpbGRfdGFzayhnZW4sIHNlZWQ9c2VlZCAqIDFfMDAwXzAwMyArIGkgKiA3ICsgciwgbnVtX3BhaXJzPW51bV9wYWlycykKICAgICAgICAgICAgaWYgaXNfd2VsbF9mb3JtZWQoY2FuZGlkYXRlKToKICAgICAgICAgICAgICAgIGd0ID0gY2FuZGlkYXRlCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGkgKz0gMQogICAgICAgIGlmIGd0IGlzIG5vdCBOb25lOgogICAgICAgICAgICB0YXNrcy5hcHBlbmQoZ3QudGFzaykKICAgIHJldHVybiB0YXNrcwoKCmRlZiB0YXNrX3RvX2V4YW1wbGUodGFzazogVGFzaykgLT4gVHJhaW5FeGFtcGxlOgogICAgIiIiT25lIChwcm9tcHQsIGNvbXBsZXRpb24pIGV4YW1wbGUgZnJvbSBhIHRhc2sncyAoc2luZ2xlKSB0ZXN0IHBhaXIuIiIiCiAgICB0ZXN0ID0gdGFzay50ZXN0WzBdCiAgICBwcm9tcHQgPSBidWlsZF9wcm9tcHQodGFzay50cmFpbiwgdGVzdC5pbnB1dCkKICAgIGNvbXBsZXRpb24gPSBDT01QTEVUSU9OX1BSRUZJWCArIGdyaWRfdG9fc3RyKHRlc3Qub3V0cHV0KQogICAgcmV0dXJuIFRyYWluRXhhbXBsZShwcm9tcHQ9cHJvbXB0LCBjb21wbGV0aW9uPWNvbXBsZXRpb24pCgoKZGVmIHRhc2tzX3RvX2V4YW1wbGVzKHRhc2tzOiBsaXN0W1Rhc2tdKSAtPiBsaXN0W1RyYWluRXhhbXBsZV06CiAgICByZXR1cm4gW3Rhc2tfdG9fZXhhbXBsZSh0KSBmb3IgdCBpbiB0YXNrc10KCgpkZWYgZ2VuZXJhdGVfZXhhbXBsZXMobjogaW50LCBzZWVkOiBpbnQgPSAwKSAtPiBsaXN0W1RyYWluRXhhbXBsZV06CiAgICAiIiJDb252ZW5pZW5jZTogYnVpbGQgbiBzeW50aGV0aWMgdGFza3MgYW5kIHJldHVybiB0aGVpciBmaW5lLXR1bmluZyBleGFtcGxlcy4iIiIKICAgIHJldHVybiB0YXNrc190b19leGFtcGxlcyhidWlsZF9zeW50aGV0aWNfdGFza3Mobiwgc2VlZD1zZWVkKSkKCgpkZWYgc2F2ZV9leGFtcGxlc19qc29ubChleGFtcGxlczogbGlzdFtUcmFpbkV4YW1wbGVdLCBwYXRoOiBzdHIgfCBQYXRoKSAtPiBQYXRoOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggb3BlbihwYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGV4IGluIGV4YW1wbGVzOgogICAgICAgICAgICBmLndyaXRlKGpzb24uZHVtcHMoeyJwcm9tcHQiOiBleC5wcm9tcHQsICJjb21wbGV0aW9uIjogZXguY29tcGxldGlvbn0pICsgIlxuIikKICAgIHJldHVybiBwYXRoCgoKZGVmIGxvYWRfZXhhbXBsZXNfanNvbmwocGF0aDogc3RyIHwgUGF0aCkgLT4gbGlzdFtUcmFpbkV4YW1wbGVdOgogICAgZXhhbXBsZXM6IGxpc3RbVHJhaW5FeGFtcGxlXSA9IFtdCiAgICB3aXRoIG9wZW4ocGF0aCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBmb3IgbGluZSBpbiBmOgogICAgICAgICAgICByb3cgPSBqc29uLmxvYWRzKGxpbmUpCiAgICAgICAgICAgIGV4YW1wbGVzLmFwcGVuZChUcmFpbkV4YW1wbGUocHJvbXB0PXJvd1sicHJvbXB0Il0sIGNvbXBsZXRpb249cm93WyJjb21wbGV0aW9uIl0pKQogICAgcmV0dXJuIGV4YW1wbGVzCg==",
"src/arc/synth/generators.py": "IiIiUHJvY2VkdXJhbCBBUkMtbGlrZSB0YXNrIGdlbmVyYXRvcnMuCgpFYWNoIGdlbmVyYXRvciBzYW1wbGVzIE9ORSBncmlkLT5ncmlkIHRyYW5zZm9ybWF0aW9uIHBsdXMgYSBzZXQgb2YgaW5wdXQgZ3JpZHM7CmFwcGx5aW5nIHRoZSB0cmFuc2Zvcm0gdG8gZXZlcnkgaW5wdXQgeWllbGRzIGEgc2VsZi1jb25zaXN0ZW50IHRhc2suIEJ1aWxkaW5nCmZyb20gYSBzaW5nbGUgdHJhbnNmb3JtIG1lYW5zIHJ1bGUtY29uc2lzdGVuY3kgaXMgZ3VhcmFudGVlZCBieSBjb25zdHJ1Y3Rpb24g4oCUCnRoZSB0ZXN0IHN1aXRlIGp1c3QgcmUtY2hlY2tzIGB0cmFuc2Zvcm0oaW5wdXQpID09IG91dHB1dGAgZm9yIGV2ZXJ5IHBhaXIuCgpBIGxhcmdlLCBkaXZlcnNlIHN5bnRoZXRpYyBjb3JwdXMgZnJvbSB0aGVzZSBnZW5lcmF0b3JzIGlzIHdoYXQgbGV0cyBiYXNlCmZpbmUtdHVuaW5nIGdpdmUgdGhlIG1vZGVsIGEgc3Ryb25nIEFSQyBwcmlvciwgc28gcGVyLXRhc2sgdGVzdC10aW1lIHRyYWluaW5nCnN0YXJ0cyBmcm9tIGEgZ29vZCBwbGFjZSByYXRoZXIgdGhhbiBjb2xkICh0aGUgZWRnZSBiZWhpbmQgdGhlIDIwMjUgd2lubmVyKS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYWJjCmZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBDYWxsYWJsZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuLmF1Z21lbnQgaW1wb3J0IHN5bW1ldHJ5CmZyb20gLi5pby5ncmlkIGltcG9ydCBHcmlkLCBmcm9tX251bXB5LCBpc192YWxpZF9ncmlkLCB0b19udW1weQpmcm9tIC4uaW8ubG9hZGVyIGltcG9ydCBQYWlyLCBUYXNrCmZyb20gLi5zb2x2ZXJzLmRzbC5wcmltaXRpdmVzIGltcG9ydCAoCiAgICBjb2xvcm1hcF9wcm9ncmFtLAogICAgY3JvcF90b19jb250ZW50LAogICAgc2NhbGVfcHJvZ3JhbSwKICAgIHRpbGVfcHJvZ3JhbSwKKQoKVHJhbnNmb3JtID0gQ2FsbGFibGVbW0dyaWRdLCBHcmlkXQpCRyA9IDAKCgojIC0tLS0gcmFuZG9tIGhlbHBlcnMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KZGVmIF9wYWxldHRlKHJuZzogbnAucmFuZG9tLkdlbmVyYXRvciwgazogaW50IHwgTm9uZSA9IE5vbmUpIC0+IGxpc3RbaW50XToKICAgICIiIkEgdGFzayBwYWxldHRlOiBiYWNrZ3JvdW5kIDAgcGx1cyBhIGZldyBkaXN0aW5jdCBub24temVybyBzeW1ib2xzLiIiIgogICAgayA9IGsgb3IgaW50KHJuZy5pbnRlZ2VycygyLCA1KSkKICAgIG5vbnplcm8gPSBsaXN0KHJuZy5jaG9pY2UocmFuZ2UoMSwgMTApLCBzaXplPW1pbihrLCA5KSwgcmVwbGFjZT1GYWxzZSkpCiAgICByZXR1cm4gW0JHXSArIFtpbnQoYykgZm9yIGMgaW4gbm9uemVyb10KCgpkZWYgX3JhbmRvbV9ncmlkKAogICAgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yLAogICAgaDogaW50LAogICAgdzogaW50LAogICAgcGFsZXR0ZTogbGlzdFtpbnRdLAogICAgd2VpZ2h0czogbGlzdFtmbG9hdF0gfCBOb25lID0gTm9uZSwKKSAtPiBHcmlkOgogICAgYXJyID0gcm5nLmNob2ljZShwYWxldHRlLCBzaXplPShoLCB3KSwgcD13ZWlnaHRzKQogICAgcmV0dXJuIGZyb21fbnVtcHkobnAuYXNhcnJheShhcnIsIGR0eXBlPW5wLmludDgpKQoKCmRlZiBfc2l6ZShybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IsIGxvOiBpbnQsIGhpOiBpbnQpIC0+IGludDoKICAgIHJldHVybiBpbnQocm5nLmludGVnZXJzKGxvLCBoaSArIDEpKQoKCiMgLS0tLSBub24tRFNMIHRyYW5zZm9ybXMgKG51bXB5KSAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBfZ3Jhdml0eV9kb3duKGc6IEdyaWQpIC0+IEdyaWQ6CiAgICBhcnIgPSB0b19udW1weShnKQogICAgaCwgdyA9IGFyci5zaGFwZQogICAgb3V0ID0gbnAuZnVsbF9saWtlKGFyciwgQkcpCiAgICBmb3IgYyBpbiByYW5nZSh3KToKICAgICAgICBjb2wgPSBhcnJbOiwgY10KICAgICAgICBub24gPSBjb2xbY29sICE9IEJHXQogICAgICAgIGlmIGxlbihub24pOgogICAgICAgICAgICBvdXRbaCAtIGxlbihub24pIDosIGNdID0gbm9uCiAgICByZXR1cm4gZnJvbV9udW1weShvdXQpCgoKZGVmIF9taXJyb3JfY29uY2F0X2goZzogR3JpZCkgLT4gR3JpZDoKICAgIGFyciA9IHRvX251bXB5KGcpCiAgICByZXR1cm4gZnJvbV9udW1weShucC5oc3RhY2soW2FyciwgbnAuZmxpcGxyKGFycildKSkKCgpkZWYgX2JvcmRlcihjb2xvcjogaW50LCB3aWR0aDogaW50ID0gMSkgLT4gVHJhbnNmb3JtOgogICAgZGVmIGYoZzogR3JpZCkgLT4gR3JpZDoKICAgICAgICByZXR1cm4gZnJvbV9udW1weShucC5wYWQodG9fbnVtcHkoZyksIHdpZHRoLCBjb25zdGFudF92YWx1ZXM9Y29sb3IpKQoKICAgIHJldHVybiBmCgoKIyAtLS0tIGdlbmVyYXRvciBpbnRlcmZhY2UgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIEdlbmVyYXRlZFRhc2s6CiAgICB0YXNrOiBUYXNrCiAgICBydWxlOiBzdHIKICAgIHRyYW5zZm9ybTogVHJhbnNmb3JtCgoKY2xhc3MgR2VuZXJhdG9yKGFiYy5BQkMpOgogICAgbmFtZTogc3RyID0gImdlbmVyYXRvciIKCiAgICBAYWJjLmFic3RyYWN0bWV0aG9kCiAgICBkZWYgc2FtcGxlKHNlbGYsIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvciwgbl9pbnB1dHM6IGludCkgLT4gdHVwbGVbVHJhbnNmb3JtLCBsaXN0W0dyaWRdXToKICAgICAgICAiIiJSZXR1cm4gKHRyYW5zZm9ybSwgaW5wdXRzKSDigJQgaW5wdXRzIHNpemVkIHNvIG91dHB1dHMgc3RheSA8PSAzMHgzMC4iIiIKICAgICAgICByYWlzZSBOb3RJbXBsZW1lbnRlZEVycm9yCgoKY2xhc3MgUmVjb2xvckdlbmVyYXRvcihHZW5lcmF0b3IpOgogICAgbmFtZSA9ICJyZWNvbG9yIgoKICAgIGRlZiBzYW1wbGUoc2VsZiwgcm5nLCBuX2lucHV0cyk6CiAgICAgICAgcGFsZXR0ZSA9IF9wYWxldHRlKHJuZykKICAgICAgICBzaHVmZmxlZCA9IGxpc3Qocm5nLnBlcm11dGF0aW9uKHBhbGV0dGUpKQogICAgICAgIG1hcHBpbmcgPSB7aW50KHMpOiBpbnQoZCkgZm9yIHMsIGQgaW4gemlwKHBhbGV0dGUsIHNodWZmbGVkLCBzdHJpY3Q9RmFsc2UpfQogICAgICAgIHRyYW5zZm9ybSA9IGNvbG9ybWFwX3Byb2dyYW0obWFwcGluZykKICAgICAgICBpbnB1dHMgPSBbCiAgICAgICAgICAgIF9yYW5kb21fZ3JpZChybmcsIF9zaXplKHJuZywgMywgMTIpLCBfc2l6ZShybmcsIDMsIDEyKSwgcGFsZXR0ZSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobl9pbnB1dHMpCiAgICAgICAgXQogICAgICAgIHJldHVybiB0cmFuc2Zvcm0sIGlucHV0cwoKCmNsYXNzIENvbG9yU3dhcEdlbmVyYXRvcihHZW5lcmF0b3IpOgogICAgbmFtZSA9ICJjb2xvcl9zd2FwIgoKICAgIGRlZiBzYW1wbGUoc2VsZiwgcm5nLCBuX2lucHV0cyk6CiAgICAgICAgcGFsZXR0ZSA9IF9wYWxldHRlKHJuZywgaz0zKQogICAgICAgIGEsIGIgPSAoaW50KHgpIGZvciB4IGluIHJuZy5jaG9pY2UocGFsZXR0ZVsxOl0sIHNpemU9MiwgcmVwbGFjZT1GYWxzZSkpCiAgICAgICAgbWFwcGluZyA9IHthOiBiLCBiOiBhfQogICAgICAgIHRyYW5zZm9ybSA9IGNvbG9ybWFwX3Byb2dyYW0obWFwcGluZykKICAgICAgICBpbnB1dHMgPSBbCiAgICAgICAgICAgIF9yYW5kb21fZ3JpZChybmcsIF9zaXplKHJuZywgMywgMTIpLCBfc2l6ZShybmcsIDMsIDEyKSwgcGFsZXR0ZSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobl9pbnB1dHMpCiAgICAgICAgXQogICAgICAgIHJldHVybiB0cmFuc2Zvcm0sIGlucHV0cwoKCmNsYXNzIFN5bW1ldHJ5R2VuZXJhdG9yKEdlbmVyYXRvcik6CiAgICBuYW1lID0gInN5bW1ldHJ5IgoKICAgIGRlZiBzYW1wbGUoc2VsZiwgcm5nLCBuX2lucHV0cyk6CiAgICAgICAgb3AgPSBzdHIocm5nLmNob2ljZShbbiBmb3IgbiBpbiBzeW1tZXRyeS5ENF9OQU1FUyBpZiBuICE9ICJpZGVudGl0eSJdKSkKICAgICAgICB0cmFuc2Zvcm0gPSAobGFtYmRhIG5hbWU6IChsYW1iZGEgZzogc3ltbWV0cnkuYXBwbHkobmFtZSwgZykpKShvcCkKICAgICAgICBwYWxldHRlID0gX3BhbGV0dGUocm5nKQogICAgICAgIGlucHV0cyA9IFsKICAgICAgICAgICAgX3JhbmRvbV9ncmlkKHJuZywgX3NpemUocm5nLCAzLCAxNCksIF9zaXplKHJuZywgMywgMTQpLCBwYWxldHRlKQogICAgICAgICAgICBmb3IgXyBpbiByYW5nZShuX2lucHV0cykKICAgICAgICBdCiAgICAgICAgcmV0dXJuIHRyYW5zZm9ybSwgaW5wdXRzCgoKY2xhc3MgU2NhbGVHZW5lcmF0b3IoR2VuZXJhdG9yKToKICAgIG5hbWUgPSAic2NhbGUiCgogICAgZGVmIHNhbXBsZShzZWxmLCBybmcsIG5faW5wdXRzKToKICAgICAgICBmeSwgZnggPSBpbnQocm5nLmludGVnZXJzKDEsIDQpKSwgaW50KHJuZy5pbnRlZ2VycygxLCA0KSkKICAgICAgICBpZiAoZnksIGZ4KSA9PSAoMSwgMSk6CiAgICAgICAgICAgIGZ4ID0gMgogICAgICAgIHRyYW5zZm9ybSA9IHNjYWxlX3Byb2dyYW0oZnksIGZ4KQogICAgICAgIHBhbGV0dGUgPSBfcGFsZXR0ZShybmcpCiAgICAgICAgbWF4X2gsIG1heF93ID0gMzAgLy8gZnksIDMwIC8vIGZ4CiAgICAgICAgaW5wdXRzID0gWwogICAgICAgICAgICBfcmFuZG9tX2dyaWQocm5nLCBfc2l6ZShybmcsIDIsIG1pbihtYXhfaCwgMTApKSwgX3NpemUocm5nLCAyLCBtaW4obWF4X3csIDEwKSksIHBhbGV0dGUpCiAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKG5faW5wdXRzKQogICAgICAgIF0KICAgICAgICByZXR1cm4gdHJhbnNmb3JtLCBpbnB1dHMKCgpjbGFzcyBUaWxlR2VuZXJhdG9yKEdlbmVyYXRvcik6CiAgICBuYW1lID0gInRpbGUiCgogICAgZGVmIHNhbXBsZShzZWxmLCBybmcsIG5faW5wdXRzKToKICAgICAgICBueSwgbnggPSBpbnQocm5nLmludGVnZXJzKDEsIDQpKSwgaW50KHJuZy5pbnRlZ2VycygxLCA0KSkKICAgICAgICBpZiAobnksIG54KSA9PSAoMSwgMSk6CiAgICAgICAgICAgIG54ID0gMgogICAgICAgIHRyYW5zZm9ybSA9IHRpbGVfcHJvZ3JhbShueSwgbngpCiAgICAgICAgcGFsZXR0ZSA9IF9wYWxldHRlKHJuZykKICAgICAgICBtYXhfaCwgbWF4X3cgPSAzMCAvLyBueSwgMzAgLy8gbngKICAgICAgICBpbnB1dHMgPSBbCiAgICAgICAgICAgIF9yYW5kb21fZ3JpZChybmcsIF9zaXplKHJuZywgMiwgbWluKG1heF9oLCAxMCkpLCBfc2l6ZShybmcsIDIsIG1pbihtYXhfdywgMTApKSwgcGFsZXR0ZSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobl9pbnB1dHMpCiAgICAgICAgXQogICAgICAgIHJldHVybiB0cmFuc2Zvcm0sIGlucHV0cwoKCmNsYXNzIE1pcnJvckNvbmNhdEdlbmVyYXRvcihHZW5lcmF0b3IpOgogICAgbmFtZSA9ICJtaXJyb3JfY29uY2F0IgoKICAgIGRlZiBzYW1wbGUoc2VsZiwgcm5nLCBuX2lucHV0cyk6CiAgICAgICAgcGFsZXR0ZSA9IF9wYWxldHRlKHJuZykKICAgICAgICBpbnB1dHMgPSBbCiAgICAgICAgICAgIF9yYW5kb21fZ3JpZChybmcsIF9zaXplKHJuZywgMywgMTIpLCBfc2l6ZShybmcsIDIsIDE0KSwgcGFsZXR0ZSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobl9pbnB1dHMpCiAgICAgICAgXQogICAgICAgIHJldHVybiBfbWlycm9yX2NvbmNhdF9oLCBpbnB1dHMKCgpjbGFzcyBCb3JkZXJHZW5lcmF0b3IoR2VuZXJhdG9yKToKICAgIG5hbWUgPSAiYm9yZGVyIgoKICAgIGRlZiBzYW1wbGUoc2VsZiwgcm5nLCBuX2lucHV0cyk6CiAgICAgICAgcGFsZXR0ZSA9IF9wYWxldHRlKHJuZykKICAgICAgICBjb2xvciA9IGludChybmcuY2hvaWNlKHBhbGV0dGVbMTpdKSkKICAgICAgICB0cmFuc2Zvcm0gPSBfYm9yZGVyKGNvbG9yLCB3aWR0aD0xKQogICAgICAgIGlucHV0cyA9IFsKICAgICAgICAgICAgX3JhbmRvbV9ncmlkKHJuZywgX3NpemUocm5nLCAzLCAyNiksIF9zaXplKHJuZywgMywgMjYpLCBwYWxldHRlKQogICAgICAgICAgICBmb3IgXyBpbiByYW5nZShuX2lucHV0cykKICAgICAgICBdCiAgICAgICAgcmV0dXJuIHRyYW5zZm9ybSwgaW5wdXRzCgoKY2xhc3MgQ3JvcFRvQ29udGVudEdlbmVyYXRvcihHZW5lcmF0b3IpOgogICAgbmFtZSA9ICJjcm9wX3RvX2NvbnRlbnQiCgogICAgZGVmIHNhbXBsZShzZWxmLCBybmcsIG5faW5wdXRzKToKICAgICAgICBwYWxldHRlID0gX3BhbGV0dGUocm5nKQogICAgICAgIGlucHV0cyA9IFtzZWxmLl9jYW52YXMocm5nLCBwYWxldHRlKSBmb3IgXyBpbiByYW5nZShuX2lucHV0cyldCiAgICAgICAgcmV0dXJuIGNyb3BfdG9fY29udGVudCwgaW5wdXRzCgogICAgZGVmIF9jYW52YXMoc2VsZiwgcm5nLCBwYWxldHRlKSAtPiBHcmlkOgogICAgICAgIEgsIFcgPSBfc2l6ZShybmcsIDgsIDE2KSwgX3NpemUocm5nLCA4LCAxNikKICAgICAgICBhcnIgPSBucC5mdWxsKChILCBXKSwgQkcsIGR0eXBlPW5wLmludDgpCiAgICAgICAgb2gsIG93ID0gX3NpemUocm5nLCAyLCBtYXgoMiwgSCAtIDIpKSwgX3NpemUocm5nLCAyLCBtYXgoMiwgVyAtIDIpKQogICAgICAgIHIwLCBjMCA9IGludChybmcuaW50ZWdlcnMoMCwgSCAtIG9oICsgMSkpLCBpbnQocm5nLmludGVnZXJzKDAsIFcgLSBvdyArIDEpKQogICAgICAgIG9iaiA9IG5wLmFzYXJyYXkoCiAgICAgICAgICAgIHJuZy5jaG9pY2UocGFsZXR0ZVsxOl0sIHNpemU9KG9oLCBvdykpLCBkdHlwZT1ucC5pbnQ4CiAgICAgICAgKSAgIyBub24tYmcgb2JqZWN0CiAgICAgICAgYXJyW3IwIDogcjAgKyBvaCwgYzAgOiBjMCArIG93XSA9IG9iagogICAgICAgIHJldHVybiBmcm9tX251bXB5KGFycikKCgpjbGFzcyBHcmF2aXR5R2VuZXJhdG9yKEdlbmVyYXRvcik6CiAgICBuYW1lID0gImdyYXZpdHkiCgogICAgZGVmIHNhbXBsZShzZWxmLCBybmcsIG5faW5wdXRzKToKICAgICAgICBwYWxldHRlID0gX3BhbGV0dGUocm5nKQogICAgICAgICMgc3BhcnNlIGdyaWRzIHNvIGZhbGxpbmcgaXMgdmlzaWJsZSAoYmlhcyB0b3dhcmQgYmFja2dyb3VuZCkKICAgICAgICB3ZWlnaHRzID0gc2VsZi5fd2VpZ2h0cyhwYWxldHRlKQogICAgICAgIGlucHV0cyA9IFsKICAgICAgICAgICAgX3JhbmRvbV9ncmlkKHJuZywgX3NpemUocm5nLCA0LCAxNCksIF9zaXplKHJuZywgMywgMTIpLCBwYWxldHRlLCB3ZWlnaHRzKQogICAgICAgICAgICBmb3IgXyBpbiByYW5nZShuX2lucHV0cykKICAgICAgICBdCiAgICAgICAgcmV0dXJuIF9ncmF2aXR5X2Rvd24sIGlucHV0cwoKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBfd2VpZ2h0cyhwYWxldHRlOiBsaXN0W2ludF0pIC0+IGxpc3RbZmxvYXRdOgogICAgICAgIG4gPSBsZW4ocGFsZXR0ZSkKICAgICAgICB3ID0gWzAuNl0gKyBbMC40IC8gKG4gLSAxKV0gKiAobiAtIDEpICAjIDYwJSBiYWNrZ3JvdW5kCiAgICAgICAgcmV0dXJuIHcKCgpHRU5FUkFUT1JTOiB0dXBsZVtHZW5lcmF0b3IsIC4uLl0gPSAoCiAgICBSZWNvbG9yR2VuZXJhdG9yKCksCiAgICBDb2xvclN3YXBHZW5lcmF0b3IoKSwKICAgIFN5bW1ldHJ5R2VuZXJhdG9yKCksCiAgICBTY2FsZUdlbmVyYXRvcigpLAogICAgVGlsZUdlbmVyYXRvcigpLAogICAgTWlycm9yQ29uY2F0R2VuZXJhdG9yKCksCiAgICBCb3JkZXJHZW5lcmF0b3IoKSwKICAgIENyb3BUb0NvbnRlbnRHZW5lcmF0b3IoKSwKICAgIEdyYXZpdHlHZW5lcmF0b3IoKSwKKQoKCmRlZiBidWlsZF90YXNrKGdlbjogR2VuZXJhdG9yLCBzZWVkOiBpbnQsIG51bV9wYWlyczogaW50ID0gMykgLT4gR2VuZXJhdGVkVGFzazoKICAgICIiIkdlbmVyYXRlIG9uZSBzZWxmLWNvbnNpc3RlbnQgdGFzayBmcm9tIGBnZW5gLCBzZWVkZWQgZm9yIGRldGVybWluaXNtLiIiIgogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICB0cmFuc2Zvcm0sIGlucHV0cyA9IGdlbi5zYW1wbGUocm5nLCBudW1fcGFpcnMgKyAxKQogICAgcGFpcnMgPSBbUGFpcihpbnB1dD1nLCBvdXRwdXQ9dHJhbnNmb3JtKGcpKSBmb3IgZyBpbiBpbnB1dHNdCiAgICB0cmFpbiA9IHR1cGxlKHBhaXJzWzpudW1fcGFpcnNdKQogICAgdGVzdCA9IHBhaXJzW251bV9wYWlyc10KICAgIHRhc2sgPSBUYXNrKAogICAgICAgIHRhc2tfaWQ9ZiJzeW50aC17Z2VuLm5hbWV9LXtzZWVkfSIsCiAgICAgICAgdHJhaW49dHJhaW4sCiAgICAgICAgdGVzdD0oUGFpcihpbnB1dD10ZXN0LmlucHV0LCBvdXRwdXQ9dGVzdC5vdXRwdXQpLCksCiAgICApCiAgICByZXR1cm4gR2VuZXJhdGVkVGFzayh0YXNrPXRhc2ssIHJ1bGU9Z2VuLm5hbWUsIHRyYW5zZm9ybT10cmFuc2Zvcm0pCgoKZGVmIGlzX3dlbGxfZm9ybWVkKGd0OiBHZW5lcmF0ZWRUYXNrKSAtPiBib29sOgogICAgIiIiQWxsIGdyaWRzIHZhbGlkICg8PTMweDMwLCBzeW1ib2xzIDAtOSkgYW5kIHRoZSBydWxlIHJlcHJvZHVjZXMgb3V0cHV0cy4iIiIKICAgIGZvciBwYWlyIGluICgqZ3QudGFzay50cmFpbiwgKmd0LnRhc2sudGVzdCk6CiAgICAgICAgaWYgbm90IGlzX3ZhbGlkX2dyaWQocGFpci5pbnB1dCkgb3Igbm90IGlzX3ZhbGlkX2dyaWQocGFpci5vdXRwdXQpOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICBpZiBndC50cmFuc2Zvcm0ocGFpci5pbnB1dCkgIT0gcGFpci5vdXRwdXQ6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgcmV0dXJuIFRydWUK",
"src/arc/train/__init__.py": "",
"src/arc/train/finetune.py": "IiIiQmFzZSBmaW5lLXR1bmluZyBvbiB0aGUgc3ludGhldGljIGNvcnB1cyAoR1BVLCBLYWdnbGUpLgoKVHJhaW5zIGEgTG9SQSBhZGFwdGVyIG92ZXIgdGhlIHN5bnRoZXRpYyAocHJvbXB0LCBjb21wbGV0aW9uKSBjb3JwdXMgc28gdGhlIG1vZGVsCmxlYXJucyB0aGUgQVJDIEkvTyBmb3JtYXQgYW5kIGEgYnJvYWQgdHJhbnNmb3JtYXRpb24gbGlicmFyeSBCRUZPUkUgcGVyLXRhc2sgVFRULgpUaGUgcmVzdWx0aW5nIGFkYXB0ZXIgaXMgc3RhZ2VkIGFzIGEgS2FnZ2xlIERhdGFzZXQgYW5kIGxvYWRlZCBieSBgSEZNb2RlbCguLi4sCmFkYXB0ZXJfcGF0aD0uLi4pYCBhdCBpbmZlcmVuY2U7IFRUVCB0aGVuIGFkYXB0cyBmdXJ0aGVyIG9uIHRvcCBvZiBpdC4KCnRvcmNoL3RyYW5zZm9ybWVycy9wZWZ0IGltcG9ydCBsYXppbHkg4oCUIHRoaXMgbW9kdWxlIGlzIGltcG9ydC1zYWZlIG9mZi1HUFUgYnV0CmBmaW5ldHVuZSgpYCBvbmx5IHJ1bnMgd2hlcmUgdGhleSdyZSBpbnN0YWxsZWQgKEthZ2dsZSkuIE1pcnJvcnMgdGhlIG1hc2tpbmcgLwpsb29wIHN0eWxlIG9mIGBzb2x2ZXJzL2xsbS90dHQucHlgIHNvIHRoZSB0d28gdHJhaW5pbmcgcGF0aHMgc3RheSBjb25zaXN0ZW50LgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKZnJvbSAuLnNvbHZlcnMubGxtLnR0dF9kYXRhIGltcG9ydCBUcmFpbkV4YW1wbGUKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBUcmFpbkNvbmZpZzoKICAgIGxvcmFfcjogaW50ID0gMzIKICAgIGxvcmFfYWxwaGE6IGludCA9IDY0CiAgICBsb3JhX2Ryb3BvdXQ6IGZsb2F0ID0gMC4wNQogICAgdGFyZ2V0X21vZHVsZXM6IHR1cGxlW3N0ciwgLi4uXSA9ICgKICAgICAgICAicV9wcm9qIiwKICAgICAgICAia19wcm9qIiwKICAgICAgICAidl9wcm9qIiwKICAgICAgICAib19wcm9qIiwKICAgICAgICAiZ2F0ZV9wcm9qIiwKICAgICAgICAidXBfcHJvaiIsCiAgICAgICAgImRvd25fcHJvaiIsCiAgICApCiAgICBsZWFybmluZ19yYXRlOiBmbG9hdCA9IDFlLTQKICAgIGVwb2NoczogaW50ID0gMQogICAgYmF0Y2hfc2l6ZTogaW50ID0gOAogICAgZ3JhZF9hY2N1bTogaW50ID0gNAogICAgbWF4X3NlcV9sZW46IGludCA9IDIwNDgKICAgIHNlZWQ6IGludCA9IDAKCgpkZWYgX3Rva2VuaXplKHRva2VuaXplciwgZXhhbXBsZXM6IGxpc3RbVHJhaW5FeGFtcGxlXSwgbWF4X3NlcV9sZW46IGludCk6CiAgICAiIiJUb2tlbmlzZSBleGFtcGxlcyB3aXRoIHRoZSBwcm9tcHQgdG9rZW5zIG1hc2tlZCBvdXQgb2YgdGhlIGxvc3MgKC0xMDApLiIiIgogICAgZW9zID0gdG9rZW5pemVyLmVvc190b2tlbiBvciAiIgogICAgcm93cyA9IFtdCiAgICBmb3IgZXggaW4gZXhhbXBsZXM6CiAgICAgICAgcHJvbXB0X2lkcyA9IHRva2VuaXplcihleC5wcm9tcHQsIGFkZF9zcGVjaWFsX3Rva2Vucz1GYWxzZSlbImlucHV0X2lkcyJdCiAgICAgICAgY29tcF9pZHMgPSB0b2tlbml6ZXIoZXguY29tcGxldGlvbiArIGVvcywgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlKVsiaW5wdXRfaWRzIl0KICAgICAgICBpZHMgPSAocHJvbXB0X2lkcyArIGNvbXBfaWRzKVs6bWF4X3NlcV9sZW5dCiAgICAgICAgbGFiZWxzID0gKFstMTAwXSAqIGxlbihwcm9tcHRfaWRzKSArIGNvbXBfaWRzKVs6bWF4X3NlcV9sZW5dCiAgICAgICAgcm93cy5hcHBlbmQoKGlkcywgbGFiZWxzKSkKICAgIHJldHVybiByb3dzCgoKZGVmIF9jb2xsYXRlKGJhdGNoLCBwYWRfaWQsIHRvcmNoKToKICAgIG1heF9sZW4gPSBtYXgobGVuKGlkcykgZm9yIGlkcywgXyBpbiBiYXRjaCkKICAgIGlucHV0X2lkcywgbGFiZWxzLCBhdHRuID0gW10sIFtdLCBbXQogICAgZm9yIGlkcywgbGFiIGluIGJhdGNoOgogICAgICAgIHBhZCA9IG1heF9sZW4gLSBsZW4oaWRzKQogICAgICAgIGlucHV0X2lkcy5hcHBlbmQoaWRzICsgW3BhZF9pZF0gKiBwYWQpCiAgICAgICAgbGFiZWxzLmFwcGVuZChsYWIgKyBbLTEwMF0gKiBwYWQpCiAgICAgICAgYXR0bi5hcHBlbmQoWzFdICogbGVuKGlkcykgKyBbMF0gKiBwYWQpCiAgICByZXR1cm4gKAogICAgICAgIHRvcmNoLnRlbnNvcihpbnB1dF9pZHMpLAogICAgICAgIHRvcmNoLnRlbnNvcihsYWJlbHMpLAogICAgICAgIHRvcmNoLnRlbnNvcihhdHRuKSwKICAgICkKCgpkZWYgZmluZXR1bmUoCiAgICBiYXNlX21vZGVsX3BhdGg6IHN0ciwKICAgIGV4YW1wbGVzOiBsaXN0W1RyYWluRXhhbXBsZV0sCiAgICBvdXRwdXRfZGlyOiBzdHIsCiAgICBjb25maWc6IFRyYWluQ29uZmlnIHwgTm9uZSA9IE5vbmUsCiAgICBkZXZpY2U6IHN0ciA9ICJjdWRhIiwKKSAtPiBzdHI6CiAgICAiIiJGaW5lLXR1bmUgYSBMb1JBIGFkYXB0ZXIgb24gYGV4YW1wbGVzYDsgc2F2ZSB0byBgb3V0cHV0X2RpcmA7IHJldHVybiBpdC4iIiIKICAgIGltcG9ydCB0b3JjaCAgIyBub3FhOiBQTEMwNDE1CiAgICBmcm9tIHBlZnQgaW1wb3J0IExvcmFDb25maWcsIGdldF9wZWZ0X21vZGVsICAjIG5vcWE6IFBMQzA0MTUKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWxGb3JDYXVzYWxMTSwgQXV0b1Rva2VuaXplciAgIyBub3FhOiBQTEMwNDE1CgogICAgY2ZnID0gY29uZmlnIG9yIFRyYWluQ29uZmlnKCkKICAgIHRvcmNoLm1hbnVhbF9zZWVkKGNmZy5zZWVkKQoKICAgIHRva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKGJhc2VfbW9kZWxfcGF0aCkKICAgIGlmIHRva2VuaXplci5wYWRfdG9rZW5faWQgaXMgTm9uZToKICAgICAgICB0b2tlbml6ZXIucGFkX3Rva2VuID0gdG9rZW5pemVyLmVvc190b2tlbgogICAgbW9kZWwgPSBBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgYmFzZV9tb2RlbF9wYXRoLAogICAgICAgIHRvcmNoX2R0eXBlPXRvcmNoLmJmbG9hdDE2LAogICAgICAgIGRldmljZV9tYXA9ZGV2aWNlLAogICAgICAgIHVzZV9zYWZldGVuc29ycz1UcnVlLCAgIyByZWZ1c2UgcGlja2xlIC5iaW4gY2hlY2twb2ludHMgKFJDRSBzdXJmYWNlKQogICAgKQogICAgbG9yYSA9IExvcmFDb25maWcoCiAgICAgICAgcj1jZmcubG9yYV9yLAogICAgICAgIGxvcmFfYWxwaGE9Y2ZnLmxvcmFfYWxwaGEsCiAgICAgICAgbG9yYV9kcm9wb3V0PWNmZy5sb3JhX2Ryb3BvdXQsCiAgICAgICAgdGFyZ2V0X21vZHVsZXM9bGlzdChjZmcudGFyZ2V0X21vZHVsZXMpLAogICAgICAgIHRhc2tfdHlwZT0iQ0FVU0FMX0xNIiwKICAgICkKICAgIG1vZGVsID0gZ2V0X3BlZnRfbW9kZWwobW9kZWwsIGxvcmEpCiAgICBtb2RlbC50cmFpbigpCiAgICBpZiBoYXNhdHRyKG1vZGVsLCAiZ3JhZGllbnRfY2hlY2twb2ludGluZ19lbmFibGUiKToKICAgICAgICBtb2RlbC5ncmFkaWVudF9jaGVja3BvaW50aW5nX2VuYWJsZSgpCgogICAgcm93cyA9IF90b2tlbml6ZSh0b2tlbml6ZXIsIGV4YW1wbGVzLCBjZmcubWF4X3NlcV9sZW4pCiAgICBwYWRfaWQgPSB0b2tlbml6ZXIucGFkX3Rva2VuX2lkCiAgICBvcHRpbSA9IHRvcmNoLm9wdGltLkFkYW1XKAogICAgICAgIChwIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSBpZiBwLnJlcXVpcmVzX2dyYWQpLCBscj1jZmcubGVhcm5pbmdfcmF0ZQogICAgKQogICAgZ2VuID0gdG9yY2guR2VuZXJhdG9yKCkubWFudWFsX3NlZWQoY2ZnLnNlZWQpCgogICAgZm9yIGVwb2NoIGluIHJhbmdlKGNmZy5lcG9jaHMpOgogICAgICAgIG9yZGVyID0gdG9yY2gucmFuZHBlcm0obGVuKHJvd3MpLCBnZW5lcmF0b3I9Z2VuKS50b2xpc3QoKQogICAgICAgIG9wdGltLnplcm9fZ3JhZCgpCiAgICAgICAgZm9yIHN0ZXAsIHN0YXJ0IGluIGVudW1lcmF0ZShyYW5nZSgwLCBsZW4ob3JkZXIpLCBjZmcuYmF0Y2hfc2l6ZSkpOgogICAgICAgICAgICBpZHggPSBvcmRlcltzdGFydCA6IHN0YXJ0ICsgY2ZnLmJhdGNoX3NpemVdCiAgICAgICAgICAgIGlucHV0X2lkcywgbGFiZWxzLCBhdHRuID0gX2NvbGxhdGUoW3Jvd3NbaV0gZm9yIGkgaW4gaWR4XSwgcGFkX2lkLCB0b3JjaCkKICAgICAgICAgICAgb3V0ID0gbW9kZWwoCiAgICAgICAgICAgICAgICBpbnB1dF9pZHM9aW5wdXRfaWRzLnRvKGRldmljZSksCiAgICAgICAgICAgICAgICBhdHRlbnRpb25fbWFzaz1hdHRuLnRvKGRldmljZSksCiAgICAgICAgICAgICAgICBsYWJlbHM9bGFiZWxzLnRvKGRldmljZSksCiAgICAgICAgICAgICkKICAgICAgICAgICAgKG91dC5sb3NzIC8gY2ZnLmdyYWRfYWNjdW0pLmJhY2t3YXJkKCkKICAgICAgICAgICAgaWYgKHN0ZXAgKyAxKSAlIGNmZy5ncmFkX2FjY3VtID09IDA6CiAgICAgICAgICAgICAgICBvcHRpbS5zdGVwKCkKICAgICAgICAgICAgICAgIG9wdGltLnplcm9fZ3JhZCgpCiAgICAgICAgcHJpbnQoZiJlcG9jaCB7ZXBvY2ggKyAxfS97Y2ZnLmVwb2Noc30gZG9uZSIpCgogICAgbW9kZWwuc2F2ZV9wcmV0cmFpbmVkKG91dHB1dF9kaXIsIHNhZmVfc2VyaWFsaXphdGlvbj1UcnVlKSAgIyB3cml0ZSAuc2FmZXRlbnNvcnMKICAgIHRva2VuaXplci5zYXZlX3ByZXRyYWluZWQob3V0cHV0X2RpcikKICAgIHJldHVybiBvdXRwdXRfZGlyCg==",
"scripts/kaggle_submit.py": "IiIiS2FnZ2xlIG9mZmxpbmUgaW5mZXJlbmNlIGVudHJ5cG9pbnQgKE0xKykuCgpSdW5zIG9uIHRoZSBMNHg0IEdQVSB3aXRoIE5PIGludGVybmV0OiBsb2FkcyBhIGJhc2UgbW9kZWwgZnJvbSBhIHByZS1zdGFnZWQKS2FnZ2xlIERhdGFzZXQsIHJ1bnMgdGhlIERTTCArIExMTSBlbnNlbWJsZSBvdmVyIHRoZSB0ZXN0IGNoYWxsZW5nZXMgdW5kZXIgdGhlCmdsb2JhbCB0aW1lIHdhdGNoZG9nLCBhbmQgd3JpdGVzIC9rYWdnbGUvd29ya2luZy9zdWJtaXNzaW9uLmpzb24uCgpUaGUgZmluYWwgY29tcGV0aXRpb24gbm90ZWJvb2sgaXMgYSB0aGluIHdyYXBwZXIgdGhhdCBjYWxscyBgbWFpbigpYC4gS2VwdCBhcyBhCnBsYWluIHNjcmlwdCBzbyBpdCBpcyBpbXBvcnRhYmxlIGFuZCBsaW50LWNsZWFuIG9mZi1HUFU7IHRvcmNoIG9ubHkgbG9hZHMgd2hlbiBhbgpIRk1vZGVsIGlzIGFjdHVhbGx5IGNvbnN0cnVjdGVkLgoKVXNhZ2UgKGluc2lkZSB0aGUgS2FnZ2xlIG5vdGVib29rKToKICAgIGltcG9ydCBzeXM7IHN5cy5wYXRoLmFwcGVuZCgnL2thZ2dsZS9pbnB1dC88Y29kZS1kYXRhc2V0Pi9zcmMnKQogICAgZnJvbSBrYWdnbGVfc3VibWl0IGltcG9ydCBtYWluCiAgICBtYWluKG1vZGVsX3BhdGg9Jy9rYWdnbGUvaW5wdXQvPG1vZGVsLWRhdGFzZXQ+JykKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCB0aW1lCgpmcm9tIGFyYy5jb25maWcgaW1wb3J0IGdldF9jb25maWcKZnJvbSBhcmMuaW8ubG9hZGVyIGltcG9ydCBNYWxmb3JtZWRUYXNrRXJyb3IsIGxvYWRfY2hhbGxlbmdlcwpmcm9tIGFyYy5pby5zdWJtaXNzaW9uIGltcG9ydCAoCiAgICBidWlsZF9zdWJtaXNzaW9uLAogICAgZmFsbGJhY2tfZnJvbV9yYXcsCiAgICB2YWxpZGF0ZV9zdWJtaXNzaW9uLAogICAgd3JpdGVfc3VibWlzc2lvbiwKKQpmcm9tIGFyYy5waXBlbGluZSBpbXBvcnQgcnVuIGFzIHJ1bl9waXBlbGluZQoKCmRlZiBfcmVxdWlyZV9kYXRhKGNoYWxsZW5nZXNfcGF0aCkgLT4gTm9uZToKICAgICIiIkZhaWwgZmFzdCB3aXRoIGFuIGFjdGlvbmFibGUgbWVzc2FnZSBpZiB0aGUgY29tcGV0aXRpb24gZGF0YSBpcyBtaXNzaW5nIOKAlAogICAgdGhlIGNvbW1vbiBjYXVzZSBpcyBzaW1wbHkgdGhhdCB0aGUgY29tcGV0aXRpb24gZGF0YXNldCB3YXMgbmV2ZXIgYXR0YWNoZWQgdG8KICAgIHRoZSBub3RlYm9vaywgd2hpY2ggb3RoZXJ3aXNlIHN1cmZhY2VzIGFzIGEgY3J5cHRpYyBGaWxlTm90Rm91bmRFcnJvci4iIiIKICAgIGlmIG9zLnBhdGguZXhpc3RzKGNoYWxsZW5nZXNfcGF0aCk6CiAgICAgICAgcmV0dXJuCiAgICBpbnAgPSAiL2thZ2dsZS9pbnB1dCIKICAgIG1vdW50ZWQgPSBzb3J0ZWQob3MubGlzdGRpcihpbnApKSBpZiBvcy5wYXRoLmlzZGlyKGlucCkgZWxzZSBbXQogICAgbGlzdGluZyA9ICJcbiIuam9pbihmIiAgICAtIHtpbnB9L3ttfSIgZm9yIG0gaW4gbW91bnRlZCkgb3IgKAogICAgICAgICIgICAgKG5vdGhpbmcgbW91bnRlZCB1bmRlciAva2FnZ2xlL2lucHV0KSIKICAgICkKICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgIGYiQ29tcGV0aXRpb24gZGF0YSBub3QgZm91bmQ6IHtjaGFsbGVuZ2VzX3BhdGh9XG4iCiAgICAgICAgZiJPbiBLYWdnbGU6IHVzZSBBZGQgSW5wdXQgLT4gQ29tcGV0aXRpb25zIC0+ICdBUkMgUHJpemUgMjAyNiAtIEFSQy1BR0ktMicgIgogICAgICAgIGYic28gdGhlIGRhdGEgbW91bnRzIHVuZGVyIC9rYWdnbGUvaW5wdXQvLiBPciBzZXQgQVJDX0RBVEFfRElSIHRvIHRoZSBmb2xkZXIgIgogICAgICAgIGYidGhhdCBjb250YWlucyB7b3MucGF0aC5iYXNlbmFtZShzdHIoY2hhbGxlbmdlc19wYXRoKSl9LlxuIgogICAgICAgIGYiQ3VycmVudGx5IG1vdW50ZWQgdW5kZXIgL2thZ2dsZS9pbnB1dDpcbntsaXN0aW5nfSIKICAgICkKCgpkZWYgX3ByZV93cml0ZV9mYWxsYmFjayhjaGFsbGVuZ2VzX3BhdGgsIHN1Ym1pc3Npb25fcGF0aCkgLT4gaW50OgogICAgIiIiV3JpdGUgYSBjb21wbGV0ZSwgc2NoZW1hLXZhbGlkIGZhbGxiYWNrIHN1Ym1pc3Npb24gQkVGT1JFIGFueSBwYXJzaW5nIG9yCiAgICBtb2RlbCB3b3JrLCBzbyBhIGNyYXNoL09PTSBhbnl3aGVyZSBkb3duc3RyZWFtIHN0aWxsIGxlYXZlcyBhIHNjb3JlYWJsZSBmaWxlCiAgICBvbiBkaXNrLiBSZXR1cm5zIHRoZSBudW1iZXIgb2YgdGFza3MgY292ZXJlZCAoMCBpZiB0aGUgZmlsZSBpcyB1bnJlYWRhYmxlKS4iIiIKICAgIHRyeToKICAgICAgICB3aXRoIG9wZW4oY2hhbGxlbmdlc19wYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICByYXcgPSBqc29uLmxvYWQoZikKICAgICAgICBwcmVkcyA9IGZhbGxiYWNrX2Zyb21fcmF3KHJhdykKICAgICAgICB3cml0ZV9zdWJtaXNzaW9uKGJ1aWxkX3N1Ym1pc3Npb24ocHJlZHMpLCBzdWJtaXNzaW9uX3BhdGgpCiAgICAgICAgcmV0dXJuIGxlbihwcmVkcykKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCTEUwMDEg4oCUIGxhc3QtcmVzb3J0IGd1YXJkLCBuZXZlciBmYXRhbAogICAgICAgIHByaW50KGYiV0FSTklORzogY291bGQgbm90IHByZS13cml0ZSBmYWxsYmFjayBzdWJtaXNzaW9uOiB7ZXhjfSIpCiAgICAgICAgcmV0dXJuIDAKCgojIFBlci10ZXN0LWlucHV0IGF1Z21lbnRhdGlvbnMgLyBzYW1wbGVzIOKAlCB0dW5lZCBwZXIgbW9kZWwgb24gS2FnZ2xlLgpERUZBVUxUX0xMTV9LV0FSR1MgPSB7CiAgICAibnVtX2F1Z3MiOiA4LAogICAgIm51bV9zYW1wbGVzIjogMSwKICAgICJtYXhfbmV3X3Rva2VucyI6IDEwMjQsCiAgICAidGVtcGVyYXR1cmUiOiAwLjAsCn0KIyBDb3JwdXMgc2l6ZSBmb3IgcGVyLXRhc2sgdGVzdC10aW1lIHRyYWluaW5nIChsZWF2ZS1vbmUtb3V0IHggYXVnbWVudGF0aW9uKS4KREVGQVVMVF9UVFRfREFUQV9LV0FSR1MgPSB7Im51bV9hdWdzIjogMTYsICJtYXhfZXhhbXBsZXMiOiAyNTB9CgoKZGVmIF9idWlsZF9zb2x2ZXJzKG1vZGVsLCB1c2VfdHR0OiBib29sLCBsbG1fa3dhcmdzOiBkaWN0KToKICAgICIiIkFzc2VtYmxlIHRoZSBHUFUgZW5zZW1ibGU6IERTTCArIGhldXJpc3RpY3MgKyAoVFRUIG9yIHBsYWluIExMTSkuIiIiCiAgICBmcm9tIGFyYy5zb2x2ZXJzLmRzbC5zb2x2ZXIgaW1wb3J0IERTTFNvbHZlcgogICAgZnJvbSBhcmMuc29sdmVycy5pZGVudGl0eSBpbXBvcnQgQ0hFQVBfU09MVkVSUwoKICAgIGlmIHVzZV90dHQ6CiAgICAgICAgZnJvbSBhcmMuc29sdmVycy5sbG0gaW1wb3J0IExvcmFUVFRSdW5uZXIsIFRUVENvbmZpZywgVFRUU29sdmVyCgogICAgICAgIHR0dCA9IFRUVFNvbHZlcigKICAgICAgICAgICAgTG9yYVRUVFJ1bm5lcihtb2RlbCwgVFRUQ29uZmlnKCkpLAogICAgICAgICAgICBsbG1fa3dhcmdzPWxsbV9rd2FyZ3MsCiAgICAgICAgICAgIHR0dF9kYXRhX2t3YXJncz1ERUZBVUxUX1RUVF9EQVRBX0tXQVJHUywKICAgICAgICApCiAgICAgICAgcHJpbnQoImVuc2VtYmxlOiBEU0wgKyBoZXVyaXN0aWNzICsgVFRUKExvUkEpIikKICAgICAgICByZXR1cm4gW0RTTFNvbHZlcigpLCAqQ0hFQVBfU09MVkVSUywgdHR0XQoKICAgIGZyb20gYXJjLnNvbHZlcnMubGxtIGltcG9ydCBMTE1Tb2x2ZXIKCiAgICBwcmludCgiZW5zZW1ibGU6IERTTCArIGhldXJpc3RpY3MgKyBMTE0oSEZNb2RlbCkiKQogICAgcmV0dXJuIFtEU0xTb2x2ZXIoKSwgKkNIRUFQX1NPTFZFUlMsIExMTVNvbHZlcihtb2RlbCwgKipsbG1fa3dhcmdzKV0KCgpkZWYgbWFpbigKICAgIG1vZGVsX3BhdGg6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgYWRhcHRlcl9wYXRoOiBzdHIgfCBOb25lID0gTm9uZSwKICAgIHBlcl90YXNrX2J1ZGdldF9zOiBmbG9hdCA9IDE1MC4wLAogICAgbGxtX2t3YXJnczogZGljdCB8IE5vbmUgPSBOb25lLAogICAgdXNlX3R0dDogYm9vbCA9IFRydWUsCikgLT4gZGljdDoKICAgIGNmZyA9IGdldF9jb25maWcoKQogICAgbW9kZWxfcGF0aCA9IG1vZGVsX3BhdGggb3Igb3MuZW52aXJvbi5nZXQoIkFSQ19NT0RFTF9QQVRIIikKICAgIGFkYXB0ZXJfcGF0aCA9IGFkYXB0ZXJfcGF0aCBvciBvcy5lbnZpcm9uLmdldCgiQVJDX0FEQVBURVJfUEFUSCIpCiAgICBwcmludChmIm1vZGU9e2NmZy5tb2RlfSAgZGF0YV9kaXI9e2NmZy5kYXRhX2Rpcn0gIG1vZGVsX3BhdGg9e21vZGVsX3BhdGh9IikKICAgIHByaW50KGYiYWRhcHRlcl9wYXRoPXthZGFwdGVyX3BhdGh9ICB1c2VfdHR0PXt1c2VfdHR0fSIpCgogICAgY2hhbGxlbmdlc19wYXRoID0gY2ZnLmNoYWxsZW5nZXNfcGF0aCgidGVzdCIpCiAgICBfcmVxdWlyZV9kYXRhKGNoYWxsZW5nZXNfcGF0aCkgICMgY2xlYXIgZXJyb3IgaWYgdGhlIGRhdGEgaXNuJ3QgYXR0YWNoZWQKICAgIGNvdmVyZWQgPSBfcHJlX3dyaXRlX2ZhbGxiYWNrKGNoYWxsZW5nZXNfcGF0aCwgY2ZnLnN1Ym1pc3Npb25fcGF0aCkKICAgIHByaW50KGYicHJlLXdyb3RlIGZhbGxiYWNrIHN1Ym1pc3Npb24gZm9yIHtjb3ZlcmVkfSB0YXNrcyAtPiB7Y2ZnLnN1Ym1pc3Npb25fcGF0aH0iKQoKICAgIHRyeToKICAgICAgICB0YXNrcyA9IGxvYWRfY2hhbGxlbmdlcyhjaGFsbGVuZ2VzX3BhdGgpCiAgICBleGNlcHQgTWFsZm9ybWVkVGFza0Vycm9yIGFzIGV4YzoKICAgICAgICAjIFRoZSBwcmUtd3JpdHRlbiBmYWxsYmFjayBpcyBhbHJlYWR5IGEgY29tcGxldGUsIHNjb3JlYWJsZSBzdWJtaXNzaW9uOwogICAgICAgICMga2VlcCBpdCByYXRoZXIgdGhhbiBjcmFzaGluZyB0aGUga2VybmVsIHdpdGggYW4gZW1wdHkgb3V0cHV0LgogICAgICAgIHByaW50KGYiRVJST1I6IGNvdWxkIG5vdCBwYXJzZSBjaGFsbGVuZ2VzICh7ZXhjfSk7IGtlZXBpbmcgZmFsbGJhY2sgc3VibWlzc2lvbiIpCiAgICAgICAgcmV0dXJuIHsicHJvYmxlbXMiOiBbc3RyKGV4YyldLCAiZWxhcHNlZF9zIjogMC4wLCAibnVtX3Rhc2tzIjogMH0KICAgIHByaW50KGYibG9hZGVkIHtsZW4odGFza3MpfSB0ZXN0IHRhc2tzIikKCiAgICBzb2x2ZXJzID0gTm9uZQogICAgaWYgbW9kZWxfcGF0aDoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZyb20gYXJjLnNvbHZlcnMubGxtIGltcG9ydCBIRk1vZGVsICAjIGxhenk6IGltcG9ydHMgdG9yY2gKCiAgICAgICAgICAgICMgYWRhcHRlcl9wYXRoID0gdGhlIGJhc2UtZmluZS10dW5lZCAoc3ludGhldGljLWNvcnB1cykgYWRhcHRlcjsgVFRUCiAgICAgICAgICAgICMgYWRhcHRzIGZ1cnRoZXIgb24gdG9wIG9mIGl0IHBlciB0YXNrLgogICAgICAgICAgICBtb2RlbCA9IEhGTW9kZWwobW9kZWxfcGF0aCwgYWRhcHRlcl9wYXRoPWFkYXB0ZXJfcGF0aCkKICAgICAgICAgICAgc29sdmVycyA9IF9idWlsZF9zb2x2ZXJzKG1vZGVsLCB1c2VfdHR0LCBsbG1fa3dhcmdzIG9yIERFRkFVTFRfTExNX0tXQVJHUykKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzogICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgICMgQSBtb2RlbCB0aGF0IGNhbid0IGxvYWQgKG5vIEdQVSAtPiAiVG9yY2ggbm90IGNvbXBpbGVkIHdpdGggQ1VEQQogICAgICAgICAgICAjIGVuYWJsZWQiLCBPT00sIGJhZCBwYXRoKSBtdXN0IE5PVCBzaW5rIHRoZSBydW46IGZhbGwgYmFjayB0byB0aGUKICAgICAgICAgICAgIyBDUFUtc2FmZSBEU0wvaGV1cmlzdGljIGVuc2VtYmxlIGluc3RlYWQgb2YgY3Jhc2hpbmcgd2l0aCBhbgogICAgICAgICAgICAjIGFsbC1mYWxsYmFjayAoMXgxLXplcm8pIHN1Ym1pc3Npb24gdGhhdCBzY29yZXMgMC4KICAgICAgICAgICAgcHJpbnQoZiJXQVJOSU5HOiBtb2RlbCBsb2FkIGZhaWxlZCAoe2V4Y30pOyBmYWxsaW5nIGJhY2sgdG8gRFNMIGVuc2VtYmxlIikKICAgICAgICAgICAgc29sdmVycyA9IE5vbmUKICAgIGVsc2U6CiAgICAgICAgcHJpbnQoIldBUk5JTkc6IG5vIG1vZGVsX3BhdGgg4oCUIHJ1bm5pbmcgRFNML2hldXJpc3RpYyBlbnNlbWJsZSBvbmx5IikKCiAgICB0MCA9IHRpbWUubW9ub3RvbmljKCkKICAgIHByZWRpY3Rpb25zID0gcnVuX3BpcGVsaW5lKAogICAgICAgIHRhc2tzLAogICAgICAgIHNvbHZlcnM9c29sdmVycywKICAgICAgICBvdXRwdXRfcGF0aD1jZmcuc3VibWlzc2lvbl9wYXRoLAogICAgICAgIHBlcl90YXNrX2J1ZGdldF9zPXBlcl90YXNrX2J1ZGdldF9zLAogICAgICAgIHZlcmJvc2U9VHJ1ZSwKICAgICkKICAgIGVsYXBzZWQgPSB0aW1lLm1vbm90b25pYygpIC0gdDAKCiAgICBzdWJtaXNzaW9uID0gYnVpbGRfc3VibWlzc2lvbihwcmVkaWN0aW9ucykKICAgIHByb2JsZW1zID0gdmFsaWRhdGVfc3VibWlzc2lvbihzdWJtaXNzaW9uLCB0YXNrcykKICAgIHByaW50KGYic3VibWlzc2lvbjoge2NmZy5zdWJtaXNzaW9uX3BhdGh9ICBzY2hlbWFfcHJvYmxlbXM9e2xlbihwcm9ibGVtcyl9IikKICAgIGZvciBwIGluIHByb2JsZW1zWzo1XToKICAgICAgICBwcmludCgiICAtIiwgcCkKICAgIHByaW50KGYiZWxhcHNlZDoge2VsYXBzZWQgLyA2MDouMWZ9IG1pbiAgKHtlbGFwc2VkIC8gbWF4KGxlbih0YXNrcyksMSk6LjFmfXMvdGFzaykiKQogICAgcmV0dXJuIHsicHJvYmxlbXMiOiBwcm9ibGVtcywgImVsYXBzZWRfcyI6IGVsYXBzZWQsICJudW1fdGFza3MiOiBsZW4odGFza3MpfQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vZGVsLXBhdGgiLCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWFkYXB0ZXItcGF0aCIsIGRlZmF1bHQ9Tm9uZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcGVyLXRhc2stYnVkZ2V0IiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0xNTAuMCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0tbm8tdHR0IiwKICAgICAgICBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgIGhlbHA9IkRpc2FibGUgdGVzdC10aW1lIHRyYWluaW5nIChwbGFpbiBMTE0gdHJhbnNkdWN0aW9uKS4iLAogICAgKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIG1haW4oCiAgICAgICAgbW9kZWxfcGF0aD1hcmdzLm1vZGVsX3BhdGgsCiAgICAgICAgYWRhcHRlcl9wYXRoPWFyZ3MuYWRhcHRlcl9wYXRoLAogICAgICAgIHBlcl90YXNrX2J1ZGdldF9zPWFyZ3MucGVyX3Rhc2tfYnVkZ2V0LAogICAgICAgIHVzZV90dHQ9bm90IGFyZ3Mubm9fdHR0LAogICAgKQo="
}''')
ROOT = '/kaggle/working/arc_code'
for rel, b64 in FILES.items():
    dst = os.path.join(ROOT, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst, 'wb') as f:
        f.write(base64.b64decode(b64))
sys.path.insert(0, os.path.join(ROOT, 'src'))
sys.path.insert(0, os.path.join(ROOT, 'scripts'))
print('bootstrapped', len(FILES), 'files ->', ROOT)


In [ ]:
# Point these at your attached datasets/models.
MODEL_DS = None      # e.g. '/kaggle/input/qwen2.5-3b-instruct'  (None = DSL-only Phase A)
ADAPTER_DS = None    # e.g. '/kaggle/input/arc-base-ft-adapter'  (optional)


In [ ]:
from kaggle_submit import main
result = main(model_path=MODEL_DS, adapter_path=ADAPTER_DS,
              per_task_budget_s=150.0, use_ttt=True)
print(result)
assert result['problems'] == [], result['problems']
